# Enveda CASMI 2026 — Molecule ID from Mass Spectra (CPU / GPU)

Configured for **Kaggle Run All** with **internet off**. No TPU required.

Attach **three** inputs (Kaggle kernels must stay under **1 MB**, so the NP pool and RDKit wheels cannot live inside the notebook):

1. Competition data: `train.parquet`, `test.parquet`, `sample_submission.csv`
2. Class 2 COCONUT ∪ LOTUS table: attach the dataset that contains `class2_candidates.parquet` (~61 MB, ~453k structures). Kaggle may name the dataset folder `class2_candidates.parquet2`.
3. **RDKit wheel (required, ~35 MB):** do **not** use the current 20 KB stub named `rdkit-2025-9-5`. Upload `kaggle_rdkit_wheels/rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl` as a new dataset, then Add Input.

Internet stays **OFF**. The first code cell installs RDKit from that wheelhouse. You must see `rdkit 2025...` printed, not `WARNING: RDKit missing`.

That parquet is the **full** NP set (train skeletons removed, generic 50–2000 Da). It is **not** filtered on the public `test.parquet` masses.

**Single experiment vs 0.139:** preserve the representative spectrum's original
neutral-loss, precursor/CE/ion, and adduct tensors when constructing the training
dataset. The old path fabricated protonated-positive metadata for every row.
`USE_REPRESENTATIVE_METADATA = False` restores that original feature construction.
Keep the existing 0.139 Kaggle submission selected unless this run scores higher.

Pipeline:
1. Build a train-structure library (InChIKey14, exact mass, Morgan fingerprints, representative MS2).
2. Train up to **3 Spec2FP seeds** on 120k unique train skeletons (4 epochs, ~1.5 h each) and average pre-sigmoid logits. Blend with train spectral neighbors (k=20, a=0.5).
3. Rank: lock Class 1 if modified cosine ≥ 0.75. Remaining slots merge weak train hits, Class 2 COCONUT, and mass-shifted analogs by `0.42*spec + 0.46*tani + 0.12*mass`.
4. Fill leftover slots from attached COCONUT ∪ LOTUS (`class2_candidates.parquet`, mass filter at query time). Analog parents are retrieved at ±sugar/CH2/O/acetyl mass shifts.
5. Write **exactly 25** unique InChIKey14 guesses.

Output: `/kaggle/working/submission.csv`.


In [ ]:
# Offline RDKit install — MUST be the first code cell (internet OFF).
# Need a ~35 MB .whl, not the 20 KB stub dataset named rdkit-2025-9-5.
import os, sys, shutil, subprocess
from importlib import invalidate_caches
from pathlib import Path

def _rdkit_ok():
    try:
        from rdkit import Chem
        ver = getattr(getattr(Chem, "rdBase", Chem), "rdkitVersion", None) or "unknown"
        mol = Chem.MolFromSmiles("CCO")
        return bool(mol is not None), str(ver)
    except Exception:
        return False, ""

def _looks_like_wheel(path: Path) -> bool:
    name = path.name.lower()
    if name.endswith(".whl"):
        return True
    if "rdkit" in name and any(x in name for x in ("cp31", "manylinux", "win", "macosx")):
        return True
    try:
        with path.open("rb") as fh:
            magic = fh.read(4)
        # ZIP / wheel local file header
        if magic[:2] == b"PK":
            return "rdkit" in name or path.stat().st_size > 1_000_000
    except Exception:
        return False
    return False

def find_rdkit_wheels() -> list[Path]:
    inp = Path("/kaggle/input")
    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    print("listing /kaggle/input ...")
    if inp.exists():
        for p in sorted(inp.rglob("*")):
            kind = "dir" if p.is_dir() else "file"
            size = p.stat().st_size if p.is_file() else 0
            print(f"  {kind} {p} bytes={size}")
    small: list[tuple[Path, int]] = []
    hits: list[Path] = []
    if not inp.exists():
        return hits
    for p in inp.rglob("*"):
        if not p.is_file():
            continue
        parent = str(p.parent).lower()
        if "rdkit" not in parent and "rdkit" not in p.name.lower():
            continue
        size = p.stat().st_size
        if size < 5_000_000:
            small.append((p, size))
            continue
        if _looks_like_wheel(p) or "rdkit" in p.name.lower() or p.suffix.lower() == ".whl":
            hits.append(p)
    if small:
        print("rdkit-named files too small to be a wheel (need ~35 MB):")
        for p, size in small:
            print(f"  {p} bytes={size}")
    uniq: list[Path] = []
    seen: set[str] = set()
    for p in hits:
        key = str(p.resolve())
        if key in seen:
            continue
        seen.add(key)
        uniq.append(p)
    uniq.sort(
        key=lambda p: (
            0 if p.suffix.lower() == ".whl" else 1,
            0 if tag in p.name else 1,
            0 if "rdkit" in p.name.lower() else 1,
            -p.stat().st_size,
        )
    )
    return uniq

ok, ver = _rdkit_ok()
if ok:
    print("rdkit already installed", ver)
else:
    print("python", sys.version)
    wheels = find_rdkit_wheels()
    print("rdkit wheels found", [str(w) for w in wheels])
    if not wheels:
        raise FileNotFoundError(
            "No ~35 MB RDKit wheel under /kaggle/input. "
            "The attached 'rdkit-2025-9-5' file is only ~20 KB — that is not the wheel. "
            "On your laptop, upload this file as a NEW Kaggle dataset (do not reuse the stub): "
            "kaggle_rdkit_wheels/rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl "
            "(~35 MB). Then Add Input that dataset. Keep internet OFF."
        )
    src = wheels[0]
    work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/tmp")
    dest_dir = work / "_rdkit_wheels"
    dest_dir.mkdir(parents=True, exist_ok=True)
    py = f"cp{sys.version_info.major}{sys.version_info.minor}"
    dest = dest_dir / f"rdkit-2025.9.5-{py}-{py}-manylinux_2_28_x86_64.whl"
    print("copying", src, "->", dest, "src_size", src.stat().st_size)
    shutil.copy2(src, dest)
    cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", str(dest)]
    print("install", " ".join(cmd))
    subprocess.check_call(cmd)
    invalidate_caches()
    ok, ver = _rdkit_ok()
    if not ok:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
             f"--find-links={dest_dir}", "rdkit"]
        )
        invalidate_caches()
        ok, ver = _rdkit_ok()
    if not ok:
        raise ImportError(f"pip installed {src} but `from rdkit import Chem` still fails")
    print("rdkit", ver)
assert _rdkit_ok()[0], "HAS_RDKIT must be True after the offline wheel install"
print("HAS_RDKIT", True, "rdkit", _rdkit_ok()[1])


In [ ]:
# ===== Fully configured for Run All on Kaggle CPU (9 h) =====
# Session: CPU (or GPU). Do not start a TPU session.
# Internet: OFF.
# Add Data:
#   1) Enveda CASMI 2026 competition
#   2) class2_candidates.parquet (folder may be class2_candidates.parquet2)
#   3) rdkit-cp312-wheel (~35 MB .whl, not the 20 KB stub)
REQUIRE_CLASS2 = True         # fail fast if class2_candidates.parquet is not attached
FAST_DEV_RUN = False          # True = 8k-row debug pass; keep False for a real submit
PREFER_XLA = False            # TPU off: CPU/GPU path
TRAIN_DECODER = False         # Class 3 analog shifts run without the SMILES decoder
ENSEMBLE_SEEDS = 3            # average pre-sigmoid Spec2FP logits
MAX_TRAIN_SPECTRA = 120_000   # V5/V6 0.139; 280k/6ep scored 0.126
USE_REPRESENTATIVE_METADATA = True  # the only experiment: observed training features
NUM_EPOCHS = 4
BATCH_SIZE = 64
TRAIN_TIME_LIMIT_S = 4.5 * 3600.0
USE_NEIGHBOR_FP = True
NEIGHBOR_BLEND = 0.5
RANK_RESERVE_S = 2.0 * 3600.0
PER_SEED_MAX_S = 1.5 * 3600.0
DECODER_TIME_LIMIT_S = 0.6 * 3600.0
DECODER_MAX_SMILES = 50_000
TOP_N_PEAKS = 128
N_MZ_BINS = 512
FP_BITS = 2048
SESSION_BUDGET_S = 8.5 * 3600.0


In [ ]:
import os, sys, json, time, math, traceback
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("OMP_NUM_THREADS", str(min(8, os.cpu_count() or 4)))
os.environ.setdefault("MKL_NUM_THREADS", os.environ["OMP_NUM_THREADS"])

KAGGLE_WORKING = Path("/kaggle/working")
IS_KAGGLE = KAGGLE_WORKING.exists()
ROOT = KAGGLE_WORKING if IS_KAGGLE else Path(".").resolve()
(ROOT / "src" / "models").mkdir(parents=True, exist_ok=True)
(ROOT / "artifacts").mkdir(parents=True, exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

T_START = time.time()

print("ROOT", ROOT, "kaggle", IS_KAGGLE)
print("python", sys.version)
print("cpu_count", os.cpu_count())
print("internet: OFF (notebook metadata isInternetEnabled=False). Toggle Internet off in the editor.")
if Path("/kaggle/input").exists():
    attached = sorted(p.name for p in Path("/kaggle/input").iterdir())
    print("attached /kaggle/input datasets:", attached)
    if not attached:
        print(
            "WARNING: /kaggle/input is empty — Add Data for competition + "
            "class2 parquet + a ~35 MB rdkit .whl"
        )
else:
    print("not on Kaggle; using local data/")


## 1. Materialize the `src/` package (offline)

In [ ]:
import json
from pathlib import Path

FILES = json.loads('{"src/__init__.py": "\\"\\"\\"CASMI 2026 spectrum-to-structure identification pipeline.\\"\\"\\"\\n\\nfrom src.config import Config, get_config\\n\\n__all__ = [\\"Config\\", \\"get_config\\"]\\n__version__ = \\"0.1.0\\"\\n", "src/config.py": "\\"\\"\\"Centralized hyperparameters, adduct tables, and filesystem paths.\\n\\nShapes in this config are the TPU/XLA contract: every spectral tensor is padded\\nor binned to these fixed dimensions so the XLA graph is compiled once.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass, field\\nfrom pathlib import Path\\n\\n\\n# Monoisotopic masses (most abundant isotope) and the electron mass.\\nELEMENT_MASSES: dict[str, float] = {\\n    \\"H\\": 1.00782503224,\\n    \\"D\\": 2.01410177811,\\n    \\"C\\": 12.0,\\n    \\"N\\": 14.00307400443,\\n    \\"O\\": 15.99491461957,\\n    \\"F\\": 18.99840316273,\\n    \\"Na\\": 22.9897692820,\\n    \\"Mg\\": 23.985041697,\\n    \\"Si\\": 27.97692653465,\\n    \\"P\\": 30.97376199842,\\n    \\"S\\": 31.9720711744,\\n    \\"Cl\\": 34.968852682,\\n    \\"K\\": 38.9637064864,\\n    \\"Ca\\": 39.962590863,\\n    \\"Fe\\": 55.93493633,\\n    \\"Cu\\": 62.92959772,\\n    \\"Zn\\": 63.92914201,\\n    \\"Br\\": 78.9183376,\\n    \\"I\\": 126.9044719,\\n    \\"Li\\": 7.0160034366,\\n    \\"B\\": 11.009305167,\\n    \\"Se\\": 79.9165218,\\n    \\"As\\": 74.92159457,\\n    \\"e\\": 0.000548579909065,\\n}\\n\\nELECTRON_MASS: float = ELEMENT_MASSES[\\"e\\"]\\nPROTON_MASS: float = ELEMENT_MASSES[\\"H\\"] - ELECTRON_MASS\\n\\n\\ndef _default_data_dir() -> Path:\\n    here = Path(__file__).resolve().parent.parent\\n    local = here / \\"data\\"\\n    kaggle_candidates = [\\n        Path(\\"/kaggle/input/enveda-casmi26-molecule-id-mass-spectra\\"),\\n        Path(\\"/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra\\"),\\n    ]\\n    for cand in kaggle_candidates:\\n        if cand.exists():\\n            return cand\\n    if (local / \\"train.parquet\\").exists() or (local / \\"test.parquet\\").exists():\\n        return local\\n    bundled = here / \\"enveda-CASMI26-molecule-id-mass-spectra\\"\\n    if bundled.exists():\\n        return bundled\\n    return local\\n\\n\\n@dataclass\\nclass Config:\\n    \\"\\"\\"All pipeline hyperparameters. Mutate a copy rather than editing callers.\\"\\"\\"\\n\\n    # Paths\\n    data_dir: Path = field(default_factory=_default_data_dir)\\n    artifact_dir: Path = field(\\n        default_factory=lambda: Path(__file__).resolve().parent.parent / \\"artifacts\\"\\n    )\\n\\n    # Fixed spectral shapes (XLA-static)\\n    top_n_peaks: int = 128\\n    n_mz_bins: int = 512\\n    mz_bin_min: float = 0.0\\n    mz_bin_max: float = 600.0\\n    fp_bits: int = 2048\\n    morgan_radius: int = 2\\n\\n    # Peak cleaning\\n    precursor_peak_margin_da: float = 2.0\\n    min_rel_intensity: float = 0.005  # 0.5% of base peak\\n    min_mz: float = 1.0\\n\\n    # Mass filter\\n    mass_ppm: float = 15.0\\n    mass_abs_da: float = 0.02\\n    mass_fallback_da: tuple[float, ...] = (0.05, 0.2, 0.5, 2.0)\\n    max_mass_candidates: int = 4096\\n\\n    # Spectral similarity\\n    cosine_mz_tol: float = 0.05\\n    modified_cosine_mz_tol: float = 0.05\\n\\n    # Ranker\\n    w_spectral: float = 0.42\\n    w_fingerprint: float = 0.46\\n    w_mass: float = 0.12\\n    top_k: int = 25\\n    mass_score_ppm_scale: float = 10.0\\n    # V6: calibrated OR-merge below the cosine lock; 3-seed logits; neighbor fp.\\n    use_soft_merge: bool = True\\n    soft_merge_lock_cosine: float = 0.75\\n    fp_soft_mix: float = 0.5\\n    neighbor_k: int = 20\\n    neighbor_blend: float = 0.5\\n    use_neighbor_fp: bool = True\\n    use_mass_shift: bool = True\\n    ensemble_seeds: int = 3\\n\\n    # Model\\n    d_model: int = 256\\n    n_heads: int = 8\\n    n_transformer_layers: int = 3\\n    n_conv_channels: int = 128\\n    adduct_embed_dim: int = 32\\n    dropout: float = 0.10\\n    precursor_feat_dim: int = 8\\n\\n    # Training\\n    batch_size: int = 128\\n    num_epochs: int = 8\\n    learning_rate: float = 1e-3\\n    weight_decay: float = 1e-4\\n    max_grad_norm: float = 1.0\\n    focal_gamma: float = 1.5\\n    bce_pos_weight: float = 6.0\\n    num_workers: int = 0\\n    seed: int = 42\\n    val_fraction: float = 0.05\\n    max_train_spectra: int = 400_000\\n    # Single experiment vs 0.139: keep the representative spectrum\'s observed\\n    # neutral losses, precursor/CE/ion features, and adduct ID during training.\\n    use_representative_metadata: bool = True\\n    log_every: int = 50\\n    ckpt_name: str = \\"spec2fp.pt\\"\\n    train_time_limit_s: float = 6.0 * 3600.0  # leave headroom inside the 9 h Kaggle cap\\n\\n    # Inference\\n    fp_threshold: float = 0.35\\n    min_candidates: int = 25\\n\\n    # Prefix-conditioned SMILES decoder (Class 3 / de novo)\\n    smiles_max_len: int = 128\\n    smiles_prefix_len: int = 8\\n    smiles_d_model: int = 256\\n    smiles_nhead: int = 8\\n    smiles_num_layers: int = 4\\n    smiles_dim_feedforward: int = 1024\\n    smiles_dropout: float = 0.1\\n    smiles_label_smoothing: float = 0.05\\n    smiles_num_epochs: int = 6\\n    smiles_batch_size: int = 128\\n    smiles_lr: float = 1e-3\\n    smiles_train_time_limit_s: float = 1.0 * 3600.0\\n    smiles_randomize: bool = True\\n    smiles_num_samples: int = 100\\n    smiles_temperature: float = 0.75\\n    smiles_top_p: float = 0.90\\n    smiles_gen_max_len: int = 100\\n    smiles_mass_ppm: float = 20.0\\n    smiles_ckpt_name: str = \\"smiles_decoder.pt\\"\\n    smiles_vocab_name: str = \\"smiles_vocab.json\\"\\n    # Optional offline PubChem / COCONUT table (parquet/csv) for Class 2.\\n    external_candidates_path: Path | None = None\\n\\n    @property\\n    def train_path(self) -> Path:\\n        return self.data_dir / \\"train.parquet\\"\\n\\n    @property\\n    def test_path(self) -> Path:\\n        return self.data_dir / \\"test.parquet\\"\\n\\n    @property\\n    def sample_submission_path(self) -> Path:\\n        return self.data_dir / \\"sample_submission.csv\\"\\n\\n    @property\\n    def checkpoint_path(self) -> Path:\\n        return self.artifact_dir / self.ckpt_name\\n\\n    @property\\n    def smiles_checkpoint_path(self) -> Path:\\n        return self.artifact_dir / self.smiles_ckpt_name\\n\\n    @property\\n    def smiles_vocab_path(self) -> Path:\\n        return self.artifact_dir / self.smiles_vocab_name\\n\\n    @property\\n    def mz_bin_width(self) -> float:\\n        return (self.mz_bin_max - self.mz_bin_min) / float(self.n_mz_bins)\\n\\n\\n# Closed vocabulary of adducts seen in train (121 unique) plus a few extras.\\n# Unknown adducts are hashed into the last bucket.\\nADDUCT_VOCAB: tuple[str, ...] = (\\n    \\"[M+H]+\\",\\n    \\"[M-H]-\\",\\n    \\"[M+Na]+\\",\\n    \\"[M+K]+\\",\\n    \\"[M+NH4]+\\",\\n    \\"[M+Cl]-\\",\\n    \\"[M+CH2O2-H]-\\",\\n    \\"[2M+H]+\\",\\n    \\"[2M+Na]+\\",\\n    \\"[2M-H]-\\",\\n    \\"[2M+NH4]+\\",\\n    \\"[2M+K]+\\",\\n    \\"[2M+CH2O2-H]-\\",\\n    \\"[2M+C2H4O2-H]-\\",\\n    \\"[2M+Na-2H]-\\",\\n    \\"[M-H2O+H]+\\",\\n    \\"[M-2H2O+H]+\\",\\n    \\"[M]+\\",\\n    \\"[M]-\\",\\n    \\"[M+C2H4O2-H]-\\",\\n    \\"[M+2H]2+\\",\\n    \\"[M+3H]3+\\",\\n    \\"[M+H3O]+\\",\\n    \\"[M-H2O]+\\",\\n    \\"[M-H2O-H]-\\",\\n    \\"[M-CH3]-\\",\\n    \\"[M-2H]-\\",\\n    \\"[M+2Na-H]+\\",\\n    \\"[M+Ca]2+\\",\\n    \\"[M+Br]-\\",\\n    \\"[M+Li]+\\",\\n    \\"[M+C2F3O2]-\\",\\n    \\"[M+C2H4N]+\\",\\n    \\"[M-H5O3]+\\",\\n    \\"[3M+H]+\\",\\n    \\"[3M-H]-\\",\\n    \\"[3M+Na]+\\",\\n    \\"UNKNOWN\\",\\n)\\n\\nADDUCT2ID: dict[str, int] = {a: i for i, a in enumerate(ADDUCT_VOCAB)}\\nNUM_ADDUCTS: int = len(ADDUCT_VOCAB)\\n\\n\\ndef adduct_to_id(adduct: str | None) -> int:\\n    if adduct is None:\\n        return ADDUCT2ID[\\"UNKNOWN\\"]\\n    return ADDUCT2ID.get(str(adduct).strip(), ADDUCT2ID[\\"UNKNOWN\\"])\\n\\n\\ndef get_config(**overrides) -> Config:\\n    cfg = Config()\\n    for k, v in overrides.items():\\n        if not hasattr(cfg, k):\\n            raise AttributeError(f\\"Unknown config field: {k}\\")\\n        setattr(cfg, k, v)\\n    cfg.artifact_dir.mkdir(parents=True, exist_ok=True)\\n    return cfg\\n", "src/chem.py": "\\"\\"\\"Chemistry utilities: formula mass, adduct parsing, RDKit fingerprints, InChIKey14.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\nfrom dataclasses import dataclass\\nfrom functools import lru_cache\\nfrom typing import Iterable\\n\\nimport numpy as np\\n\\nfrom src.config import ELEMENT_MASSES, ELECTRON_MASS, Config\\n\\ntry:\\n    from rdkit import Chem\\n    from rdkit.Chem import rdFingerprintGenerator\\n    from rdkit import RDLogger\\n\\n    RDLogger.DisableLog(\\"rdApp.*\\")\\n    _HAS_RDKIT = True\\n    _MORGAN_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)\\nexcept Exception:  # pragma: no cover - environment without RDKit\\n    Chem = None  # type: ignore\\n    rdFingerprintGenerator = None  # type: ignore\\n    _HAS_RDKIT = False\\n    _MORGAN_GEN = None\\n\\n\\n_FORMULA_TOKEN = re.compile(r\\"([A-Z][a-z]?)(\\\\d*)\\")\\n_ADDUCT_RE = re.compile(\\n    r\\"^\\\\[(\\\\d*)M(?:([A-Za-z0-9]*)?)?(.*?)\\\\](\\\\d*)([+-])$\\"\\n)\\n_EXTRA_TOKEN = re.compile(r\\"([+-])([A-Za-z0-9]+)\\")\\n\\n\\n@dataclass(frozen=True, slots=True)\\nclass AdductInfo:\\n    \\"\\"\\"Parsed adduct used to convert precursor m/z into neutral monoisotopic mass.\\"\\"\\"\\n\\n    raw: str\\n    n_molecules: int\\n    charge: int  # signed\\n    extra_neutral_mass: float\\n    delta: float  # ion-mass offset: extra_neutral - signed_charge * m_e\\n\\n    @property\\n    def abs_charge(self) -> int:\\n        return max(abs(self.charge), 1)\\n\\n\\ndef parse_formula(formula: str | None) -> dict[str, int]:\\n    \\"\\"\\"Parse a Hill-system formula such as ``C21H30N7O17P3`` into element counts.\\"\\"\\"\\n    if not formula:\\n        return {}\\n    s = str(formula).strip().replace(\\" \\", \\"\\")\\n    if not s or s.lower() in {\\"nan\\", \\"none\\"}:\\n        return {}\\n    counts: dict[str, int] = {}\\n    idx = 0\\n    while idx < len(s):\\n        m = _FORMULA_TOKEN.match(s, idx)\\n        if m is None:\\n            # Skip leftover punctuation rather than aborting the whole formula.\\n            idx += 1\\n            continue\\n        el, num = m.group(1), m.group(2)\\n        counts[el] = counts.get(el, 0) + (int(num) if num else 1)\\n        idx = m.end()\\n    return counts\\n\\n\\nFORMULA_ELEMENTS: tuple[str, ...] = (\\"C\\", \\"H\\", \\"N\\", \\"O\\", \\"P\\", \\"S\\", \\"Cl\\", \\"Br\\")\\n\\n\\ndef formula_count_vector(formula: str | None) -> np.ndarray:\\n    \\"\\"\\"CHNOPS + Cl/Br counts as float32 length 8.\\"\\"\\"\\n    counts = parse_formula(formula)\\n    return np.array([float(counts.get(el, 0)) for el in FORMULA_ELEMENTS], dtype=np.float32)\\n\\n\\ndef formula_match_score(pred_counts: np.ndarray, formula: str | None) -> float:\\n    \\"\\"\\"Soft agreement in (0, 1] between predicted element counts and a formula.\\"\\"\\"\\n    if not formula:\\n        return 1.0\\n    cand = formula_count_vector(formula)\\n    pred = np.asarray(pred_counts, dtype=np.float32).ravel()\\n    if pred.size < 4:\\n        return 1.0\\n    dc = abs(float(pred[0]) - float(cand[0]))\\n    dh = abs(float(pred[1]) - float(cand[1])) if pred.size > 1 else 0.0\\n    dn = abs(float(pred[2]) - float(cand[2])) if pred.size > 2 else 0.0\\n    do = abs(float(pred[3]) - float(cand[3])) if pred.size > 3 else 0.0\\n    return float(np.exp(-0.5 * ((dc / 3.0) ** 2 + (dh / 8.0) ** 2 + (dn / 1.5) ** 2 + (do / 2.0) ** 2)))\\n\\n\\ndef formula_to_mass(formula: str | None) -> float:\\n    \\"\\"\\"Monoisotopic mass from a molecular formula. Unknown elements contribute 0.\\"\\"\\"\\n    counts = parse_formula(formula)\\n    mass = 0.0\\n    for el, n in counts.items():\\n        mass += ELEMENT_MASSES.get(el, 0.0) * n\\n    return float(mass)\\n\\n\\ndef _formula_mass_with_leading_count(token: str) -> float:\\n    \\"\\"\\"Parse ``2H``, ``H2O``, ``Na``, ``CH2O2`` into a monoisotopic mass.\\"\\"\\"\\n    if not token:\\n        return 0.0\\n    m = re.match(r\\"^(\\\\d+)([A-Z].*)$\\", token)\\n    if m:\\n        return int(m.group(1)) * _formula_mass_with_leading_count(m.group(2))\\n    return formula_to_mass(token)\\n\\n\\n@lru_cache(maxsize=512)\\ndef parse_adduct(adduct: str | None) -> AdductInfo:\\n    \\"\\"\\"Parse strings like ``[M+H]+``, ``[2M+Na]+``, ``[M+CH2O2-H]-``, ``[M+2H]2+``.\\"\\"\\"\\n    raw = \\"\\" if adduct is None else str(adduct).strip()\\n    if not raw:\\n        return AdductInfo(raw=raw, n_molecules=1, charge=1, extra_neutral_mass=0.0, delta=0.0)\\n\\n    m = _ADDUCT_RE.match(raw)\\n    if m is None:\\n        # Best-effort: assume [M+H]+ so downstream math stays finite.\\n        proton = ELEMENT_MASSES[\\"H\\"] - ELECTRON_MASS\\n        return AdductInfo(\\n            raw=raw, n_molecules=1, charge=1, extra_neutral_mass=ELEMENT_MASSES[\\"H\\"], delta=proton\\n        )\\n\\n    n_str, _bridge, extras, z_str, sign = m.groups()\\n    n_molecules = int(n_str) if n_str else 1\\n    abs_z = int(z_str) if z_str else 1\\n    signed_z = abs_z if sign == \\"+\\" else -abs_z\\n\\n    extra_neutral = 0.0\\n    extra_str = extras or \\"\\"\\n    if extra_str:\\n        tokens = _EXTRA_TOKEN.findall(extra_str)\\n        if not tokens and extra_str:\\n            # e.g. unexpected leftover; try as a leading-sign-less formula.\\n            extra_neutral += _formula_mass_with_leading_count(extra_str)\\n        for sgn, tok in tokens:\\n            mass = _formula_mass_with_leading_count(tok)\\n            extra_neutral += mass if sgn == \\"+\\" else -mass\\n\\n    delta = extra_neutral - signed_z * ELECTRON_MASS\\n    return AdductInfo(\\n        raw=raw,\\n        n_molecules=max(n_molecules, 1),\\n        charge=signed_z,\\n        extra_neutral_mass=float(extra_neutral),\\n        delta=float(delta),\\n    )\\n\\n\\ndef precursor_to_neutral_mass(precursor_mz: float, adduct: str | None) -> float:\\n    \\"\\"\\"Neutral monoisotopic mass from precursor m/z and adduct annotation.\\n\\n    ``mz = (n * M + extra_neutral - z * m_e) / |z|``\\n    \\"\\"\\"\\n    info = parse_adduct(adduct)\\n    mz = float(precursor_mz)\\n    if not np.isfinite(mz) or mz <= 0:\\n        return 0.0\\n    mass = (mz * info.abs_charge - info.delta) / float(info.n_molecules)\\n    if not np.isfinite(mass) or mass <= 0:\\n        return 0.0\\n    return float(mass)\\n\\n\\n# mz \\u2192 monoisotopic M for |z|=1, n=1:  M = mz + offset.\\n# Values are (extra_neutral \\u2212 z\\u00b7m_e) with opposite sign so they match\\n# the CASMI / ESI convention to \\u22655 decimal places.\\nADDUCT_OFFSETS: dict[str, float] = {\\n    \\"[M+H]+\\": -1.007276,\\n    \\"[M+NH4]+\\": -18.033826,\\n    \\"[M+Na]+\\": -22.989218,\\n    \\"[M+K]+\\": -38.963158,\\n    \\"[M-H]-\\": +1.007276,\\n    \\"[M+CH2O2-H]-\\": -44.998201,\\n    \\"[M+Cl]-\\": -34.969402,\\n    \\"[M+C2H4O2-H]-\\": -59.013851,\\n    \\"[M+Br]-\\": -78.918885,\\n}\\n\\n\\n# Tight Class 1 set. Extra adducts open extra \\u00b115 ppm windows and let\\n# wrong-mass isomers outrank the true skeleton on noisy cosine.\\nPOS_ADDUCTS: tuple[str, ...] = (\\n    \\"[M+H]+\\",\\n    \\"[M+Na]+\\",\\n)\\nNEG_ADDUCTS: tuple[str, ...] = (\\n    \\"[M-H]-\\",\\n    \\"[M+CH2O2-H]-\\",\\n)\\n# Class 2 has no MS2 library, so probe the common ESI alternatives.\\nCLASS2_POS_ADDUCTS: tuple[str, ...] = (\\n    \\"[M+H]+\\",\\n    \\"[M+Na]+\\",\\n    \\"[M+NH4]+\\",\\n    \\"[M+K]+\\",\\n)\\nCLASS2_NEG_ADDUCTS: tuple[str, ...] = (\\n    \\"[M-H]-\\",\\n    \\"[M+CH2O2-H]-\\",\\n    \\"[M+C2H4O2-H]-\\",\\n    \\"[M+Cl]-\\",\\n)\\nCOMMON_ADDUCTS: tuple[str, ...] = CLASS2_POS_ADDUCTS + CLASS2_NEG_ADDUCTS\\n\\n# 37Cl \\u2212 35Cl and 81Br \\u2212 79Br (monoisotopic minor vs major halogen adduct).\\n_CL37_MINUS_CL35: float = 36.96590258 - 34.968852682\\n_BR81_MINUS_BR79: float = 80.9162897 - 78.9183376\\n\\n\\ndef adduct_offset(adduct: str | None) -> float:\\n    \\"\\"\\"Return ``M - mz`` for a singly charged 1M ion (0 if unknown).\\"\\"\\"\\n    raw = \\"\\" if adduct is None else str(adduct).strip()\\n    if raw in ADDUCT_OFFSETS:\\n        return float(ADDUCT_OFFSETS[raw])\\n    info = parse_adduct(raw)\\n    if info.n_molecules != 1 or info.abs_charge != 1:\\n        return 0.0\\n    return float(-info.delta)\\n\\n\\ndef plausible_neutral_masses(\\n    precursor_mz: float,\\n    adduct: str | None = None,\\n    *,\\n    ionization_mode: str | None = None,\\n    exhaustive: bool = False,\\n) -> list[float]:\\n    \\"\\"\\"Neutral masses under the annotated adduct and ion-mode-matched alternatives.\\n\\n    ``exhaustive=True`` also probes the opposite polarity and halogen isotopes.\\n    The tight Class 1 window should use the default (conservative) set.\\n    \\"\\"\\"\\n    seen: set[float] = set()\\n    out: list[float] = []\\n\\n    def _add(mass: float) -> None:\\n        if mass <= 0 or not np.isfinite(mass):\\n            return\\n        key = round(mass, 4)\\n        if key in seen:\\n            return\\n        seen.add(key)\\n        out.append(float(mass))\\n\\n    mz = float(precursor_mz)\\n    ion = str(ionization_mode or \\"\\").lower()\\n    if exhaustive:\\n        pos_set, neg_set = CLASS2_POS_ADDUCTS, CLASS2_NEG_ADDUCTS\\n    else:\\n        pos_set, neg_set = POS_ADDUCTS, NEG_ADDUCTS\\n    if ion.startswith(\\"neg\\"):\\n        mode_adducts: tuple[str, ...] = neg_set\\n    elif ion.startswith(\\"pos\\"):\\n        mode_adducts = pos_set\\n    else:\\n        mode_adducts = pos_set + neg_set\\n    probes = (adduct, *mode_adducts)\\n    if exhaustive:\\n        probes = (adduct, *CLASS2_POS_ADDUCTS, *CLASS2_NEG_ADDUCTS, \\"[M+Br]-\\", \\"[M-H2O+H]+\\", \\"[M-H2O-H]-\\")\\n    for ad in probes:\\n        if not ad:\\n            continue\\n        if ad in ADDUCT_OFFSETS:\\n            _add(mz + ADDUCT_OFFSETS[ad])\\n        _add(precursor_to_neutral_mass(mz, ad))\\n    if exhaustive or (adduct and \\"Cl\\" in str(adduct)):\\n        _add(precursor_to_neutral_mass(mz, \\"[M+Cl]-\\") - _CL37_MINUS_CL35)\\n    if exhaustive or (adduct and \\"Br\\" in str(adduct)):\\n        _add(precursor_to_neutral_mass(mz, \\"[M+Br]-\\") - _BR81_MINUS_BR79)\\n    return out\\n\\n\\ndef smiles_to_mol(smiles: str | None):\\n    if not _HAS_RDKIT or not smiles:\\n        return None\\n    try:\\n        mol = Chem.MolFromSmiles(str(smiles))\\n    except Exception:\\n        return None\\n    return mol\\n\\n\\ndef inchikey14_from_smiles(smiles: str | None) -> str:\\n    \\"\\"\\"First InChIKey block (14 chars) after RDKit canonicalization. Empty on failure.\\"\\"\\"\\n    mol = smiles_to_mol(smiles)\\n    if mol is None:\\n        return \\"\\"\\n    try:\\n        key = Chem.MolToInchiKey(mol)\\n    except Exception:\\n        return \\"\\"\\n    if not key:\\n        return \\"\\"\\n    return key.split(\\"-\\")[0][:14]\\n\\n\\ndef canonical_smiles(smiles: str | None) -> str:\\n    mol = smiles_to_mol(smiles)\\n    if mol is None:\\n        return \\"\\" if not smiles else str(smiles)\\n    try:\\n        return Chem.MolToSmiles(mol, canonical=True)\\n    except Exception:\\n        return str(smiles)\\n\\n\\ndef morgan_fingerprint(smiles: str | None, n_bits: int = 2048, radius: int = 2) -> np.ndarray:\\n    \\"\\"\\"Return a uint8 vector of shape ``(n_bits,)``. Zeros if SMILES cannot be parsed.\\"\\"\\"\\n    fp = np.zeros(n_bits, dtype=np.uint8)\\n    if not _HAS_RDKIT or not smiles:\\n        return fp\\n    mol = smiles_to_mol(smiles)\\n    if mol is None:\\n        return fp\\n    try:\\n        if n_bits == 2048 and radius == 2 and _MORGAN_GEN is not None:\\n            bitvect = _MORGAN_GEN.GetFingerprint(mol)\\n        else:\\n            gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)\\n            bitvect = gen.GetFingerprint(mol)\\n        on_bits = np.asarray(list(bitvect.GetOnBits()), dtype=np.int32)\\n        on_bits = on_bits[on_bits < n_bits]\\n        fp[on_bits] = 1\\n        return fp\\n    except Exception:\\n        return fp\\n\\n\\ndef morgan_fingerprint_batch(smiles_list: Iterable[str], cfg: Config | None = None) -> np.ndarray:\\n    n_bits = 2048 if cfg is None else cfg.fp_bits\\n    radius = 2 if cfg is None else cfg.morgan_radius\\n    smiles_list = list(smiles_list)\\n    out = np.zeros((len(smiles_list), n_bits), dtype=np.uint8)\\n    for i, smi in enumerate(smiles_list):\\n        out[i] = morgan_fingerprint(smi, n_bits=n_bits, radius=radius)\\n    return out\\n\\n\\ndef pack_fingerprints(fp: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Pack bit last-axis to uint8 bytes.\\"\\"\\"\\n    return np.packbits(fp.astype(np.uint8, copy=False), axis=-1)\\n\\n\\ndef unpack_fingerprints(packed: np.ndarray, n_bits: int) -> np.ndarray:\\n    bits = np.unpackbits(packed, axis=-1)\\n    return bits[..., :n_bits]\\n\\n\\ndef has_rdkit() -> bool:\\n    return bool(_HAS_RDKIT)\\n\\n\\ndef randomize_smiles(smiles: str | None, attempts: int = 8) -> str:\\n    \\"\\"\\"Non-canonical SMILES traversal; falls back to the input string.\\"\\"\\"\\n    mol = smiles_to_mol(smiles)\\n    if mol is None:\\n        return \\"\\" if not smiles else str(smiles)\\n    for _ in range(max(int(attempts), 1)):\\n        try:\\n            out = Chem.MolToSmiles(mol, canonical=False, doRandom=True, isomericSmiles=True)\\n            if out:\\n                return out\\n        except Exception:\\n            continue\\n    return canonical_smiles(smiles)\\n\\n\\ndef exact_mass_from_smiles(smiles: str | None) -> float:\\n    mol = smiles_to_mol(smiles)\\n    if mol is None:\\n        return 0.0\\n    try:\\n        from rdkit.Chem import Descriptors\\n\\n        return float(Descriptors.ExactMolWt(mol))\\n    except Exception:\\n        return 0.0\\n", "src/preprocessing.py": "\\"\\"\\"Peak cleaning, neutral-loss features, and fixed-size TPU tensors.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any, Iterable, Sequence\\n\\nimport numpy as np\\n\\nfrom src.chem import precursor_to_neutral_mass\\nfrom src.config import Config, adduct_to_id\\n\\n\\ndef _as_1d_float(x: Any) -> np.ndarray:\\n    \\"\\"\\"Coerce scalars, 1-D arrays, or ragged nested lists/arrays to float32.\\"\\"\\"\\n    if x is None:\\n        return np.zeros((0,), dtype=np.float32)\\n    if isinstance(x, np.ndarray) and x.dtype != object and x.dtype.kind in \\"fiu\\":\\n        return np.ascontiguousarray(x, dtype=np.float32).reshape(-1)\\n    if isinstance(x, (bool, np.bool_)):\\n        return np.zeros((0,), dtype=np.float32)\\n    if isinstance(x, (int, float, np.integer, np.floating)):\\n        return np.array([float(x)], dtype=np.float32)\\n\\n    out: list[float] = []\\n\\n    def walk(v: Any) -> None:\\n        if v is None:\\n            return\\n        if isinstance(v, (bool, np.bool_)):\\n            return\\n        if isinstance(v, (int, float, np.integer, np.floating)):\\n            out.append(float(v))\\n            return\\n        if isinstance(v, np.ndarray):\\n            if v.size == 0:\\n                return\\n            if v.dtype != object and v.dtype.kind in \\"fiu\\":\\n                out.extend(np.ascontiguousarray(v, dtype=np.float32).reshape(-1).tolist())\\n                return\\n            for item in v.reshape(-1):\\n                walk(item)\\n            return\\n        if isinstance(v, (bytes, str)):\\n            s = v.decode(\\"utf-8\\", \\"ignore\\") if isinstance(v, bytes) else v\\n            s = s.strip()\\n            if not s or s.lower() in {\\"nan\\", \\"none\\", \\"null\\"}:\\n                return\\n            parts = [p.strip() for p in s.replace(\\";\\", \\",\\").split(\\",\\") if p.strip()]\\n            if len(parts) > 1:\\n                for p in parts:\\n                    walk(p)\\n                return\\n            try:\\n                out.append(float(s))\\n            except ValueError:\\n                return\\n            return\\n        if isinstance(v, dict):\\n            return\\n        if isinstance(v, (list, tuple, set)):\\n            for item in v:\\n                walk(item)\\n            return\\n        if hasattr(v, \\"tolist\\"):\\n            try:\\n                walk(v.tolist())\\n                return\\n            except Exception:\\n                pass\\n        try:\\n            out.append(float(v))\\n        except (TypeError, ValueError):\\n            return\\n\\n    walk(x)\\n    if not out:\\n        return np.zeros((0,), dtype=np.float32)\\n    return np.asarray(out, dtype=np.float32)\\n\\n\\ndef mean_collision_energy(ce: Any) -> float:\\n    try:\\n        arr = _as_1d_float(ce)\\n        arr = arr[np.isfinite(arr)]\\n        if arr.size == 0:\\n            return 0.0\\n        return float(np.mean(np.abs(arr)))\\n    except Exception:\\n        return 0.0\\n\\n\\ndef clean_peaks(\\n    mz: np.ndarray,\\n    intensity: np.ndarray,\\n    precursor_mz: float,\\n    *,\\n    cfg: Config,\\n) -> tuple[np.ndarray, np.ndarray]:\\n    \\"\\"\\"Drop precursor-region / tiny peaks and L2-renormalize intensities.\\n\\n    Rules:\\n      * ``mz`` in ``(min_mz, precursor_mz + margin]``\\n      * intensity >= ``min_rel_intensity * max(intensity)``\\n    \\"\\"\\"\\n    mz = _as_1d_float(mz)\\n    intensity = _as_1d_float(intensity)\\n    n = min(mz.size, intensity.size)\\n    mz = mz[:n]\\n    intensity = intensity[:n]\\n    if n == 0:\\n        return mz, intensity\\n\\n    finite = np.isfinite(mz) & np.isfinite(intensity) & (intensity > 0)\\n    mz = mz[finite]\\n    intensity = intensity[finite]\\n    if mz.size == 0:\\n        return mz, intensity\\n\\n    prec = float(precursor_mz) if np.isfinite(precursor_mz) else 1e9\\n    keep = (mz >= cfg.min_mz) & (mz <= prec + cfg.precursor_peak_margin_da)\\n    mz = mz[keep]\\n    intensity = intensity[keep]\\n    if mz.size == 0:\\n        return mz, intensity\\n\\n    base = float(np.max(intensity))\\n    if base > 0:\\n        intensity = intensity / base\\n        keep = intensity >= cfg.min_rel_intensity\\n        mz = mz[keep]\\n        intensity = intensity[keep]\\n    if mz.size == 0:\\n        return mz, intensity\\n\\n    base = float(np.max(intensity))\\n    if base > 0:\\n        intensity = intensity / base\\n    return mz.astype(np.float32, copy=False), intensity.astype(np.float32, copy=False)\\n\\n\\ndef top_n_pad(\\n    mz: np.ndarray,\\n    intensity: np.ndarray,\\n    precursor_mz: float,\\n    *,\\n    cfg: Config,\\n) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:\\n    \\"\\"\\"Return fixed-length ``(top_n,)`` arrays: mz, intensity, neutral loss, mask.\\"\\"\\"\\n    n = cfg.top_n_peaks\\n    out_mz = np.zeros(n, dtype=np.float32)\\n    out_int = np.zeros(n, dtype=np.float32)\\n    out_nl = np.zeros(n, dtype=np.float32)\\n    out_mask = np.zeros(n, dtype=np.float32)\\n    if mz.size == 0:\\n        return out_mz, out_int, out_nl, out_mask\\n\\n    k = min(n, int(mz.size))\\n    if mz.size > n:\\n        idx = np.argpartition(intensity, -k)[-k:]\\n        idx = idx[np.argsort(mz[idx])]\\n    else:\\n        idx = np.argsort(mz)\\n        k = int(idx.size)\\n    sel_mz = mz[idx][:k]\\n    sel_int = intensity[idx][:k]\\n    # Re-normalize after top-N so cosine is well scaled.\\n    s = float(np.max(sel_int)) if k else 0.0\\n    if s > 0:\\n        sel_int = sel_int / s\\n    prec = float(precursor_mz) if np.isfinite(precursor_mz) else 0.0\\n    nl = np.clip(prec - sel_mz, 0.0, None).astype(np.float32)\\n    out_mz[:k] = sel_mz\\n    out_int[:k] = sel_int\\n    out_nl[:k] = nl\\n    out_mask[:k] = 1.0\\n    return out_mz, out_int, out_nl, out_mask\\n\\n\\ndef bin_spectrum(\\n    mz: np.ndarray,\\n    intensity: np.ndarray,\\n    mask: np.ndarray | None = None,\\n    *,\\n    cfg: Config,\\n) -> np.ndarray:\\n    \\"\\"\\"Max-pool intensities into a fixed ``(n_mz_bins,)`` histogram (XLA-static).\\"\\"\\"\\n    out = np.zeros(cfg.n_mz_bins, dtype=np.float32)\\n    mz = _as_1d_float(mz)\\n    intensity = _as_1d_float(intensity)\\n    n = min(mz.size, intensity.size)\\n    if n == 0:\\n        return out\\n    mz = mz[:n]\\n    intensity = intensity[:n]\\n    if mask is not None:\\n        m = _as_1d_float(mask)[:n] > 0\\n        mz = mz[m]\\n        intensity = intensity[m]\\n    if mz.size == 0:\\n        return out\\n    width = cfg.mz_bin_width\\n    if width <= 0:\\n        return out\\n    bins = np.floor((mz - cfg.mz_bin_min) / width).astype(np.int32)\\n    valid = (bins >= 0) & (bins < cfg.n_mz_bins) & np.isfinite(intensity)\\n    if not np.any(valid):\\n        return out\\n    np.maximum.at(out, bins[valid], intensity[valid])\\n    peak = float(out.max())\\n    if peak > 0:\\n        out /= peak\\n    return out\\n\\n\\ndef merge_peak_lists(\\n    peak_lists: Sequence[tuple[np.ndarray, np.ndarray]],\\n    precursor_mz: float,\\n    *,\\n    cfg: Config,\\n    bin_da: float = 0.01,\\n) -> tuple[np.ndarray, np.ndarray]:\\n    \\"\\"\\"Merge several cleaned spectra by max-pooling into ``bin_da`` buckets.\\"\\"\\"\\n    if not peak_lists:\\n        return np.zeros((0,), dtype=np.float32), np.zeros((0,), dtype=np.float32)\\n    all_mz = []\\n    all_int = []\\n    for mz, inten in peak_lists:\\n        mz = _as_1d_float(mz)\\n        inten = _as_1d_float(inten)\\n        if mz.size:\\n            all_mz.append(mz)\\n            all_int.append(inten)\\n    if not all_mz:\\n        return np.zeros((0,), dtype=np.float32), np.zeros((0,), dtype=np.float32)\\n    mz = np.concatenate(all_mz)\\n    inten = np.concatenate(all_int)\\n    mz, inten = clean_peaks(mz, inten, precursor_mz, cfg=cfg)\\n    if mz.size == 0:\\n        return mz, inten\\n    keys = np.round(mz / bin_da).astype(np.int64)\\n    order = np.argsort(keys)\\n    keys = keys[order]\\n    mz = mz[order]\\n    inten = inten[order]\\n    uniq, start = np.unique(keys, return_index=True)\\n    # max intensity per bin, mz = intensity-weighted mean\\n    merged_mz = np.empty(uniq.size, dtype=np.float32)\\n    merged_int = np.empty(uniq.size, dtype=np.float32)\\n    start = np.append(start, keys.size)\\n    for i in range(uniq.size):\\n        sl = slice(int(start[i]), int(start[i + 1]))\\n        w = inten[sl]\\n        merged_int[i] = float(w.max())\\n        sw = float(w.sum())\\n        merged_mz[i] = float((mz[sl] * w).sum() / sw) if sw > 0 else float(mz[sl].mean())\\n    peak = float(merged_int.max()) if merged_int.size else 0.0\\n    if peak > 0:\\n        merged_int = merged_int / peak\\n    return merged_mz, merged_int\\n\\n\\ndef precursor_feature_vector(\\n    *,\\n    precursor_mz: float,\\n    neutral_mass: float,\\n    adduct: str | None,\\n    collision_energy: Any,\\n    ionization_mode: str | None,\\n    n_peaks: int,\\n    cfg: Config,\\n) -> np.ndarray:\\n    \\"\\"\\"Fixed-length precursor / metadata features (length ``cfg.precursor_feat_dim``).\\"\\"\\"\\n    from src.chem import parse_adduct\\n\\n    info = parse_adduct(adduct)\\n    ion = 1.0 if str(ionization_mode or \\"\\").lower().startswith(\\"pos\\") else 0.0\\n    ce = mean_collision_energy(collision_energy)\\n    feat = np.array(\\n        [\\n            float(neutral_mass) / 1000.0,\\n            float(precursor_mz) / 1000.0,\\n            float(np.log1p(max(neutral_mass, 0.0))),\\n            float(info.abs_charge),\\n            float(info.n_molecules),\\n            float(ce) / 200.0,\\n            ion,\\n            float(n_peaks) / float(cfg.top_n_peaks),\\n        ],\\n        dtype=np.float32,\\n    )\\n    if feat.size < cfg.precursor_feat_dim:\\n        feat = np.pad(feat, (0, cfg.precursor_feat_dim - feat.size))\\n    return feat[: cfg.precursor_feat_dim]\\n\\n\\ndef featurize_spectrum(\\n    mz: Any,\\n    intensity: Any,\\n    precursor_mz: float,\\n    adduct: str | None,\\n    *,\\n    cfg: Config,\\n    collision_energy: Any = None,\\n    ionization_mode: str | None = None,\\n) -> dict[str, np.ndarray | int | float]:\\n    \\"\\"\\"Clean one MS2 spectrum into the static tensors consumed by the model.\\"\\"\\"\\n    mz_arr = _as_1d_float(mz)\\n    int_arr = _as_1d_float(intensity)\\n    prec = float(precursor_mz) if precursor_mz is not None else 0.0\\n    mz_c, int_c = clean_peaks(mz_arr, int_arr, prec, cfg=cfg)\\n    if mz_c.size == 0:\\n        # Fall back to unfiltered peaks so we never emit an empty example.\\n        mz_c, int_c = mz_arr, int_arr\\n        finite = np.isfinite(mz_c) & np.isfinite(int_c)\\n        mz_c, int_c = mz_c[finite], int_c[finite]\\n    peak_mz, peak_int, peak_nl, peak_mask = top_n_pad(mz_c, int_c, prec, cfg=cfg)\\n    binned = bin_spectrum(peak_mz, peak_int, peak_mask, cfg=cfg)\\n    neutral = precursor_to_neutral_mass(prec, adduct)\\n    n_peaks = int(peak_mask.sum())\\n    feats = precursor_feature_vector(\\n        precursor_mz=prec,\\n        neutral_mass=neutral,\\n        adduct=adduct,\\n        collision_energy=collision_energy,\\n        ionization_mode=ionization_mode,\\n        n_peaks=n_peaks,\\n        cfg=cfg,\\n    )\\n    return {\\n        \\"peak_mz\\": peak_mz,\\n        \\"peak_intensity\\": peak_int,\\n        \\"peak_nl\\": peak_nl,\\n        \\"peak_mask\\": peak_mask,\\n        \\"binned\\": binned,\\n        \\"precursor_feat\\": feats,\\n        \\"adduct_id\\": int(adduct_to_id(adduct)),\\n        \\"adduct\\": str(adduct or \\"\\"),\\n        \\"ionization_mode\\": str(ionization_mode or \\"\\"),\\n        \\"neutral_mass\\": float(neutral),\\n        \\"precursor_mz\\": float(prec),\\n    }\\n\\n\\ndef featurize_aggregated(\\n    spectra: Sequence[dict[str, Any]],\\n    *,\\n    cfg: Config,\\n    precursor_key: str = \\"precursor_mz\\",\\n    mz_key: str = \\"ms2_mzs\\",\\n    int_key: str = \\"ms2_normalized_intensities\\",\\n    adduct_key: str = \\"adduct\\",\\n    ce_key: str = \\"collision_energy_ev\\",\\n    ion_key: str = \\"ionization_mode\\",\\n) -> dict[str, np.ndarray | int | float]:\\n    \\"\\"\\"Merge a molecule\'s multi-energy spectra (same adduct) by max-intensity peak fusion.\\"\\"\\"\\n    if not spectra:\\n        raise ValueError(\\"featurize_aggregated requires at least one spectrum\\")\\n    # Group by rounded precursor so mixed adducts are not blindly pooled.\\n    best: dict[str, np.ndarray | int | float] | None = None\\n    best_peaks = -1\\n    from collections import defaultdict\\n\\n    groups: dict[tuple[str, int], list[dict[str, Any]]] = defaultdict(list)\\n    for row in spectra:\\n        adduct = str(row.get(adduct_key) or \\"\\")\\n        prec = float(row.get(precursor_key) or 0.0)\\n        groups[(adduct, int(round(prec * 1000.0)))].append(row)\\n\\n    merged_features: list[dict[str, np.ndarray | int | float]] = []\\n    for (adduct, _prec_key), rows in groups.items():\\n        peak_lists = []\\n        prec_vals = []\\n        ces = []\\n        ion = rows[0].get(ion_key)\\n        for row in rows:\\n            prec = float(row.get(precursor_key) or 0.0)\\n            prec_vals.append(prec)\\n            ces.append(row.get(ce_key))\\n            mz_c, int_c = clean_peaks(\\n                _as_1d_float(row.get(mz_key)),\\n                _as_1d_float(row.get(int_key)),\\n                prec,\\n                cfg=cfg,\\n            )\\n            peak_lists.append((mz_c, int_c))\\n        prec_mean = float(np.mean(prec_vals)) if prec_vals else 0.0\\n        mz_m, int_m = merge_peak_lists(peak_lists, prec_mean, cfg=cfg)\\n        feat = featurize_spectrum(\\n            mz_m,\\n            int_m,\\n            prec_mean,\\n            adduct,\\n            cfg=cfg,\\n            collision_energy=ces,\\n            ionization_mode=ion,\\n        )\\n        merged_features.append(feat)\\n        n = int(feat[\\"peak_mask\\"].sum()) if isinstance(feat[\\"peak_mask\\"], np.ndarray) else 0\\n        if n > best_peaks:\\n            best_peaks = n\\n            best = feat\\n\\n    assert best is not None\\n    # Average binned spectra across adduct groups for a molecule-level view.\\n    if len(merged_features) > 1:\\n        stacked = np.stack([f[\\"binned\\"] for f in merged_features], axis=0)\\n        best = dict(best)\\n        best[\\"binned\\"] = stacked.max(axis=0).astype(np.float32)\\n        best[\\"group_features\\"] = merged_features  # type: ignore[assignment]\\n    else:\\n        best = dict(best)\\n        best[\\"group_features\\"] = merged_features  # type: ignore[assignment]\\n    return best\\n\\n\\ndef stack_features(rows: Iterable[dict[str, Any]]) -> dict[str, np.ndarray]:\\n    \\"\\"\\"Stack a list of ``featurize_spectrum`` dicts into batched numpy arrays.\\"\\"\\"\\n    rows = list(rows)\\n    if not rows:\\n        raise ValueError(\\"no features to stack\\")\\n    keys = [\\n        \\"peak_mz\\",\\n        \\"peak_intensity\\",\\n        \\"peak_nl\\",\\n        \\"peak_mask\\",\\n        \\"binned\\",\\n        \\"precursor_feat\\",\\n    ]\\n    out: dict[str, np.ndarray] = {k: np.stack([r[k] for r in rows], axis=0) for k in keys}\\n    out[\\"adduct_id\\"] = np.asarray([int(r[\\"adduct_id\\"]) for r in rows], dtype=np.int64)\\n    out[\\"neutral_mass\\"] = np.asarray([float(r[\\"neutral_mass\\"]) for r in rows], dtype=np.float32)\\n    out[\\"precursor_mz\\"] = np.asarray([float(r[\\"precursor_mz\\"]) for r in rows], dtype=np.float32)\\n    return out\\n", "src/data.py": "\\"\\"\\"Parquet loading, structure tables, and TPU-static PyTorch datasets.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\nfrom typing import Any, Iterator, Sequence\\n\\nimport numpy as np\\nimport pandas as pd\\nimport pyarrow.parquet as pq\\nimport torch\\nfrom torch.utils.data import Dataset\\n\\nfrom src.chem import formula_to_mass, morgan_fingerprint\\nfrom src.config import ADDUCT_VOCAB, NUM_ADDUCTS, Config\\nfrom src.preprocessing import featurize_spectrum, stack_features\\n\\n\\nTRAIN_COLUMNS = [\\n    \\"ingest_lib\\",\\n    \\"normalized_smiles\\",\\n    \\"inchikey\\",\\n    \\"inchikey14\\",\\n    \\"molecular_formula\\",\\n    \\"ionization_mode\\",\\n    \\"instrument_type\\",\\n    \\"adduct\\",\\n    \\"adduct_orig\\",\\n    \\"precursor_mz\\",\\n    \\"precursor_error_ppm\\",\\n    \\"ms2_mzs\\",\\n    \\"ms2_normalized_intensities\\",\\n    \\"num_peaks\\",\\n    \\"base_peak_intensity\\",\\n    \\"collision_energy_ev\\",\\n    \\"collision_energy_orig\\",\\n    \\"collision_energy_orig_units\\",\\n]\\n\\n\\ndef resolve_split_path(path: Path | str, split: str) -> Path:\\n    p = Path(path)\\n    if p.exists():\\n        return p\\n    raise FileNotFoundError(f\\"Missing {split} parquet: {p}\\")\\n\\n\\ndef read_parquet(path: Path | str, columns: Sequence[str] | None = None) -> pd.DataFrame:\\n    return pq.read_table(str(path), columns=list(columns) if columns else None).to_pandas()\\n\\n\\ndef iter_train_row_groups(\\n    path: Path | str,\\n    columns: Sequence[str] | None = None,\\n) -> Iterator[pd.DataFrame]:\\n    pf = pq.ParquetFile(str(path))\\n    cols = list(columns) if columns else None\\n    for i in range(pf.num_row_groups):\\n        yield pf.read_row_group(i, columns=cols).to_pandas()\\n\\n\\ndef load_train_slice(cfg: Config, n_rows: int, columns: Sequence[str] | None = None) -> pd.DataFrame:\\n    \\"\\"\\"Read the first ``n_rows`` training spectra without loading the full 2.5M table.\\"\\"\\"\\n    pf = pq.ParquetFile(str(cfg.train_path))\\n    cols = list(columns) if columns else None\\n    chunks: list[pd.DataFrame] = []\\n    remaining = int(n_rows)\\n    for i in range(pf.num_row_groups):\\n        if remaining <= 0:\\n            break\\n        table = pf.read_row_group(i, columns=cols)\\n        take = min(remaining, table.num_rows)\\n        chunks.append(table.slice(0, take).to_pandas())\\n        remaining -= take\\n    if not chunks:\\n        return pd.DataFrame()\\n    return pd.concat(chunks, ignore_index=True)\\n\\n\\ndef load_test(cfg: Config) -> pd.DataFrame:\\n    return read_parquet(cfg.test_path)\\n\\n\\n@dataclass\\nclass StructureRecord:\\n    smiles: str\\n    inchikey14: str\\n    exact_mass: float\\n    formula: str\\n\\n\\ndef build_structure_table(train_df: pd.DataFrame) -> pd.DataFrame:\\n    \\"\\"\\"Unique 2D skeletons from a train frame, keyed by InChIKey14.\\"\\"\\"\\n    cols = [c for c in (\\"normalized_smiles\\", \\"inchikey14\\", \\"molecular_formula\\") if c in train_df.columns]\\n    sub = train_df[cols].dropna(subset=[\\"normalized_smiles\\", \\"inchikey14\\"])\\n    # Prefer the first SMILES per skeleton (already RDKit-normalized in the dump).\\n    sub = sub.drop_duplicates(subset=[\\"inchikey14\\"], keep=\\"first\\").reset_index(drop=True)\\n    masses = np.array(\\n        [formula_to_mass(f) if isinstance(f, str) else 0.0 for f in sub[\\"molecular_formula\\"].tolist()],\\n        dtype=np.float64,\\n    )\\n    # If formula mass failed, leave 0; retrieval will skip zeros.\\n    sub = sub.copy()\\n    sub[\\"exact_mass\\"] = masses\\n    return sub\\n\\n\\ndef fingerprints_for_smiles(smiles: Sequence[str], cfg: Config) -> np.ndarray:\\n    fps = np.zeros((len(smiles), cfg.fp_bits), dtype=np.uint8)\\n    for i, smi in enumerate(smiles):\\n        fps[i] = morgan_fingerprint(smi, n_bits=cfg.fp_bits, radius=cfg.morgan_radius)\\n    return fps\\n\\n\\nclass SpectrumFingerprintDataset(Dataset):\\n    \\"\\"\\"Fixed-shape tensors only: peak stacks, binned spectrum, precursor feats, fingerprints.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        peak_mz: np.ndarray,\\n        peak_intensity: np.ndarray,\\n        peak_nl: np.ndarray,\\n        peak_mask: np.ndarray,\\n        binned: np.ndarray,\\n        precursor_feat: np.ndarray,\\n        adduct_id: np.ndarray,\\n        fingerprints: np.ndarray,\\n    ) -> None:\\n        self.peak_mz = np.ascontiguousarray(peak_mz, dtype=np.float32)\\n        self.peak_intensity = np.ascontiguousarray(peak_intensity, dtype=np.float32)\\n        self.peak_nl = np.ascontiguousarray(peak_nl, dtype=np.float32)\\n        self.peak_mask = np.ascontiguousarray(peak_mask, dtype=np.float32)\\n        self.binned = np.ascontiguousarray(binned, dtype=np.float32)\\n        self.precursor_feat = np.ascontiguousarray(precursor_feat, dtype=np.float32)\\n        self.adduct_id = np.ascontiguousarray(adduct_id, dtype=np.int64)\\n        fp = np.ascontiguousarray(fingerprints)\\n        if fp.dtype != np.float32:\\n            fp = fp.astype(np.float32)\\n        self.fingerprints = fp\\n        n = self.peak_mz.shape[0]\\n        assert self.peak_intensity.shape[0] == n\\n        assert self.peak_nl.shape[0] == n\\n        assert self.peak_mask.shape[0] == n\\n        assert self.binned.shape[0] == n\\n        assert self.precursor_feat.shape[0] == n\\n        assert self.adduct_id.shape[0] == n\\n        assert self.fingerprints.shape[0] == n\\n\\n    def __len__(self) -> int:\\n        return int(self.peak_mz.shape[0])\\n\\n    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:\\n        return {\\n            \\"peak_mz\\": torch.from_numpy(self.peak_mz[idx]),\\n            \\"peak_intensity\\": torch.from_numpy(self.peak_intensity[idx]),\\n            \\"peak_nl\\": torch.from_numpy(self.peak_nl[idx]),\\n            \\"peak_mask\\": torch.from_numpy(self.peak_mask[idx]),\\n            \\"binned\\": torch.from_numpy(self.binned[idx]),\\n            \\"precursor_feat\\": torch.from_numpy(self.precursor_feat[idx]),\\n            \\"adduct_id\\": torch.tensor(self.adduct_id[idx], dtype=torch.long),\\n            \\"fingerprint\\": torch.from_numpy(self.fingerprints[idx]),\\n        }\\n\\n\\ndef collate_fixed(batch: list[dict[str, torch.Tensor]]) -> dict[str, torch.Tensor]:\\n    \\"\\"\\"Default collate is already static-shape-safe; kept explicit for XLA loaders.\\"\\"\\"\\n    keys = batch[0].keys()\\n    out: dict[str, torch.Tensor] = {}\\n    for k in keys:\\n        out[k] = torch.stack([b[k] for b in batch], dim=0)\\n    return out\\n\\n\\ndef featurize_dataframe(\\n    df: pd.DataFrame,\\n    cfg: Config,\\n    *,\\n    compute_fingerprints: bool = True,\\n    smiles_col: str = \\"normalized_smiles\\",\\n) -> tuple[dict[str, np.ndarray], np.ndarray | None]:\\n    \\"\\"\\"Vector of static examples from a spectra dataframe.\\"\\"\\"\\n    feats: list[dict[str, Any]] = []\\n    mzs = df[\\"ms2_mzs\\"].tolist()\\n    ints = df[\\"ms2_normalized_intensities\\"].tolist()\\n    precs = df[\\"precursor_mz\\"].to_numpy()\\n    adducts = df[\\"adduct\\"].tolist() if \\"adduct\\" in df.columns else [\\"\\"] * len(df)\\n    ces = df[\\"collision_energy_ev\\"].tolist() if \\"collision_energy_ev\\" in df.columns else [None] * len(df)\\n    ions = df[\\"ionization_mode\\"].tolist() if \\"ionization_mode\\" in df.columns else [None] * len(df)\\n    for i in range(len(df)):\\n        feats.append(\\n            featurize_spectrum(\\n                mzs[i],\\n                ints[i],\\n                float(precs[i] or 0.0),\\n                adducts[i],\\n                cfg=cfg,\\n                collision_energy=ces[i],\\n                ionization_mode=ions[i],\\n            )\\n        )\\n    stacked = stack_features(feats)\\n    fps: np.ndarray | None = None\\n    if compute_fingerprints and smiles_col in df.columns:\\n        smiles = df[smiles_col].astype(str).tolist()\\n        cache: dict[str, np.ndarray] = {}\\n        fps = np.zeros((len(smiles), cfg.fp_bits), dtype=np.uint8)\\n        if \\"inchikey14\\" in df.columns:\\n            keys = df[\\"inchikey14\\"].astype(str).tolist()\\n        else:\\n            keys = smiles\\n        for i, (key, smi) in enumerate(zip(keys, smiles)):\\n            hit = cache.get(key)\\n            if hit is None:\\n                hit = morgan_fingerprint(smi, n_bits=cfg.fp_bits, radius=cfg.morgan_radius)\\n                cache[key] = hit\\n            fps[i] = hit\\n    return stacked, fps\\n\\n\\ndef dataset_from_dataframe(df: pd.DataFrame, cfg: Config) -> SpectrumFingerprintDataset:\\n    stacked, fps = featurize_dataframe(df, cfg, compute_fingerprints=True)\\n    if fps is None:\\n        fps = np.zeros((len(df), cfg.fp_bits), dtype=np.float32)\\n    return SpectrumFingerprintDataset(\\n        peak_mz=stacked[\\"peak_mz\\"],\\n        peak_intensity=stacked[\\"peak_intensity\\"],\\n        peak_nl=stacked[\\"peak_nl\\"],\\n        peak_mask=stacked[\\"peak_mask\\"],\\n        binned=stacked[\\"binned\\"],\\n        precursor_feat=stacked[\\"precursor_feat\\"],\\n        adduct_id=stacked[\\"adduct_id\\"],\\n        fingerprints=fps,\\n    )\\n\\n\\ndef dataset_from_structure_index(\\n    index,\\n    cfg: Config,\\n    *,\\n    max_n: int | None = None,\\n    seed: int = 42,\\n) -> SpectrumFingerprintDataset:\\n    \\"\\"\\"Train from the library, retaining the representative\'s observed features.\\n\\n    The 0.139 baseline reconstructed every row as protonated/positive with no\\n    collision energy and used neutral mass for peak losses. Inference instead\\n    calls featurize_spectrum with the observed precursor and metadata. Copy its\\n    saved tensors here so both paths use the same feature definition. Setting\\n    cfg.use_representative_metadata=False reproduces the original dataset path.\\n    \\"\\"\\"\\n    from src.preprocessing import bin_spectrum, precursor_feature_vector\\n\\n    if getattr(index, \\"peak_mz\\", None) is None:\\n        raise ValueError(\\"structure index has no representative peaks\\")\\n    n = int(index.smiles.shape[0])\\n    rng = np.random.default_rng(seed)\\n    take = np.arange(n)\\n    if max_n is not None and n > int(max_n):\\n        take = np.sort(rng.choice(n, size=int(max_n), replace=False))\\n    peak_mz = np.ascontiguousarray(index.peak_mz[take], dtype=np.float32)\\n    peak_int = np.ascontiguousarray(index.peak_intensity[take], dtype=np.float32)\\n    peak_mask = np.ascontiguousarray(index.peak_mask[take], dtype=np.float32)\\n    mass = np.ascontiguousarray(index.exact_mass[take], dtype=np.float32)\\n    fps = np.ascontiguousarray(index.fingerprints[take], dtype=np.float32)\\n    n_take = int(peak_mz.shape[0])\\n    binned = np.zeros((n_take, cfg.n_mz_bins), dtype=np.float32)\\n    observed = bool(cfg.use_representative_metadata)\\n    if observed:\\n        expected = {\\n            \\"peak_nl\\": (n, cfg.top_n_peaks),\\n            \\"precursor_feat\\": (n, cfg.precursor_feat_dim),\\n            \\"adduct_id\\": (n,),\\n        }\\n        for name, shape in expected.items():\\n            arr = getattr(index, name, None)\\n            if arr is None or arr.shape != shape:\\n                raise ValueError(\\n                    f\\"Representative metadata {name} must have shape {shape}; \\"\\n                    \\"rebuild the train index with build_library_from_train/frame.\\"\\n                )\\n        peak_nl = np.ascontiguousarray(index.peak_nl[take], dtype=np.float32)\\n        feats = np.ascontiguousarray(index.precursor_feat[take], dtype=np.float32)\\n        adduct_id = np.ascontiguousarray(index.adduct_id[take], dtype=np.int64)\\n        if not np.isfinite(peak_nl).all() or not np.isfinite(feats).all():\\n            raise ValueError(\\"Missing or non-finite representative spectrum features\\")\\n        if np.any((adduct_id < 0) | (adduct_id >= NUM_ADDUCTS)):\\n            raise ValueError(\\"Missing or invalid representative adduct IDs\\")\\n    else:\\n        # Keep this branch identical to the measured 0.139 feature construction.\\n        peak_nl = np.clip(mass[:, None] - peak_mz, 0.0, None) * peak_mask\\n        feats = np.zeros((n_take, cfg.precursor_feat_dim), dtype=np.float32)\\n        adduct_id = np.zeros((n_take,), dtype=np.int64)\\n    for i in range(n_take):\\n        binned[i] = bin_spectrum(peak_mz[i], peak_int[i], peak_mask[i], cfg=cfg)\\n        if not observed:\\n            n_peaks = int(peak_mask[i].sum())\\n            prec = float(mass[i]) + 1.007276\\n            feats[i] = precursor_feature_vector(\\n                precursor_mz=prec,\\n                neutral_mass=float(mass[i]),\\n                adduct=\\"[M+H]+\\",\\n                collision_energy=None,\\n                ionization_mode=\\"positive\\",\\n                n_peaks=n_peaks,\\n                cfg=cfg,\\n            )\\n        if (i + 1) % 20_000 == 0:\\n            print(f\\"[dataset] binned {i+1}/{n_take}\\")\\n    ids, counts = np.unique(adduct_id, return_counts=True)\\n    print(\\n        f\\"[dataset] representative_metadata={observed} \\"\\n        f\\"adduct_counts={dict(zip((ADDUCT_VOCAB[int(i)] for i in ids), counts.tolist()))}\\"\\n    )\\n    return SpectrumFingerprintDataset(\\n        peak_mz=peak_mz,\\n        peak_intensity=peak_int,\\n        peak_nl=peak_nl.astype(np.float32),\\n        peak_mask=peak_mask,\\n        binned=binned,\\n        precursor_feat=feats,\\n        adduct_id=adduct_id,\\n        fingerprints=fps,\\n    )\\n\\n\\ndef assert_static_shapes(batch: dict[str, torch.Tensor | np.ndarray], cfg: Config) -> None:\\n    \\"\\"\\"Raise if any spectral tensor violates the XLA static-shape contract.\\"\\"\\"\\n    n = cfg.top_n_peaks\\n    b = cfg.n_mz_bins\\n    f = cfg.fp_bits\\n    p = cfg.precursor_feat_dim\\n\\n    def _shape(x) -> tuple[int, ...]:\\n        if hasattr(x, \\"shape\\"):\\n            return tuple(int(s) for s in x.shape)\\n        raise TypeError(type(x))\\n\\n    checks = {\\n        \\"peak_mz\\": (n,),\\n        \\"peak_intensity\\": (n,),\\n        \\"peak_nl\\": (n,),\\n        \\"peak_mask\\": (n,),\\n        \\"binned\\": (b,),\\n        \\"precursor_feat\\": (p,),\\n    }\\n    for key, tail in checks.items():\\n        if key not in batch:\\n            continue\\n        shape = _shape(batch[key])\\n        if shape[-len(tail) :] != tail:\\n            raise AssertionError(f\\"{key} has shape {shape}, expected trailing {tail}\\")\\n    if \\"fingerprint\\" in batch:\\n        shape = _shape(batch[\\"fingerprint\\"])\\n        if shape[-1] != f:\\n            raise AssertionError(f\\"fingerprint last dim {shape[-1]} != {f}\\")\\n    if \\"adduct_id\\" in batch:\\n        shape = _shape(batch[\\"adduct_id\\"])\\n        if len(shape) not in (0, 1):\\n            raise AssertionError(f\\"adduct_id rank {shape}\\")\\n", "src/model.py": "\\"\\"\\"Spectrum \\u2192 Morgan/ECFP4 fingerprint model (1D CNN + self-attention).\\n\\nAll sequence axes are the configured static lengths (top-N peaks, mz bins) so\\nPyTorch/XLA compiles a single graph.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\n\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\n\\nfrom src.config import NUM_ADDUCTS, Config, get_config\\n\\n\\ndef _sinusoidal_mz_encoding(mz: torch.Tensor, dim: int, max_mz: float = 1000.0) -> torch.Tensor:\\n    \\"\\"\\"Fourier features of m/z. ``mz`` is ``(B, N)`` in Daltons.\\"\\"\\"\\n    if dim % 2 != 0:\\n        raise ValueError(\\"sinusoidal dim must be even\\")\\n    half = dim // 2\\n    device = mz.device\\n    freq = torch.exp(torch.linspace(0.0, math.log(max_mz), half, device=device))\\n    # (B, N, half)\\n    angles = mz.unsqueeze(-1) / freq.view(1, 1, half).clamp_min(1e-3)\\n    return torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)\\n\\n\\nclass ConvBlock(nn.Module):\\n    def __init__(self, channels: int, kernel_size: int = 5, dropout: float = 0.1) -> None:\\n        super().__init__()\\n        pad = kernel_size // 2\\n        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=pad)\\n        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=pad)\\n        self.norm1 = nn.BatchNorm1d(channels)\\n        self.norm2 = nn.BatchNorm1d(channels)\\n        self.drop = nn.Dropout(dropout)\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        h = self.drop(F.gelu(self.norm1(self.conv1(x))))\\n        h = self.drop(F.gelu(self.norm2(self.conv2(h))))\\n        return x + h\\n\\n\\nclass Spec2FP(nn.Module):\\n    \\"\\"\\"Encode a padded peak list + binned spectrum into 2048 fingerprint logits.\\"\\"\\"\\n\\n    def __init__(self, cfg: Config | None = None) -> None:\\n        super().__init__()\\n        cfg = cfg or get_config()\\n        self.cfg = cfg\\n        d = cfg.d_model\\n        self.peak_in = nn.Linear(6 + 32, d)  # raw feats + fourier(mz)\\n        encoder_layer = nn.TransformerEncoderLayer(\\n            d_model=d,\\n            nhead=cfg.n_heads,\\n            dim_feedforward=d * 4,\\n            dropout=cfg.dropout,\\n            activation=\\"gelu\\",\\n            batch_first=True,\\n            norm_first=True,\\n        )\\n        self.peak_encoder = nn.TransformerEncoder(\\n            encoder_layer, num_layers=cfg.n_transformer_layers, enable_nested_tensor=False\\n        )\\n        self.cls_token = nn.Parameter(torch.randn(1, 1, d) * 0.02)\\n\\n        self.bin_proj = nn.Conv1d(1, cfg.n_conv_channels, kernel_size=7, padding=3)\\n        self.bin_blocks = nn.Sequential(\\n            ConvBlock(cfg.n_conv_channels, 7, cfg.dropout),\\n            ConvBlock(cfg.n_conv_channels, 5, cfg.dropout),\\n            ConvBlock(cfg.n_conv_channels, 3, cfg.dropout),\\n        )\\n        self.bin_pool = nn.AdaptiveAvgPool1d(1)\\n        self.bin_out = nn.Linear(cfg.n_conv_channels, d)\\n\\n        self.adduct_emb = nn.Embedding(NUM_ADDUCTS, cfg.adduct_embed_dim)\\n        self.meta = nn.Sequential(\\n            nn.Linear(cfg.precursor_feat_dim + cfg.adduct_embed_dim, d),\\n            nn.GELU(),\\n            nn.Dropout(cfg.dropout),\\n            nn.Linear(d, d),\\n        )\\n        fused = d * 3\\n        self.head = nn.Sequential(\\n            nn.Linear(fused, d * 2),\\n            nn.GELU(),\\n            nn.Dropout(cfg.dropout),\\n            nn.Linear(d * 2, d * 2),\\n            nn.GELU(),\\n            nn.Dropout(cfg.dropout),\\n            nn.Linear(d * 2, cfg.fp_bits),\\n        )\\n        self._init()\\n\\n    def _init(self) -> None:\\n        for m in self.modules():\\n            if isinstance(m, nn.Linear):\\n                nn.init.xavier_uniform_(m.weight)\\n                if m.bias is not None:\\n                    nn.init.zeros_(m.bias)\\n\\n    def forward(\\n        self,\\n        peak_mz: torch.Tensor,\\n        peak_intensity: torch.Tensor,\\n        peak_nl: torch.Tensor,\\n        peak_mask: torch.Tensor,\\n        binned: torch.Tensor,\\n        precursor_feat: torch.Tensor,\\n        adduct_id: torch.Tensor,\\n    ) -> torch.Tensor:\\n        # peak tokens: (B, N, d)\\n        mz_n = peak_mz / 1000.0\\n        nl_n = peak_nl / 1000.0\\n        fourier = _sinusoidal_mz_encoding(peak_mz, 32)\\n        raw = torch.stack(\\n            [\\n                mz_n,\\n                peak_intensity,\\n                nl_n,\\n                torch.log1p(peak_mz.clamp_min(0.0)),\\n                torch.log1p(peak_nl.clamp_min(0.0)),\\n                peak_intensity.sqrt(),\\n            ],\\n            dim=-1,\\n        )\\n        tokens = self.peak_in(torch.cat([raw, fourier], dim=-1))\\n        bsz = tokens.size(0)\\n        cls = self.cls_token.expand(bsz, -1, -1)\\n        tokens = torch.cat([cls, tokens], dim=1)\\n        # Transformer key padding: True = ignore. CLS is never padding.\\n        pad = peak_mask <= 0\\n        cls_pad = torch.zeros(bsz, 1, dtype=torch.bool, device=pad.device)\\n        key_padding = torch.cat([cls_pad, pad], dim=1)\\n        encoded = self.peak_encoder(tokens, src_key_padding_mask=key_padding)\\n        peak_vec = encoded[:, 0, :]\\n\\n        bin_h = self.bin_proj(binned.unsqueeze(1))\\n        bin_h = self.bin_blocks(bin_h)\\n        bin_vec = self.bin_out(self.bin_pool(bin_h).squeeze(-1))\\n\\n        meta_in = torch.cat([precursor_feat, self.adduct_emb(adduct_id.long())], dim=-1)\\n        meta_vec = self.meta(meta_in)\\n        fused = torch.cat([peak_vec, bin_vec, meta_vec], dim=-1)\\n        return self.head(fused)\\n\\n\\nclass FocalBCELoss(nn.Module):\\n    \\"\\"\\"Multi-label focal BCE on logits (fingerprint bits).\\"\\"\\"\\n\\n    def __init__(self, gamma: float = 1.5, pos_weight: float = 6.0) -> None:\\n        super().__init__()\\n        self.gamma = gamma\\n        self.register_buffer(\\"pos_weight\\", torch.tensor(float(pos_weight)))\\n\\n    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:\\n        targets = targets.float()\\n        bce = F.binary_cross_entropy_with_logits(\\n            logits, targets, reduction=\\"none\\", pos_weight=self.pos_weight\\n        )\\n        p = torch.sigmoid(logits)\\n        pt = torch.where(targets > 0.5, p, 1.0 - p)\\n        loss = bce * (1.0 - pt).clamp_min(0.0).pow(self.gamma)\\n        return loss.mean()\\n\\n\\ndef model_from_config(cfg: Config | None = None) -> Spec2FP:\\n    return Spec2FP(cfg or get_config())\\n", "src/models/__init__.py": "\\"\\"\\"Re-export Spec2FP and the prefix-conditioned SMILES decoder.\\"\\"\\"\\n\\nfrom src.model import FocalBCELoss, Spec2FP, model_from_config\\nfrom src.models.smiles_decoder import (\\n    LightweightSmilesDecoder,\\n    decoder_from_config,\\n    generate_candidates_for_molecule,\\n    sample_autoregressive,\\n)\\n\\n__all__ = [\\n    \\"Spec2FP\\",\\n    \\"FocalBCELoss\\",\\n    \\"model_from_config\\",\\n    \\"LightweightSmilesDecoder\\",\\n    \\"decoder_from_config\\",\\n    \\"generate_candidates_for_molecule\\",\\n    \\"sample_autoregressive\\",\\n]\\n", "src/models/smiles_decoder.py": "\\"\\"\\"Prefix-conditioned causal Transformer for de novo SMILES generation.\\n\\nConditioning (predicted Morgan fingerprint + precursor mass + adduct) is\\nprojected into a handful of virtual prefix tokens and prepended to the SMILES\\nembeddings, so a standard causal self-attention stack is sufficient \\u2014 no\\ncross-attention stream, and all sequence lengths stay static for XLA.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nfrom typing import Optional\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\n\\nfrom src.chem import canonical_smiles, exact_mass_from_smiles, inchikey14_from_smiles, morgan_fingerprint, smiles_to_mol\\nfrom src.config import NUM_ADDUCTS, Config\\nfrom src.retrieval import tanimoto\\nfrom src.smiles_tokenizer import SmilesTokenizer\\n\\n\\nclass SinusoidalMassEmbedding(nn.Module):\\n    \\"\\"\\"Encodes continuous precursor neutral mass into a dense vector.\\"\\"\\"\\n\\n    def __init__(self, dim: int = 64) -> None:\\n        super().__init__()\\n        self.dim = dim\\n\\n    def forward(self, mass: torch.Tensor) -> torch.Tensor:\\n        # mass shape: (B, 1) or (B,)\\n        if mass.dim() == 1:\\n            mass = mass.unsqueeze(-1)\\n        device = mass.device\\n        half_dim = self.dim // 2\\n        emb = math.log(10000) / max(half_dim - 1, 1)\\n        emb = torch.exp(torch.arange(half_dim, device=device, dtype=mass.dtype) * -emb)\\n        ang = mass * emb.unsqueeze(0)\\n        return torch.cat([torch.sin(ang), torch.cos(ang)], dim=-1)\\n\\n\\nclass LightweightSmilesDecoder(nn.Module):\\n    def __init__(\\n        self,\\n        vocab_size: int,\\n        fp_dim: int = 2048,\\n        num_adducts: int = 16,\\n        d_model: int = 256,\\n        nhead: int = 8,\\n        num_layers: int = 4,\\n        dim_feedforward: int = 1024,\\n        prefix_len: int = 8,\\n        max_seq_len: int = 128,\\n        dropout: float = 0.1,\\n    ) -> None:\\n        super().__init__()\\n        self.d_model = d_model\\n        self.prefix_len = prefix_len\\n        self.max_seq_len = max_seq_len\\n        self.vocab_size = vocab_size\\n\\n        self.mass_encoder = SinusoidalMassEmbedding(dim=64)\\n        self.adduct_embed = nn.Embedding(num_adducts, 32)\\n\\n        cond_input_dim = fp_dim + 64 + 32\\n        self.cond_mlp = nn.Sequential(\\n            nn.Linear(cond_input_dim, d_model * 2),\\n            nn.GELU(),\\n            nn.LayerNorm(d_model * 2),\\n            nn.Linear(d_model * 2, prefix_len * d_model),\\n        )\\n\\n        self.token_embed = nn.Embedding(vocab_size, d_model)\\n        self.pos_embed = nn.Embedding(max_seq_len + prefix_len, d_model)\\n\\n        decoder_layer = nn.TransformerEncoderLayer(\\n            d_model=d_model,\\n            nhead=nhead,\\n            dim_feedforward=dim_feedforward,\\n            dropout=dropout,\\n            activation=\\"gelu\\",\\n            batch_first=True,\\n            norm_first=True,\\n        )\\n        self.transformer = nn.TransformerEncoder(\\n            decoder_layer, num_layers=num_layers, enable_nested_tensor=False\\n        )\\n        self.norm = nn.LayerNorm(d_model)\\n        self.head = nn.Linear(d_model, vocab_size, bias=False)\\n        self.head.weight = self.token_embed.weight\\n\\n        total = prefix_len + max_seq_len\\n        causal = torch.triu(torch.ones(total, total, dtype=torch.bool), diagonal=1)\\n        causal[:prefix_len, :prefix_len] = False\\n        self.register_buffer(\\"causal_mask\\", causal, persistent=False)\\n\\n    def _causal_mask(self, total_len: int, device: torch.device) -> torch.Tensor:\\n        if total_len == self.causal_mask.size(0):\\n            return self.causal_mask.to(device)\\n        mask = torch.triu(torch.ones(total_len, total_len, dtype=torch.bool, device=device), diagonal=1)\\n        mask[: self.prefix_len, : self.prefix_len] = False\\n        return mask\\n\\n    def forward(\\n        self,\\n        fp: torch.Tensor,\\n        mass: torch.Tensor,\\n        adduct_idx: torch.Tensor,\\n        tgt_tokens: torch.Tensor,\\n        tgt_mask: Optional[torch.Tensor] = None,\\n    ) -> torch.Tensor:\\n        \\"\\"\\"\\n        fp: (B, 2048) predicted probabilities or ground-truth bits\\n        mass: (B, 1) neutral precursor mass\\n        adduct_idx: (B,) integer adduct class\\n        tgt_tokens: (B, L) SMILES token ids (BOS-prefixed, PAD-padded)\\n        tgt_mask: (B, L) True = padding (ignored by attention)\\n        \\"\\"\\"\\n        bsz, length = tgt_tokens.shape\\n        device = fp.device\\n\\n        mass_enc = self.mass_encoder(mass)\\n        adduct_enc = self.adduct_embed(adduct_idx.long())\\n        cond_vec = torch.cat([fp.float(), mass_enc, adduct_enc], dim=-1)\\n        prefix = self.cond_mlp(cond_vec).view(bsz, self.prefix_len, self.d_model)\\n\\n        token_emb = self.token_embed(tgt_tokens)\\n        seq = torch.cat([prefix, token_emb], dim=1)\\n        pos = torch.arange(0, self.prefix_len + length, device=device).unsqueeze(0)\\n        seq = seq + self.pos_embed(pos)\\n\\n        total_len = self.prefix_len + length\\n        causal_mask = self._causal_mask(total_len, device)\\n        key_padding = None\\n        if tgt_mask is not None:\\n            prefix_pad = torch.zeros(bsz, self.prefix_len, dtype=torch.bool, device=device)\\n            key_padding = torch.cat([prefix_pad, tgt_mask.bool()], dim=1)\\n\\n        out = self.transformer(seq, mask=causal_mask, src_key_padding_mask=key_padding)\\n        out = self.norm(out)\\n        return self.head(out[:, self.prefix_len :, :])\\n\\n\\ndef decoder_from_config(tokenizer: SmilesTokenizer, cfg: Config) -> LightweightSmilesDecoder:\\n    return LightweightSmilesDecoder(\\n        vocab_size=tokenizer.vocab_size,\\n        fp_dim=cfg.fp_bits,\\n        num_adducts=NUM_ADDUCTS,\\n        d_model=cfg.smiles_d_model,\\n        nhead=cfg.smiles_nhead,\\n        num_layers=cfg.smiles_num_layers,\\n        dim_feedforward=cfg.smiles_dim_feedforward,\\n        prefix_len=cfg.smiles_prefix_len,\\n        max_seq_len=cfg.smiles_max_len,\\n        dropout=cfg.smiles_dropout,\\n    )\\n\\n\\ndef count_parameters(model: nn.Module) -> int:\\n    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))\\n\\n\\ndef _top_p_logits(logits: torch.Tensor, top_p: float) -> torch.Tensor:\\n    if top_p >= 1.0:\\n        return logits\\n    sorted_logits, sorted_idx = torch.sort(logits, dim=-1, descending=True)\\n    probs = F.softmax(sorted_logits, dim=-1)\\n    cum = torch.cumsum(probs, dim=-1)\\n    mask = cum > top_p\\n    mask[..., 1:] = mask[..., :-1].clone()\\n    mask[..., 0] = False\\n    sorted_logits = sorted_logits.masked_fill(mask, float(\\"-inf\\"))\\n    return torch.zeros_like(logits).scatter_(-1, sorted_idx, sorted_logits)\\n\\n\\n@torch.no_grad()\\ndef sample_autoregressive(\\n    model: LightweightSmilesDecoder,\\n    fp: torch.Tensor,\\n    mass: torch.Tensor,\\n    adduct_idx: torch.Tensor,\\n    tokenizer: SmilesTokenizer,\\n    *,\\n    max_len: int = 100,\\n    temp: float = 0.75,\\n    top_p: float = 0.90,\\n) -> list[list[int]]:\\n    \\"\\"\\"CPU-friendly ancestral sampling. Sequence length grows; do not run on XLA.\\"\\"\\"\\n    model.eval()\\n    device = fp.device\\n    bsz = int(fp.shape[0])\\n    max_len = min(int(max_len), model.max_seq_len)\\n    tokens = torch.full((bsz, 1), tokenizer.bos_id, dtype=torch.long, device=device)\\n    finished = torch.zeros(bsz, dtype=torch.bool, device=device)\\n    temp = max(float(temp), 1e-5)\\n\\n    for _ in range(max_len - 1):\\n        pad_mask = tokens == tokenizer.pad_id\\n        logits = model(fp, mass, adduct_idx, tokens, tgt_mask=pad_mask)\\n        step = logits[:, -1, :] / temp\\n        step[:, tokenizer.pad_id] = float(\\"-inf\\")\\n        step[:, tokenizer.bos_id] = float(\\"-inf\\")\\n        step = _top_p_logits(step, top_p)\\n        probs = F.softmax(step, dim=-1)\\n        nxt = torch.multinomial(probs, num_samples=1).squeeze(-1)\\n        nxt = torch.where(finished, torch.full_like(nxt, tokenizer.pad_id), nxt)\\n        tokens = torch.cat([tokens, nxt.unsqueeze(-1)], dim=1)\\n        finished = finished | (nxt == tokenizer.eos_id)\\n        if bool(finished.all()):\\n            break\\n    return [row.tolist() for row in tokens.cpu()]\\n\\n\\ndef compute_tanimoto_against_pred(gen_fp: np.ndarray, pred_fp_bits: np.ndarray) -> float:\\n    return float(tanimoto(pred_fp_bits, gen_fp.reshape(1, -1))[0])\\n\\n\\ndef generate_candidates_for_molecule(\\n    model: LightweightSmilesDecoder,\\n    pred_fp: torch.Tensor,\\n    neutral_mass: float,\\n    adduct_idx: int,\\n    tokenizer: SmilesTokenizer,\\n    num_samples: int = 100,\\n    mass_tolerance_ppm: float = 20.0,\\n    target_k: int = 25,\\n    temp: float = 0.75,\\n    top_p: float = 0.90,\\n    max_len: int = 100,\\n    keep_mass_misses: bool = False,\\n) -> list[str]:\\n    \\"\\"\\"Generate, mass-filter, InChIKey14-dedup, and rank de novo SMILES (Class 3).\\"\\"\\"\\n    model.eval()\\n    device = next(model.parameters()).device\\n    if pred_fp.dim() == 1:\\n        pred_fp = pred_fp.unsqueeze(0)\\n    pred_fp = pred_fp.float().to(device)\\n    n = int(num_samples)\\n    b_fp = pred_fp.repeat(n, 1)\\n    b_mass = torch.full((n, 1), float(neutral_mass), dtype=torch.float32, device=device)\\n    b_adduct = torch.full((n,), int(adduct_idx), dtype=torch.long, device=device)\\n\\n    with torch.no_grad():\\n        generated_tokens = sample_autoregressive(\\n            model,\\n            b_fp,\\n            b_mass,\\n            b_adduct,\\n            tokenizer,\\n            max_len=max_len,\\n            temp=temp,\\n            top_p=top_p,\\n        )\\n\\n    pred_np = pred_fp[0].detach().cpu().numpy()\\n    pred_fp_bits = (pred_np >= 0.5).astype(np.float32)\\n    if pred_fp_bits.sum() == 0:\\n        k = min(64, pred_np.size)\\n        top = np.argpartition(pred_np, -k)[-k:]\\n        pred_fp_bits = np.zeros_like(pred_np, dtype=np.float32)\\n        pred_fp_bits[top] = 1.0\\n\\n    valid: dict[str, dict[str, float | str]] = {}\\n    misses: dict[str, dict[str, float | str]] = {}\\n    query_mass = float(neutral_mass)\\n    for token_seq in generated_tokens:\\n        smiles_str = tokenizer.decode(token_seq)\\n        mol = smiles_to_mol(smiles_str)\\n        if mol is None:\\n            continue\\n        calc_mass = exact_mass_from_smiles(smiles_str)\\n        if calc_mass <= 0:\\n            continue\\n        ppm_error = (\\n            abs(calc_mass - query_mass) / query_mass * 1e6 if query_mass > 0 else 1e9\\n        )\\n        key = inchikey14_from_smiles(smiles_str)\\n        if not key:\\n            continue\\n        can = canonical_smiles(smiles_str)\\n        gen_fp = morgan_fingerprint(can, n_bits=pred_fp_bits.size)\\n        score = compute_tanimoto_against_pred(gen_fp, pred_fp_bits) - (ppm_error * 0.001)\\n        bucket = valid if ppm_error <= mass_tolerance_ppm else misses\\n        prev = bucket.get(key)\\n        if prev is None or score > float(prev[\\"score\\"]):\\n            bucket[key] = {\\"smiles\\": can, \\"score\\": float(score)}\\n\\n    ranked = sorted(valid.values(), key=lambda x: float(x[\\"score\\"]), reverse=True)\\n    out = [str(item[\\"smiles\\"]) for item in ranked[: int(target_k)]]\\n    if keep_mass_misses and len(out) < int(target_k):\\n        extra = sorted(misses.values(), key=lambda x: float(x[\\"score\\"]), reverse=True)\\n        have = set(out)\\n        for item in extra:\\n            smi = str(item[\\"smiles\\"])\\n            if smi in have:\\n                continue\\n            out.append(smi)\\n            have.add(smi)\\n            if len(out) >= int(target_k):\\n                break\\n    return out\\n", "src/chem_tokenizer.py": "\\"\\"\\"Regex SMILES tokenizer for natural-product chemical language models.\\n\\nBracketed atoms (``[C@@H]``, ``[nH]``, ``[O-]``), multi-letter halogens, and\\n``%10``-style macrocycle ring closures are kept as *single* tokens so the\\ndecoder does not waste length on shattered brackets.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom collections import Counter\\nfrom pathlib import Path\\nfrom typing import Dict, Iterable, List, Sequence\\n\\nimport re\\n\\nPACKAGED_VOCAB_PATH = Path(__file__).resolve().parent / \\"smiles_vocab.json\\"\\n\\n# Always keep these as single IDs even if train SMILES were de-stereo\'d.\\n# Natural-product generation still needs them as atomic units.\\nRESERVED_CHEM_TOKENS: tuple[str, ...] = (\\n    \\"/\\",\\n    \\"\\\\\\\\\\",\\n    \\"+\\",\\n    \\".\\",\\n    \\"@\\",\\n    \\"[C@@H]\\",\\n    \\"[C@H]\\",\\n    \\"[C@@]\\",\\n    \\"[C@]\\",\\n    \\"[NH3+]\\",\\n    \\"[NH2]\\",\\n    \\"[nH+]\\",\\n    \\"[O+]\\",\\n    \\"[NH+]\\",\\n    \\"%13\\",\\n    \\"%14\\",\\n    \\"%15\\",\\n)\\n\\n\\nclass SMILESTokenizer:\\n    \\"\\"\\"Regex tokenizer: longer / more specific matches precede single characters.\\"\\"\\"\\n\\n    def __init__(self) -> None:\\n        # ORDER MATTERS: longer/specific matches must precede single-character matches.\\n        pattern = (\\n            r\\"(\\"\\n            r\\"\\\\[[^\\\\]]+\\\\]|\\"  # 1. Bracketed atoms ([C@@H], [nH], [O-], [13C])\\n            r\\"Br|Cl|\\"  # 2. Multi-letter halogens (before B / C)\\n            r\\"c|n|o|s|p|b|\\"  # 3. Aromatic lower-case atoms\\n            r\\"C|N|O|S|P|F|I|B|\\"  # 4. Aliphatic upper-case atoms\\n            r\\"\\\\%[0-9]{2}|\\"  # 5. Double-digit ring closures (%10, %99)\\n            r\\"\\\\(|\\\\)|\\\\.|=|#|\\"  # 6. Branching, disconnection, bonds\\n            r\\"-|\\\\+|\\\\\\\\\\\\\\\\|\\\\/|\\"  # 7. Charges and cis/trans stereochem\\n            r\\":|~|@|\\"  # 8. Aromatic / arbitrary bonds, chiral @\\n            r\\"[0-9]\\"  # 9. Single-digit ring closures\\n            r\\")\\"\\n        )\\n        self.pattern = re.compile(pattern)\\n\\n    def tokenize(self, smiles: str) -> List[str]:\\n        \\"\\"\\"Split a SMILES string into chemically meaningful tokens.\\"\\"\\"\\n        if not smiles:\\n            return []\\n        tokens: List[str] = []\\n        for chunk in self.pattern.split(str(smiles)):\\n            if not chunk:\\n                continue\\n            if self.pattern.fullmatch(chunk):\\n                tokens.append(chunk)\\n            else:\\n                # Unknown leftover (e.g. \'Z\'): fall back to characters.\\n                tokens.extend(list(chunk))\\n        return tokens\\n\\n\\nclass SMILESVocabulary:\\n    \\"\\"\\"Token \\u2194 ID map with PAD/BOS/EOS/UNK. Vocab size is frozen for XLA.\\"\\"\\"\\n\\n    def __init__(self) -> None:\\n        self.pad_token = \\"<PAD>\\"\\n        self.bos_token = \\"<BOS>\\"\\n        self.eos_token = \\"<EOS>\\"\\n        self.unk_token = \\"<UNK>\\"\\n\\n        self.special_tokens = [self.pad_token, self.bos_token, self.eos_token, self.unk_token]\\n\\n        self.idx2token: Dict[int, str] = {i: tok for i, tok in enumerate(self.special_tokens)}\\n        self.token2idx: Dict[str, int] = {tok: i for i, tok in enumerate(self.special_tokens)}\\n\\n        self.pad_idx = self.token2idx[self.pad_token]\\n        self.bos_idx = self.token2idx[self.bos_token]\\n        self.eos_idx = self.token2idx[self.eos_token]\\n        self.unk_idx = self.token2idx[self.unk_token]\\n\\n        self.tokenizer = SMILESTokenizer()\\n\\n    def __len__(self) -> int:\\n        return len(self.idx2token)\\n\\n    def add_token(self, token: str) -> None:\\n        if token not in self.token2idx:\\n            idx = len(self.idx2token)\\n            self.token2idx[token] = idx\\n            self.idx2token[idx] = token\\n\\n    def build_vocab(self, smiles_iterable: Iterable[str], min_freq: int = 1) -> None:\\n        \\"\\"\\"Add tokens seen at least ``min_freq`` times (drops isotopic junk).\\"\\"\\"\\n        freqs: Counter[str] = Counter()\\n        for smiles in smiles_iterable:\\n            if not smiles or str(smiles).lower() in {\\"nan\\", \\"none\\"}:\\n                continue\\n            freqs.update(self.tokenizer.tokenize(str(smiles)))\\n        # Stable order: specials already present; then by descending freq, then token.\\n        ranked = sorted(freqs.items(), key=lambda kv: (-kv[1], kv[0]))\\n        for token, freq in ranked:\\n            if freq >= min_freq:\\n                self.add_token(token)\\n        for token in RESERVED_CHEM_TOKENS:\\n            self.add_token(token)\\n        print(f\\"Vocabulary built: {len(self)} total tokens.\\")\\n\\n    def encode(self, smiles: str, add_bos: bool = True, add_eos: bool = True) -> List[int]:\\n        tokens = self.tokenizer.tokenize(smiles)\\n        token_ids = [self.token2idx.get(tok, self.unk_idx) for tok in tokens]\\n        if add_bos:\\n            token_ids = [self.bos_idx] + token_ids\\n        if add_eos:\\n            token_ids = token_ids + [self.eos_idx]\\n        return token_ids\\n\\n    def encode_padded(self, smiles: str, max_len: int) -> List[int]:\\n        \\"\\"\\"BOS + tokens + EOS, truncated/padded to ``max_len`` (XLA-static).\\"\\"\\"\\n        max_len = int(max_len)\\n        ids = self.encode(smiles, add_bos=True, add_eos=True)\\n        if len(ids) > max_len:\\n            # Keep BOS; drop the tail before EOS; force EOS in the last slot.\\n            ids = ids[: max_len - 1] + [self.eos_idx]\\n        if len(ids) < max_len:\\n            ids = ids + [self.pad_idx] * (max_len - len(ids))\\n        return ids[:max_len]\\n\\n    def decode(self, token_ids: Sequence[int], strip_special: bool = True) -> str:\\n        tokens: List[str] = []\\n        for idx in token_ids:\\n            tok = self.idx2token.get(int(idx), self.unk_token)\\n            if tok == self.eos_token:\\n                break\\n            if strip_special and tok in self.special_tokens:\\n                continue\\n            if tok == self.unk_token and strip_special:\\n                continue\\n            tokens.append(tok)\\n        return \\"\\".join(tokens)\\n\\n    def save(self, path: Path | str) -> None:\\n        path = Path(path)\\n        path.parent.mkdir(parents=True, exist_ok=True)\\n        payload = {\\n            \\"token2idx\\": self.token2idx,\\n            \\"idx2token\\": [self.idx2token[i] for i in range(len(self.idx2token))],\\n        }\\n        path.write_text(json.dumps(payload, indent=4))\\n\\n    @classmethod\\n    def load(cls, path: Path | str) -> \\"SMILESVocabulary\\":\\n        raw = json.loads(Path(path).read_text())\\n        vocab = cls()\\n        if isinstance(raw, dict) and \\"idx2token\\" in raw:\\n            tokens = list(raw[\\"idx2token\\"])\\n        elif isinstance(raw, dict) and \\"token2idx\\" in raw:\\n            mapping = raw[\\"token2idx\\"]\\n            size = max(int(v) for v in mapping.values()) + 1\\n            tokens = [\\"\\"] * size\\n            for tok, idx in mapping.items():\\n                tokens[int(idx)] = tok\\n        elif isinstance(raw, dict) and all(isinstance(v, int) for v in raw.values()):\\n            # Bare token2idx map (scripts/build_vocab.py export).\\n            size = max(int(v) for v in raw.values()) + 1\\n            tokens = [\\"\\"] * size\\n            for tok, idx in raw.items():\\n                tokens[int(idx)] = tok\\n        elif isinstance(raw, list):\\n            tokens = list(raw)\\n        else:\\n            raise ValueError(f\\"Unrecognized vocab JSON in {path}\\")\\n        for tok in tokens:\\n            if tok and tok not in vocab.token2idx:\\n                vocab.add_token(tok)\\n        return vocab\\n\\n    @classmethod\\n    def from_token2idx(cls, token2idx: Dict[str, int]) -> \\"SMILESVocabulary\\":\\n        vocab = cls()\\n        size = max(token2idx.values()) + 1 if token2idx else len(vocab)\\n        ordered = [\\"\\"] * size\\n        for tok, idx in token2idx.items():\\n            ordered[int(idx)] = tok\\n        for tok in ordered:\\n            if tok and tok not in vocab.token2idx:\\n                vocab.add_token(tok)\\n        return vocab\\n\\n\\ndef load_packaged_vocabulary() -> SMILESVocabulary:\\n    \\"\\"\\"Load the frozen offline vocab shipped in ``src/smiles_vocab.json``.\\"\\"\\"\\n    if PACKAGED_VOCAB_PATH.exists():\\n        return SMILESVocabulary.load(PACKAGED_VOCAB_PATH)\\n    return SMILESVocabulary()\\n", "src/smiles_tokenizer.py": "\\"\\"\\"Compatibility wrapper: decoder/train code uses ``SmilesTokenizer``.\\n\\nThe chemically correct regex lives in ``src/chem_tokenizer.py``. This module\\nexposes the padded encode/decode API the Spec2FP decoder already calls, and\\nloads the frozen offline vocab from ``src/smiles_vocab.json`` when present.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Iterable, Sequence\\n\\nfrom src.chem_tokenizer import (\\n    PACKAGED_VOCAB_PATH,\\n    SMILESTokenizer,\\n    SMILESVocabulary,\\n    load_packaged_vocabulary,\\n)\\n\\nPAD, BOS, EOS, UNK = \\"<PAD>\\", \\"<BOS>\\", \\"<EOS>\\", \\"<UNK>\\"\\n_RAW = SMILESTokenizer()\\n\\n\\ndef tokenize_smiles(smiles: str) -> list[str]:\\n    return _RAW.tokenize(smiles)\\n\\n\\nclass SmilesTokenizer:\\n    \\"\\"\\"Thin adapter around ``SMILESVocabulary`` with TPU-padded ``encode``.\\"\\"\\"\\n\\n    def __init__(self, vocab: SMILESVocabulary | None = None) -> None:\\n        self.vocab = vocab or load_packaged_vocabulary()\\n\\n    @property\\n    def pad_id(self) -> int:\\n        return self.vocab.pad_idx\\n\\n    @property\\n    def bos_id(self) -> int:\\n        return self.vocab.bos_idx\\n\\n    @property\\n    def eos_id(self) -> int:\\n        return self.vocab.eos_idx\\n\\n    @property\\n    def unk_id(self) -> int:\\n        return self.vocab.unk_idx\\n\\n    @property\\n    def vocab_size(self) -> int:\\n        return len(self.vocab)\\n\\n    @property\\n    def token_to_id(self) -> dict[str, int]:\\n        return self.vocab.token2idx\\n\\n    @property\\n    def id_to_token(self) -> list[str]:\\n        return [self.vocab.idx2token[i] for i in range(len(self.vocab))]\\n\\n    def encode(self, smiles: str, max_len: int) -> list[int]:\\n        return self.vocab.encode_padded(smiles, max_len)\\n\\n    def decode(self, ids: Sequence[int], skip_special: bool = True) -> str:\\n        return self.vocab.decode(ids, strip_special=skip_special)\\n\\n    def save(self, path: Path | str) -> None:\\n        self.vocab.save(path)\\n\\n    @classmethod\\n    def load(cls, path: Path | str) -> \\"SmilesTokenizer\\":\\n        return cls(SMILESVocabulary.load(path))\\n\\n    @classmethod\\n    def from_tokens(cls, tokens: Sequence[str]) -> \\"SmilesTokenizer\\":\\n        vocab = SMILESVocabulary()\\n        for tok in tokens:\\n            if tok:\\n                vocab.add_token(str(tok))\\n        return cls(vocab)\\n\\n    @classmethod\\n    def default(cls) -> \\"SmilesTokenizer\\":\\n        return cls(load_packaged_vocabulary())\\n\\n\\ndef build_tokenizer(\\n    smiles_list: Iterable[str],\\n    *,\\n    min_count: int = 5,\\n    max_size: int | None = None,\\n) -> SmilesTokenizer:\\n    \\"\\"\\"Fit on ``smiles_list``. Prefer the packaged vocab when it already exists.\\"\\"\\"\\n    if PACKAGED_VOCAB_PATH.exists() and min_count >= 5:\\n        return SmilesTokenizer.default()\\n    vocab = SMILESVocabulary()\\n    vocab.build_vocab(smiles_list, min_freq=min_count)\\n    if max_size is not None and len(vocab) > max_size:\\n        # Keep specials + the first (max_size) tokens in insertion order.\\n        keep = [vocab.idx2token[i] for i in range(min(max_size, len(vocab)))]\\n        trimmed = SMILESVocabulary()\\n        for tok in keep:\\n            if tok not in trimmed.token2idx:\\n                trimmed.add_token(tok)\\n        vocab = trimmed\\n    return SmilesTokenizer(vocab)\\n", "src/smiles_vocab.json": "{\\n    \\"<PAD>\\": 0,\\n    \\"<BOS>\\": 1,\\n    \\"<EOS>\\": 2,\\n    \\"<UNK>\\": 3,\\n    \\"C\\": 4,\\n    \\"c\\": 5,\\n    \\"(\\": 6,\\n    \\")\\": 7,\\n    \\"O\\": 8,\\n    \\"1\\": 9,\\n    \\"2\\": 10,\\n    \\"=\\": 11,\\n    \\"N\\": 12,\\n    \\"n\\": 13,\\n    \\"3\\": 14,\\n    \\"F\\": 15,\\n    \\"S\\": 16,\\n    \\"4\\": 17,\\n    \\"-\\": 18,\\n    \\"o\\": 19,\\n    \\"Cl\\": 20,\\n    \\"[nH]\\": 21,\\n    \\"s\\": 22,\\n    \\"#\\": 23,\\n    \\"Br\\": 24,\\n    \\"5\\": 25,\\n    \\"P\\": 26,\\n    \\"[O-]\\": 27,\\n    \\"[N+]\\": 28,\\n    \\"6\\": 29,\\n    \\"7\\": 30,\\n    \\"[S+]\\": 31,\\n    \\"I\\": 32,\\n    \\"8\\": 33,\\n    \\"[n+]\\": 34,\\n    \\"9\\": 35,\\n    \\"[o+]\\": 36,\\n    \\"[2H]\\": 37,\\n    \\"[P+]\\": 38,\\n    \\"[Si]\\": 39,\\n    \\"%10\\": 40,\\n    \\"B\\": 41,\\n    \\"[N-]\\": 42,\\n    \\"%11\\": 43,\\n    \\"[C-]\\": 44,\\n    \\"[As]\\": 45,\\n    \\"[SH]\\": 46,\\n    \\"[n-]\\": 47,\\n    \\"%12\\": 48,\\n    \\"[Se]\\": 49,\\n    \\"/\\": 50,\\n    \\"\\\\\\\\\\": 51,\\n    \\"+\\": 52,\\n    \\".\\": 53,\\n    \\"@\\": 54,\\n    \\"[C@@H]\\": 55,\\n    \\"[C@H]\\": 56,\\n    \\"[C@@]\\": 57,\\n    \\"[C@]\\": 58,\\n    \\"[NH3+]\\": 59,\\n    \\"[NH2]\\": 60,\\n    \\"[nH+]\\": 61,\\n    \\"[O+]\\": 62,\\n    \\"[NH+]\\": 63,\\n    \\"%13\\": 64,\\n    \\"%14\\": 65,\\n    \\"%15\\": 66\\n}", "src/smiles_train.py": "\\"\\"\\"TPU-static training for the prefix-conditioned SMILES decoder.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nimport time\\nfrom pathlib import Path\\nfrom typing import Any, Sequence\\n\\nimport numpy as np\\nimport torch\\nfrom torch.utils.data import DataLoader, Dataset, RandomSampler, SequentialSampler\\n\\nfrom src.chem import randomize_smiles\\nfrom src.config import NUM_ADDUCTS, Config, get_config\\nfrom src.models.smiles_decoder import LightweightSmilesDecoder, decoder_from_config\\nfrom src.smiles_tokenizer import SmilesTokenizer, build_tokenizer\\nfrom src.tpu_trainer import get_device, set_seed, split_dataset, try_import_xla\\n\\n\\nclass SmilesLMDataset(Dataset):\\n    \\"\\"\\"Each item is already padded to ``cfg.smiles_max_len`` (XLA-static).\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        smiles: Sequence[str],\\n        masses: np.ndarray,\\n        fingerprints: np.ndarray,\\n        tokenizer: SmilesTokenizer,\\n        cfg: Config,\\n        *,\\n        adduct_ids: np.ndarray | None = None,\\n        randomize: bool | None = None,\\n    ) -> None:\\n        self.smiles = [str(s) for s in smiles]\\n        self.masses = np.ascontiguousarray(masses, dtype=np.float32).reshape(-1)\\n        self.fingerprints = np.ascontiguousarray(fingerprints, dtype=np.float32)\\n        self.tokenizer = tokenizer\\n        self.cfg = cfg\\n        self.max_len = int(cfg.smiles_max_len)\\n        self.randomize = cfg.smiles_randomize if randomize is None else bool(randomize)\\n        if adduct_ids is None:\\n            self.adduct_ids = None\\n        else:\\n            self.adduct_ids = np.ascontiguousarray(adduct_ids, dtype=np.int64).reshape(-1)\\n        n = len(self.smiles)\\n        assert self.masses.shape[0] == n\\n        assert self.fingerprints.shape[0] == n\\n\\n    def __len__(self) -> int:\\n        return len(self.smiles)\\n\\n    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:\\n        smi = self.smiles[idx]\\n        if self.randomize:\\n            smi = randomize_smiles(smi) or smi\\n        ids = self.tokenizer.encode(smi, self.max_len)\\n        tokens = torch.tensor(ids, dtype=torch.long)\\n        pad = tokens == self.tokenizer.pad_id\\n        if self.adduct_ids is None:\\n            adduct = int(np.random.randint(0, NUM_ADDUCTS))\\n        else:\\n            if np.random.rand() < 0.3:\\n                adduct = int(np.random.randint(0, NUM_ADDUCTS))\\n            else:\\n                adduct = int(self.adduct_ids[idx] % NUM_ADDUCTS)\\n        return {\\n            \\"fp\\": torch.from_numpy(self.fingerprints[idx]),\\n            \\"mass\\": torch.tensor([float(self.masses[idx])], dtype=torch.float32),\\n            \\"adduct_idx\\": torch.tensor(adduct, dtype=torch.long),\\n            \\"tgt_tokens\\": tokens,\\n            \\"tgt_mask\\": pad,\\n        }\\n\\n\\ndef collate_smiles(batch: list[dict[str, torch.Tensor]]) -> dict[str, torch.Tensor]:\\n    keys = batch[0].keys()\\n    return {k: torch.stack([b[k] for b in batch], dim=0) for k in keys}\\n\\n\\ndef _pad_to_bs(batch: dict[str, torch.Tensor], target: int) -> dict[str, torch.Tensor]:\\n    n = int(batch[\\"tgt_tokens\\"].shape[0])\\n    if n == target:\\n        return batch\\n    if n > target:\\n        return {k: v[:target] for k, v in batch.items()}\\n    pad = target - n\\n    out = {}\\n    for k, v in batch.items():\\n        pad_shape = (pad,) + tuple(v.shape[1:])\\n        z = torch.zeros(pad_shape, dtype=v.dtype, device=v.device)\\n        if k == \\"tgt_mask\\":\\n            z = torch.ones(pad_shape, dtype=v.dtype, device=v.device)\\n        out[k] = torch.cat([v, z], dim=0)\\n    return out\\n\\n\\ndef _shifted_labels(tgt_tokens: torch.Tensor, pad_id: int) -> torch.Tensor:\\n    labels = torch.full_like(tgt_tokens, pad_id)\\n    labels[:, :-1] = tgt_tokens[:, 1:]\\n    return labels\\n\\n\\ndef dataset_from_structures(\\n    smiles: Sequence[str],\\n    masses: np.ndarray,\\n    fingerprints: np.ndarray,\\n    tokenizer: SmilesTokenizer,\\n    cfg: Config,\\n    adduct_ids: np.ndarray | None = None,\\n) -> SmilesLMDataset:\\n    return SmilesLMDataset(smiles, masses, fingerprints, tokenizer, cfg, adduct_ids=adduct_ids)\\n\\n\\ndef save_decoder_checkpoint(\\n    model: LightweightSmilesDecoder,\\n    tokenizer: SmilesTokenizer,\\n    path: Path,\\n    cfg: Config,\\n    extra: dict[str, Any] | None = None,\\n) -> None:\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    tokenizer.save(cfg.smiles_vocab_path)\\n    payload = {\\n        \\"model\\": {k: v.detach().cpu() for k, v in model.state_dict().items()},\\n        \\"vocab\\": tokenizer.id_to_token,\\n        \\"extra\\": extra or {},\\n    }\\n    torch.save(payload, path)\\n\\n\\ndef load_decoder(\\n    path: Path | str,\\n    cfg: Config | None = None,\\n    map_location: str | torch.device = \\"cpu\\",\\n) -> tuple[LightweightSmilesDecoder, SmilesTokenizer]:\\n    cfg = cfg or get_config()\\n    try:\\n        blob = torch.load(str(path), map_location=map_location, weights_only=False)\\n    except TypeError:\\n        blob = torch.load(str(path), map_location=map_location)\\n    if \\"vocab\\" in blob:\\n        tokenizer = SmilesTokenizer.from_tokens(blob[\\"vocab\\"])\\n    elif cfg.smiles_vocab_path.exists():\\n        tokenizer = SmilesTokenizer.load(cfg.smiles_vocab_path)\\n    else:\\n        tokenizer = SmilesTokenizer.default()\\n    model = decoder_from_config(tokenizer, cfg)\\n    model.load_state_dict(blob[\\"model\\"] if \\"model\\" in blob else blob)\\n    model.eval()\\n    return model, tokenizer\\n\\n\\ndef train_smiles_decoder(\\n    dataset: SmilesLMDataset,\\n    tokenizer: SmilesTokenizer,\\n    cfg: Config | None = None,\\n    *,\\n    device=None,\\n    time_limit_s: float | None = None,\\n) -> LightweightSmilesDecoder:\\n    cfg = cfg or get_config()\\n    set_seed(cfg.seed)\\n    cfg.artifact_dir.mkdir(parents=True, exist_ok=True)\\n    xm, pl = try_import_xla()\\n    if device is None:\\n        device, use_xla = get_device(prefer_xla=True)\\n    else:\\n        use_xla = xm is not None and \\"xla\\" in str(device).lower()\\n\\n    bs = int(cfg.smiles_batch_size)\\n    n_epochs = int(cfg.smiles_num_epochs)\\n    train_ds, val_ds = split_dataset(dataset, cfg.val_fraction, cfg.seed)\\n    train_loader = DataLoader(\\n        train_ds,\\n        batch_size=bs,\\n        sampler=RandomSampler(train_ds),\\n        num_workers=cfg.num_workers,\\n        collate_fn=collate_smiles,\\n        drop_last=False,\\n        pin_memory=torch.cuda.is_available() and not use_xla,\\n    )\\n    val_loader = (\\n        DataLoader(\\n            val_ds,\\n            batch_size=bs,\\n            sampler=SequentialSampler(val_ds),\\n            collate_fn=collate_smiles,\\n            drop_last=False,\\n        )\\n        if len(val_ds) > 0\\n        else None\\n    )\\n\\n    model = decoder_from_config(tokenizer, cfg).to(device)\\n    pad_id = tokenizer.pad_id\\n    criterion = torch.nn.CrossEntropyLoss(\\n        ignore_index=pad_id, label_smoothing=float(cfg.smiles_label_smoothing)\\n    )\\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.smiles_lr, weight_decay=cfg.weight_decay)\\n    n_steps = max(1, math.ceil(len(train_ds) / bs) * n_epochs)\\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_steps)\\n\\n    probe = collate_smiles([dataset[0], dataset[min(1, len(dataset) - 1)]])\\n    assert tuple(probe[\\"tgt_tokens\\"].shape)[1] == cfg.smiles_max_len, probe[\\"tgt_tokens\\"].shape\\n    assert tuple(probe[\\"fp\\"].shape)[1] == cfg.fp_bits, probe[\\"fp\\"].shape\\n\\n    t0 = time.time()\\n    limit = cfg.smiles_train_time_limit_s if time_limit_s is None else time_limit_s\\n    global_step = 0\\n    best_val = float(\\"inf\\")\\n    print(\\n        f\\"[smiles] device={device} xla={use_xla} n_train={len(train_ds)} \\"\\n        f\\"n_val={len(val_ds)} vocab={tokenizer.vocab_size} max_len={cfg.smiles_max_len}\\"\\n    )\\n\\n    def _loss_on(batch: dict[str, torch.Tensor]) -> tuple[torch.Tensor, int]:\\n        if not use_xla:\\n            batch = {k: v.to(device) for k, v in batch.items()}\\n        n = int(batch[\\"tgt_tokens\\"].shape[0])\\n        batch = _pad_to_bs(batch, bs)\\n        logits = model(\\n            fp=batch[\\"fp\\"],\\n            mass=batch[\\"mass\\"],\\n            adduct_idx=batch[\\"adduct_idx\\"],\\n            tgt_tokens=batch[\\"tgt_tokens\\"],\\n            tgt_mask=batch[\\"tgt_mask\\"],\\n        )\\n        labels = _shifted_labels(batch[\\"tgt_tokens\\"], pad_id)\\n        loss = criterion(logits[:n].reshape(-1, tokenizer.vocab_size), labels[:n].reshape(-1))\\n        return loss, n\\n\\n    for epoch in range(n_epochs):\\n        model.train()\\n        if use_xla and pl is not None:\\n            iterator = pl.ParallelLoader(train_loader, [device]).per_device_loader(device)\\n        else:\\n            iterator = train_loader\\n        running = 0.0\\n        n_seen = 0\\n        for batch in iterator:\\n            if limit and (time.time() - t0) > limit:\\n                print(\\"[smiles] time limit reached, stopping\\")\\n                save_decoder_checkpoint(\\n                    model, tokenizer, cfg.smiles_checkpoint_path, cfg, extra={\\"epoch\\": epoch}\\n                )\\n                return model\\n            optimizer.zero_grad(set_to_none=True)\\n            loss, n = _loss_on(batch)\\n            loss.backward()\\n            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)\\n            if use_xla and xm is not None:\\n                xm.optimizer_step(optimizer)\\n                xm.mark_step()\\n            else:\\n                optimizer.step()\\n            scheduler.step()\\n            running += float(loss.detach().cpu().item()) * n\\n            n_seen += n\\n            global_step += 1\\n            if global_step % cfg.log_every == 0:\\n                print(\\n                    f\\"[smiles] epoch={epoch+1} step={global_step} \\"\\n                    f\\"loss={running / max(n_seen, 1):.4f} lr={scheduler.get_last_lr()[0]:.2e}\\"\\n                )\\n\\n        msg = f\\"[smiles] epoch {epoch+1}/{n_epochs} train_loss={running / max(n_seen, 1):.4f}\\"\\n        if val_loader is not None and len(val_ds) > 0:\\n            model.eval()\\n            vloss = []\\n            with torch.no_grad():\\n                it = (\\n                    pl.ParallelLoader(val_loader, [device]).per_device_loader(device)\\n                    if use_xla and pl is not None\\n                    else val_loader\\n                )\\n                for batch in it:\\n                    loss, _n = _loss_on(batch)\\n                    vloss.append(float(loss.detach().cpu().item()))\\n                    if use_xla and xm is not None:\\n                        xm.mark_step()\\n            vl = float(np.mean(vloss) if vloss else 0.0)\\n            msg += f\\" val_loss={vl:.4f}\\"\\n            if vl < best_val:\\n                best_val = vl\\n                save_decoder_checkpoint(\\n                    model,\\n                    tokenizer,\\n                    cfg.smiles_checkpoint_path,\\n                    cfg,\\n                    extra={\\"epoch\\": epoch, \\"val_loss\\": best_val},\\n                )\\n        else:\\n            save_decoder_checkpoint(\\n                model, tokenizer, cfg.smiles_checkpoint_path, cfg, extra={\\"epoch\\": epoch}\\n            )\\n        print(msg)\\n\\n    save_decoder_checkpoint(\\n        model, tokenizer, cfg.smiles_checkpoint_path, cfg, extra={\\"epoch\\": n_epochs}\\n    )\\n    return model\\n\\n\\ndef fit_tokenizer_from_smiles(smiles: Sequence[str], cfg: Config) -> SmilesTokenizer:\\n    tok = build_tokenizer(smiles, min_count=3, max_size=96)\\n    tok.save(cfg.smiles_vocab_path)\\n    return tok\\n", "src/retrieval.py": "\\"\\"\\"Candidate retrieval: precursor-mass filter + vectorized spectral similarity.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\nfrom src.chem import precursor_to_neutral_mass\\nfrom src.config import Config\\nfrom src.preprocessing import bin_spectrum\\n\\n\\n@dataclass\\nclass StructureIndex:\\n    \\"\\"\\"Mass-sorted unique 2D structures (one row per InChIKey14).\\"\\"\\"\\n\\n    smiles: np.ndarray  # object\\n    inchikey14: np.ndarray  # object / U14\\n    exact_mass: np.ndarray  # float64\\n    fingerprints: np.ndarray  # uint8 (N, fp_bits) or (N, 0) if packed-only\\n    order: np.ndarray  # argsort of exact_mass\\n    sorted_mass: np.ndarray\\n    peak_mz: np.ndarray | None = None  # (N, top_n)\\n    peak_intensity: np.ndarray | None = None\\n    peak_mask: np.ndarray | None = None\\n    formulas: np.ndarray | None = None  # object / str\\n    fp_packed: np.ndarray | None = None  # uint8 (N, fp_bits/8)\\n    ionization_mode: np.ndarray | None = None  # object / str\\n    # Original featurize_spectrum outputs for the same representative as peak_mz.\\n    # Optional for structure-only candidate libraries; required by corrected training.\\n    peak_nl: np.ndarray | None = None  # (N, top_n)\\n    precursor_feat: np.ndarray | None = None  # (N, precursor_feat_dim)\\n    adduct_id: np.ndarray | None = None  # (N,)\\n\\n    @classmethod\\n    def build(\\n        cls,\\n        smiles: list[str] | np.ndarray,\\n        inchikey14: list[str] | np.ndarray,\\n        exact_mass: np.ndarray,\\n        fingerprints: np.ndarray,\\n        peak_mz: np.ndarray | None = None,\\n        peak_intensity: np.ndarray | None = None,\\n        peak_mask: np.ndarray | None = None,\\n        formulas: list[str] | np.ndarray | None = None,\\n        fp_packed: np.ndarray | None = None,\\n        ionization_mode: list[str] | np.ndarray | None = None,\\n        peak_nl: np.ndarray | None = None,\\n        precursor_feat: np.ndarray | None = None,\\n        adduct_id: np.ndarray | None = None,\\n    ) -> \\"StructureIndex\\":\\n        mass = np.asarray(exact_mass, dtype=np.float64)\\n        order = np.argsort(mass, kind=\\"mergesort\\")\\n        packed = None if fp_packed is None else np.asarray(fp_packed, dtype=np.uint8)\\n        return cls(\\n            smiles=np.asarray(smiles, dtype=object),\\n            inchikey14=np.asarray(inchikey14, dtype=object),\\n            exact_mass=mass,\\n            fingerprints=np.asarray(fingerprints),\\n            order=order,\\n            sorted_mass=mass[order],\\n            peak_mz=None if peak_mz is None else np.asarray(peak_mz, dtype=np.float32),\\n            peak_intensity=None\\n            if peak_intensity is None\\n            else np.asarray(peak_intensity, dtype=np.float32),\\n            peak_mask=None if peak_mask is None else np.asarray(peak_mask, dtype=np.float32),\\n            formulas=None if formulas is None else np.asarray(formulas, dtype=object),\\n            fp_packed=packed,\\n            ionization_mode=None\\n            if ionization_mode is None\\n            else np.asarray(ionization_mode, dtype=object),\\n            peak_nl=None if peak_nl is None else np.asarray(peak_nl, dtype=np.float32),\\n            precursor_feat=None\\n            if precursor_feat is None\\n            else np.asarray(precursor_feat, dtype=np.float32),\\n            adduct_id=None if adduct_id is None else np.asarray(adduct_id, dtype=np.int64),\\n        )\\n\\n    def mass_window(\\n        self,\\n        query_mass: float,\\n        *,\\n        ppm: float,\\n        abs_da: float,\\n    ) -> np.ndarray:\\n        \\"\\"\\"Return *unsorted* structure indices whose exact mass is within the window.\\"\\"\\"\\n        if not np.isfinite(query_mass) or query_mass <= 0:\\n            return np.zeros((0,), dtype=np.int64)\\n        tol = max(float(abs_da), float(ppm) * query_mass * 1e-6)\\n        lo = query_mass - tol\\n        hi = query_mass + tol\\n        left = int(np.searchsorted(self.sorted_mass, lo, side=\\"left\\"))\\n        right = int(np.searchsorted(self.sorted_mass, hi, side=\\"right\\"))\\n        if right <= left:\\n            return np.zeros((0,), dtype=np.int64)\\n        return self.order[left:right].astype(np.int64, copy=False)\\n\\n    def query_masses(\\n        self,\\n        masses: np.ndarray | list[float],\\n        cfg: Config,\\n        *,\\n        use_fallback: bool = False,\\n        ppm: float | None = None,\\n        abs_da: float | None = None,\\n        max_candidates: int | None = None,\\n    ) -> np.ndarray:\\n        \\"\\"\\"Union of candidates for several plausible query masses (multi-adduct).\\"\\"\\"\\n        ppm_v = float(cfg.mass_ppm if ppm is None else ppm)\\n        abs_v = float(cfg.mass_abs_da if abs_da is None else abs_da)\\n        cap = int(cfg.max_mass_candidates if max_candidates is None else max_candidates)\\n        idx_sets: list[np.ndarray] = []\\n        mass_arr = np.asarray(masses, dtype=np.float64).ravel()\\n        for m in mass_arr:\\n            hit = self.mass_window(float(m), ppm=ppm_v, abs_da=abs_v)\\n            if hit.size:\\n                idx_sets.append(hit)\\n        if not idx_sets and use_fallback:\\n            for extra in cfg.mass_fallback_da:\\n                for m in mass_arr:\\n                    hit = self.mass_window(float(m), ppm=0.0, abs_da=float(extra))\\n                    if hit.size:\\n                        idx_sets.append(hit)\\n                if idx_sets:\\n                    break\\n        if not idx_sets:\\n            return np.zeros((0,), dtype=np.int64)\\n        uniq = np.unique(np.concatenate(idx_sets))\\n        if uniq.size > cap:\\n            primary = float(mass_arr[0]) if mass_arr.size else 0.0\\n            err = np.abs(self.exact_mass[uniq] - primary)\\n            keep = np.argpartition(err, cap - 1)[:cap]\\n            uniq = uniq[keep]\\n        return uniq.astype(np.int64)\\n\\n    def query_masses_progressive(\\n        self,\\n        masses: np.ndarray | list[float],\\n        cfg: Config,\\n        *,\\n        min_hits: int = 25,\\n        stages: list[tuple[float, float]] | None = None,\\n    ) -> np.ndarray:\\n        \\"\\"\\"Widen ppm / Da until at least ``min_hits`` unique skeletons (or the last stage).\\"\\"\\"\\n        seen: list[np.ndarray] = []\\n        have: set[int] = set()\\n        if stages is None:\\n            stages = [\\n                (float(cfg.mass_ppm), float(cfg.mass_abs_da)),\\n                (25.0, 0.03),\\n                (50.0, 0.05),\\n            ]\\n        for ppm, abs_da in stages:\\n            hit = self.query_masses(\\n                masses,\\n                cfg,\\n                use_fallback=False,\\n                ppm=ppm,\\n                abs_da=abs_da,\\n                max_candidates=max(int(cfg.max_mass_candidates), 8192),\\n            )\\n            if hit.size:\\n                new = [int(i) for i in hit.tolist() if int(i) not in have]\\n                if new:\\n                    have.update(new)\\n                    seen.append(np.asarray(new, dtype=np.int64))\\n            if len(have) >= int(min_hits):\\n                break\\n        if not seen:\\n            return np.zeros((0,), dtype=np.int64)\\n        return np.concatenate(seen)\\n\\n\\ndef binned_cosine(\\n    query_mz: np.ndarray,\\n    query_int: np.ndarray,\\n    query_mask: np.ndarray,\\n    cand_mz: np.ndarray,\\n    cand_int: np.ndarray,\\n    cand_mask: np.ndarray,\\n    cfg: Config,\\n) -> np.ndarray:\\n    \\"\\"\\"Cosine similarity of fixed-bin spectra. ``cand_*`` has shape ``(N, top_n)``.\\"\\"\\"\\n    q = bin_spectrum(query_mz, query_int, query_mask, cfg=cfg)\\n    qn = float(np.linalg.norm(q))\\n    if qn <= 0 or cand_mz.size == 0:\\n        return np.zeros((cand_mz.shape[0],), dtype=np.float32)\\n    n = cand_mz.shape[0]\\n    scores = np.zeros(n, dtype=np.float32)\\n    # Small N after mass filter: loop is fine and keeps memory bounded.\\n    for i in range(n):\\n        c = bin_spectrum(cand_mz[i], cand_int[i], cand_mask[i] if cand_mask is not None else None, cfg=cfg)\\n        cn = float(np.linalg.norm(c))\\n        if cn <= 0:\\n            continue\\n        scores[i] = float(np.dot(q, c) / (qn * cn))\\n    return scores\\n\\n\\ndef peak_cosine_matrix(\\n    query_mz: np.ndarray,\\n    query_int: np.ndarray,\\n    query_mask: np.ndarray,\\n    cand_mz: np.ndarray,\\n    cand_int: np.ndarray,\\n    cand_mask: np.ndarray,\\n    tol: float,\\n) -> np.ndarray:\\n    \\"\\"\\"Greedy peak-to-peak cosine, vectorized match search with a numpy greedy pass.\\n\\n    Shapes: query ``(P,)``, candidates ``(N, P)``.\\n    \\"\\"\\"\\n    q_m = query_mask > 0\\n    q_mz = query_mz[q_m]\\n    q_i = query_int[q_m]\\n    n = cand_mz.shape[0]\\n    scores = np.zeros(n, dtype=np.float32)\\n    if q_mz.size == 0:\\n        return scores\\n    q_norm = float(np.sqrt(np.dot(q_i, q_i)))\\n    if q_norm <= 0:\\n        return scores\\n    for i in range(n):\\n        c_m = cand_mask[i] > 0\\n        c_mz = cand_mz[i, c_m]\\n        c_i = cand_int[i, c_m]\\n        if c_mz.size == 0:\\n            continue\\n        c_norm = float(np.sqrt(np.dot(c_i, c_i)))\\n        if c_norm <= 0:\\n            continue\\n        # Pairwise |\\u0394m/z|\\n        delta = np.abs(c_mz[:, None] - q_mz[None, :])\\n        match = delta <= tol\\n        if not match.any():\\n            continue\\n        # Greedy: highest intensity product first.\\n        prod = (c_i[:, None] * q_i[None, :]) * match\\n        used_c = np.zeros(c_mz.size, dtype=bool)\\n        used_q = np.zeros(q_mz.size, dtype=bool)\\n        dot = 0.0\\n        flat = prod.ravel()\\n        order = np.argsort(flat)[::-1]\\n        n_c = c_mz.size\\n        n_q = q_mz.size\\n        for idx in order:\\n            val = float(flat[idx])\\n            if val <= 0:\\n                break\\n            ci = int(idx) // n_q\\n            qi = int(idx) % n_q\\n            if used_c[ci] or used_q[qi]:\\n                continue\\n            used_c[ci] = True\\n            used_q[qi] = True\\n            dot += val\\n        scores[i] = float(dot / (q_norm * c_norm))\\n    return scores\\n\\n\\ndef modified_cosine(\\n    query_mz: np.ndarray,\\n    query_int: np.ndarray,\\n    query_mask: np.ndarray,\\n    query_prec: float,\\n    cand_mz: np.ndarray,\\n    cand_int: np.ndarray,\\n    cand_mask: np.ndarray,\\n    cand_prec: np.ndarray,\\n    tol: float,\\n    *,\\n    return_n_match: bool = False,\\n) -> np.ndarray | tuple[np.ndarray, np.ndarray]:\\n    \\"\\"\\"GNPS-style modified cosine: match fragments *or* precursor-shifted fragments.\\"\\"\\"\\n    q_m = query_mask > 0\\n    q_mz = query_mz[q_m]\\n    q_i = query_int[q_m]\\n    n = cand_mz.shape[0]\\n    scores = np.zeros(n, dtype=np.float32)\\n    n_match = np.zeros(n, dtype=np.float32)\\n    if q_mz.size == 0:\\n        return (scores, n_match) if return_n_match else scores\\n    q_norm = float(np.sqrt(np.dot(q_i, q_i)))\\n    if q_norm <= 0:\\n        return (scores, n_match) if return_n_match else scores\\n    for i in range(n):\\n        c_m = cand_mask[i] > 0\\n        c_mz = cand_mz[i, c_m]\\n        c_i = cand_int[i, c_m]\\n        if c_mz.size == 0:\\n            continue\\n        c_norm = float(np.sqrt(np.dot(c_i, c_i)))\\n        if c_norm <= 0:\\n            continue\\n        shift = float(query_prec) - float(cand_prec[i])\\n        delta = np.abs(c_mz[:, None] - q_mz[None, :])\\n        delta_shift = np.abs(c_mz[:, None] - (q_mz[None, :] - shift))\\n        match = (delta <= tol) | (delta_shift <= tol)\\n        if not match.any():\\n            continue\\n        prod = (c_i[:, None] * q_i[None, :]) * match\\n        used_c = np.zeros(c_mz.size, dtype=bool)\\n        used_q = np.zeros(q_mz.size, dtype=bool)\\n        dot = 0.0\\n        n_q = q_mz.size\\n        flat = prod.ravel()\\n        order = np.argsort(flat)[::-1]\\n        n_hits = 0\\n        for idx in order:\\n            val = float(flat[idx])\\n            if val <= 0:\\n                break\\n            ci = int(idx) // n_q\\n            qi = int(idx) % n_q\\n            if used_c[ci] or used_q[qi]:\\n                continue\\n            used_c[ci] = True\\n            used_q[qi] = True\\n            dot += val\\n            n_hits += 1\\n        scores[i] = float(dot / (q_norm * c_norm))\\n        n_match[i] = float(n_hits)\\n    if return_n_match:\\n        return scores, n_match\\n    return scores\\n\\n\\ndef tanimoto(query_fp: np.ndarray, cand_fp: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Tanimoto between a 1-D query fingerprint and ``(N, bits)`` candidates.\\"\\"\\"\\n    q = query_fp.astype(np.float32, copy=False).ravel()\\n    c = cand_fp.astype(np.float32, copy=False)\\n    if c.ndim == 1:\\n        c = c[None, :]\\n    inter = (c * q[None, :]).sum(axis=1)\\n    union = c.sum(axis=1) + q.sum() - inter\\n    return (inter / np.clip(union, 1e-6, None)).astype(np.float32)\\n\\n\\ndef fingerprint_knn(\\n    query_fp: np.ndarray,\\n    cand_fp: np.ndarray,\\n    *,\\n    k: int = 25,\\n    threshold: float = 0.35,\\n    chunk: int = 8192,\\n    exclude: set[int] | None = None,\\n) -> tuple[np.ndarray, np.ndarray]:\\n    \\"\\"\\"Top-``k`` Tanimoto neighbors. ``cand_fp`` may be uint8 of shape ``(N, bits)``.\\"\\"\\"\\n    n = int(cand_fp.shape[0])\\n    if n == 0:\\n        return np.zeros((0,), dtype=np.int64), np.zeros((0,), dtype=np.float32)\\n    q = np.asarray(query_fp, dtype=np.float32).ravel()\\n    bits = (q >= float(threshold)).astype(np.uint8)\\n    if int(bits.sum()) == 0:\\n        kk = min(64, q.size)\\n        topb = np.argpartition(q, -kk)[-kk:]\\n        bits = np.zeros_like(bits)\\n        bits[topb] = 1\\n    qsum = int(bits.sum())\\n    scores = np.empty(n, dtype=np.float32)\\n    for start in range(0, n, int(chunk)):\\n        sl = slice(start, min(start + int(chunk), n))\\n        c = np.asarray(cand_fp[sl])\\n        if c.dtype != np.uint8:\\n            c = (c >= 0.5).astype(np.uint8)\\n        inter = np.bitwise_and(c, bits).sum(axis=1).astype(np.float32)\\n        csum = c.sum(axis=1).astype(np.float32)\\n        scores[sl] = inter / np.clip(csum + qsum - inter, 1e-6, None)\\n    if exclude:\\n        for i in exclude:\\n            if 0 <= int(i) < n:\\n                scores[int(i)] = -1.0\\n    k = min(int(k), n)\\n    pick = np.argpartition(-scores, k - 1)[:k]\\n    pick = pick[np.argsort(-scores[pick])]\\n    pick = pick[scores[pick] >= 0]\\n    return pick.astype(np.int64), scores[pick]\\n\\n\\ndef fingerprint_cosine(query_fp: np.ndarray, cand_fp: np.ndarray) -> np.ndarray:\\n    q = query_fp.astype(np.float32, copy=False).ravel()\\n    c = cand_fp.astype(np.float32, copy=False)\\n    if c.ndim == 1:\\n        c = c[None, :]\\n    qn = float(np.linalg.norm(q))\\n    cn = np.linalg.norm(c, axis=1)\\n    if qn <= 0:\\n        return np.zeros(c.shape[0], dtype=np.float32)\\n    return (c @ q / (np.clip(cn, 1e-6, None) * qn)).astype(np.float32)\\n\\n\\ndef query_neutral_masses(precursor_mz: np.ndarray, adducts: np.ndarray | list[str]) -> np.ndarray:\\n    masses = []\\n    for mz, ad in zip(np.asarray(precursor_mz).ravel(), list(adducts)):\\n        masses.append(precursor_to_neutral_mass(float(mz), str(ad)))\\n    return np.asarray(masses, dtype=np.float64)\\n\\n\\ndef _is_structure_table(path: Path) -> bool:\\n    name = path.name.lower()\\n    return name.endswith(\\".parquet\\") or name.endswith(\\".csv\\") or name.endswith(\\".csv.gz\\")\\n\\n\\ndef load_external_structure_index(path: str | None, cfg: Config) -> StructureIndex | None:\\n    \\"\\"\\"Class 2: optional LOTUS / PubChem / COCONUT table.\\n\\n    Expected columns: ``smiles`` or ``normalized_smiles``, optional\\n    ``inchikey14``, ``exact_mass`` / ``molecular_formula``.\\n    Packed Morgan bits stay packed (~100 MB for 400k rows). Mass-window hits\\n    are unpacked at query time in ``_fingerprints_for``.\\n    \\"\\"\\"\\n    from pathlib import Path\\n\\n    import pandas as pd\\n\\n    from src.chem import formula_to_mass, inchikey14_from_smiles, morgan_fingerprint\\n\\n    keywords = (\\"coconut\\", \\"pubchem\\", \\"chembl\\", \\"lotus\\", \\"external\\", \\"class2\\", \\"candidate\\")\\n    skip_names = {\\"train.parquet\\", \\"test.parquet\\", \\"sample_submission.csv\\"}\\n    here = Path(__file__).resolve().parent.parent\\n    candidates: list[Path] = []\\n    if path:\\n        candidates.append(Path(path))\\n    if cfg.external_candidates_path:\\n        candidates.append(Path(cfg.external_candidates_path))\\n    for name in (\\n        \\"class2_candidates.parquet\\",\\n        \\"class2_candidates.csv.gz\\",\\n        \\"class2_candidates.csv\\",\\n        \\"lotus.parquet\\",\\n        \\"pubchem_candidates.parquet\\",\\n        \\"coconut.parquet\\",\\n        \\"external_candidates.parquet\\",\\n        \\"pubchem_candidates.csv\\",\\n        \\"coconut.csv\\",\\n    ):\\n        candidates.append(cfg.data_dir / name)\\n        candidates.append(cfg.artifact_dir / name)\\n        candidates.append(here / \\"data\\" / name)\\n        candidates.append(Path(\\"/kaggle/working\\") / name)\\n        candidates.append(Path(\\"/kaggle/working\\") / \\"data\\" / name)\\n        candidates.append(Path(\\"/kaggle/input\\") / name)\\n        candidates.append(Path(\\"/kaggle/input/class2_candidates.parquet2\\") / name)\\n        candidates.append(Path(\\"/kaggle/input/class2-candidates-parquet2\\") / name)\\n        candidates.append(Path(\\"/kaggle/input/class2_candidates.parquet\\") / name)\\n    search_roots = [\\n        Path(\\"/kaggle/input\\"),\\n        Path(\\"/kaggle/working\\"),\\n        Path(cfg.data_dir),\\n        Path(cfg.artifact_dir),\\n        here / \\"data\\",\\n    ]\\n    for root in search_roots:\\n        if not root.exists():\\n            continue\\n        try:\\n            for p in root.rglob(\\"*\\"):\\n                if not p.is_file() or p.name in skip_names:\\n                    continue\\n                if not _is_structure_table(p):\\n                    continue\\n                low = p.name.lower()\\n                if any(k in low for k in keywords):\\n                    candidates.append(p)\\n        except Exception:\\n            continue\\n    seen: set[str] = set()\\n    hit = None\\n    for p in candidates:\\n        key = str(p.resolve()) if p.exists() else str(p)\\n        if key in seen:\\n            continue\\n        seen.add(key)\\n        if p.exists() and p.is_file():\\n            hit = p\\n            break\\n    if hit is None:\\n        print(\\"[class2] no external structure table found\\")\\n        return None\\n    name = hit.name.lower()\\n    try:\\n        if name.endswith(\\".parquet\\"):\\n            df = pd.read_parquet(hit)\\n        else:\\n            df = pd.read_csv(hit)\\n    except Exception as exc:\\n        print(f\\"[class2] failed to read {hit}: {exc}\\")\\n        return None\\n    smi_col = \\"normalized_smiles\\" if \\"normalized_smiles\\" in df.columns else \\"smiles\\"\\n    if smi_col not in df.columns:\\n        print(f\\"[class2] {hit} has no smiles column, skipping\\")\\n        return None\\n    smiles = df[smi_col].astype(str).tolist()\\n    if \\"inchikey14\\" in df.columns:\\n        keys = df[\\"inchikey14\\"].astype(str).tolist()\\n    else:\\n        keys = [inchikey14_from_smiles(s) or s for s in smiles]\\n    if \\"exact_mass\\" in df.columns:\\n        masses = df[\\"exact_mass\\"].to_numpy(dtype=np.float64)\\n    elif \\"molecular_formula\\" in df.columns:\\n        masses = np.array([formula_to_mass(f) for f in df[\\"molecular_formula\\"].astype(str)], dtype=np.float64)\\n    else:\\n        from src.chem import exact_mass_from_smiles\\n\\n        masses = np.array([exact_mass_from_smiles(s) for s in smiles], dtype=np.float64)\\n    n = len(smiles)\\n    packed = None\\n    if \\"fp_packed\\" in df.columns:\\n        packed_rows = []\\n        for x in df[\\"fp_packed\\"].tolist():\\n            if isinstance(x, (bytes, bytearray, memoryview)):\\n                packed_rows.append(np.frombuffer(x, dtype=np.uint8))\\n            else:\\n                packed_rows.append(np.asarray(x, dtype=np.uint8).ravel())\\n        packed = np.stack(packed_rows).astype(np.uint8, copy=False)\\n        fps = np.zeros((n, 0), dtype=np.uint8)\\n        print(f\\"[class2] keeping packed Morgan fingerprints {packed.shape} from {hit} (lazy unpack)\\")\\n    elif \\"fingerprint\\" in df.columns:\\n        fps = np.stack(df[\\"fingerprint\\"].to_numpy())\\n    elif n <= 250_000:\\n        print(f\\"[class2] computing Morgan fingerprints for {n} structures from {hit} ...\\")\\n        fps = np.stack([morgan_fingerprint(s, n_bits=cfg.fp_bits, radius=cfg.morgan_radius) for s in smiles])\\n    else:\\n        print(f\\"[class2] skipping full-table fingerprints for {n} structures (computed on mass hits)\\")\\n        fps = np.zeros((n, 0), dtype=np.uint8)\\n    formulas = None\\n    if \\"formula\\" in df.columns:\\n        formulas = df[\\"formula\\"].astype(str).tolist()\\n    elif \\"molecular_formula\\" in df.columns:\\n        formulas = df[\\"molecular_formula\\"].astype(str).tolist()\\n    print(f\\"[class2] loaded {n} structures from {hit} mass {float(np.nanmin(masses)):.1f}\\u2013{float(np.nanmax(masses)):.1f} Da\\")\\n    return StructureIndex.build(smiles, keys, masses, fps, formulas=formulas, fp_packed=packed)\\n", "src/analogs.py": "\\"\\"\\"Mass-aware analog expansion of retrieved SMILES (Class 2 / pad-to-25).\\n\\nApplies a small set of NP-like biotransformations with RDKit. New skeletons\\nthat land inside the query mass window are ranked first; remaining analogs\\nfill leftover MRR@25 slots.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Iterable, Sequence\\n\\nimport numpy as np\\n\\nfrom src.chem import exact_mass_from_smiles, inchikey14_from_smiles, smiles_to_mol\\n\\n# Neutral-mass deltas for parent retrieval (hexose, deoxyhexose, pentose, ...).\\nMASS_SHIFTS: tuple[tuple[str, float], ...] = (\\n    (\\"hexose\\", 162.052823),\\n    (\\"deoxyhexose\\", 146.057909),\\n    (\\"pentose\\", 132.042259),\\n    (\\"glucuronide\\", 176.032088),\\n    (\\"methyl\\", 14.015650),\\n    (\\"oxygen\\", 15.994915),\\n    (\\"acetyl\\", 42.010565),\\n    (\\"water\\", 18.010565),\\n)\\n\\n# SMARTS: reactant >> product. Keep this list tiny \\u2014 it runs per test molecule.\\n_RXN_SMARTS: tuple[str, ...] = (\\n    \\"[cH:1]>>[c:1]O\\",           # aromatic hydroxylation\\n    \\"[CH2:1]>>[CH:1]O\\",         # aliphatic hydroxylation\\n    \\"[OH:1]>>[O:1]C\\",           # O-methylation\\n    \\"[NH:1]>>[N:1]C\\",           # N-methylation\\n    \\"[O:1][CH3]>>[OH:1]\\",       # O-demethylation\\n    \\"[N:1][CH3]>>[NH:1]\\",       # N-demethylation\\n    \\"[c:1][CH3]>>[cH:1]\\",       # aromatic demethylation\\n    \\"[cH:1]>>[c:1]C\\",           # aromatic methylation\\n    \\"[cH:1]>>[c:1]Cl\\",          # aromatic chlorination\\n    \\"[OH:1]>>[O:1]C(=O)C\\",      # O-acetylation\\n    \\"[C:1](=O)[OH]>>[C:1](=O)OC\\",  # methyl ester\\n    \\"[C:1](=O)O[CH3]>>[C:1](=O)O\\",  # ester hydrolysis\\n    # O-glycosylation / glycosidic cleavage (hexose-like ring).\\n    \\"[OH:1]>>[O:1][C@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@H]1O\\",\\n    \\"[#8:1]-[#6]1-[#8]-[#6]-[#6]-[#6]-[#6]-1>>[#8H:1]\\",\\n    \\"[OH:1]>>[O:1]C(=O)C\\",  # acetyl (aliphatic) already above; keep glycosyl pair\\n)\\n\\n\\ndef _reactions():\\n    if not hasattr(_reactions, \\"_cache\\"):\\n        try:\\n            from rdkit.Chem import AllChem\\n\\n            rxns = []\\n            for smarts in _RXN_SMARTS:\\n                try:\\n                    rxn = AllChem.ReactionFromSmarts(smarts)\\n                    if rxn is not None:\\n                        rxns.append(rxn)\\n                except Exception:\\n                    continue\\n            _reactions._cache = rxns  # type: ignore[attr-defined]\\n        except Exception:\\n            _reactions._cache = []  # type: ignore[attr-defined]\\n    return _reactions._cache  # type: ignore[attr-defined]\\n\\n\\ndef shifted_parent_masses(query_masses: Sequence[float]) -> list[float]:\\n    \\"\\"\\"Query masses of putative glycosylated / analog parents.\\"\\"\\"\\n    out: list[float] = []\\n    seen: set[float] = set()\\n    for raw in query_masses:\\n        m = float(raw)\\n        if m <= 0 or not np.isfinite(m):\\n            continue\\n        for _, delta in MASS_SHIFTS:\\n            for parent in (m + delta, m - delta):\\n                key = round(parent, 4)\\n                if key in seen or parent <= 0:\\n                    continue\\n                seen.add(key)\\n                out.append(parent)\\n    return out\\n\\n\\ndef expand_smiles(\\n    smiles_list: Sequence[str],\\n    *,\\n    query_mass: float,\\n    mass_ppm: float = 50.0,\\n    max_per_parent: int = 12,\\n    max_total: int = 80,\\n    in_window_only: bool = False,\\n) -> list[str]:\\n    \\"\\"\\"Return unique analog SMILES, mass-matching ones first.\\"\\"\\"\\n    rxns = _reactions()\\n    if not rxns:\\n        return []\\n    in_window: list[str] = []\\n    other: list[str] = []\\n    seen: set[str] = set()\\n    for parent in smiles_list:\\n        mol = smiles_to_mol(parent)\\n        if mol is None:\\n            continue\\n        parent_key = inchikey14_from_smiles(parent)\\n        if parent_key:\\n            seen.add(parent_key)\\n        n_from_parent = 0\\n        for rxn in rxns:\\n            if n_from_parent >= int(max_per_parent):\\n                break\\n            try:\\n                products = rxn.RunReactants((mol,))\\n            except Exception:\\n                continue\\n            for tup in products:\\n                if not tup:\\n                    continue\\n                prod = tup[0]\\n                try:\\n                    from rdkit import Chem\\n\\n                    Chem.SanitizeMol(prod)\\n                    smi = Chem.MolToSmiles(prod, canonical=True)\\n                except Exception:\\n                    continue\\n                key = inchikey14_from_smiles(smi)\\n                if not key or key in seen:\\n                    continue\\n                seen.add(key)\\n                n_from_parent += 1\\n                mass = exact_mass_from_smiles(smi)\\n                ppm = (\\n                    abs(mass - query_mass) / query_mass * 1e6\\n                    if query_mass > 0 and mass > 0\\n                    else 1e9\\n                )\\n                if ppm <= float(mass_ppm):\\n                    in_window.append(smi)\\n                elif not in_window_only:\\n                    other.append(smi)\\n                if len(in_window) + len(other) >= int(max_total):\\n                    return in_window + other\\n    return in_window if in_window_only else in_window + other\\n\\n\\ndef expand_unique(smiles_iter: Iterable[str], **kwargs) -> list[str]:\\n    return expand_smiles(list(smiles_iter), **kwargs)\\n", "src/neighbors.py": "\\"\\"\\"MS2Query-style spectral-neighbor fingerprints (binned cosine kNN).\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom typing import Any\\n\\nimport numpy as np\\n\\nfrom src.config import Config\\nfrom src.preprocessing import bin_spectrum\\n\\n\\ndef _l2_normalize(x: np.ndarray, dtype=np.float32) -> np.ndarray:\\n    arr = np.asarray(x, dtype=np.float32)\\n    if arr.ndim == 1:\\n        n = float(np.linalg.norm(arr))\\n        if n < 1e-8:\\n            return np.zeros_like(arr, dtype=dtype)\\n        return (arr / n).astype(dtype, copy=False)\\n    n = np.linalg.norm(arr, axis=-1, keepdims=True)\\n    return (arr / np.clip(n, 1e-8, None)).astype(dtype, copy=False)\\n\\n\\ndef _softmax(x: np.ndarray, temperature: float = 0.15) -> np.ndarray:\\n    z = np.asarray(x, dtype=np.float64) / max(float(temperature), 1e-6)\\n    z = z - float(np.max(z))\\n    e = np.exp(z)\\n    s = float(e.sum())\\n    if s <= 0:\\n        return np.full(x.shape, 1.0 / max(x.size, 1), dtype=np.float32)\\n    return (e / s).astype(np.float32)\\n\\n\\ndef _ion_key(value: Any) -> str:\\n    s = str(value or \\"\\").strip().lower()\\n    if s.startswith(\\"neg\\") or s in {\\"-\\", \\"n\\"}:\\n        return \\"negative\\"\\n    if s.startswith(\\"pos\\") or s in {\\"+\\", \\"p\\"}:\\n        return \\"positive\\"\\n    return s\\n\\n\\ndef blend_fingerprints(\\n    spec2fp: np.ndarray,\\n    neighbor_fp: np.ndarray,\\n    alpha: float = 0.5,\\n) -> np.ndarray:\\n    \\"\\"\\"``fp = a * spec2fp + (1-a) * neighbor_fp``, clipped to [0, 1].\\"\\"\\"\\n    a = float(np.clip(alpha, 0.0, 1.0))\\n    out = a * np.asarray(spec2fp, dtype=np.float32) + (1.0 - a) * np.asarray(neighbor_fp, dtype=np.float32)\\n    return np.clip(out, 0.0, 1.0).astype(np.float32)\\n\\n\\n@dataclass\\nclass SpectralNeighborIndex:\\n    \\"\\"\\"L2-normalized binned library spectra for chunked cosine kNN.\\"\\"\\"\\n\\n    binned: np.ndarray  # (N, n_bins) float16/float32, L2-normalized\\n    fingerprints: np.ndarray  # (N, bits) uint8 / float\\n    ion_code: np.ndarray | None = None  # 0 unknown, 1 pos, 2 neg\\n    chunk: int = 4096\\n\\n    @classmethod\\n    def from_structure_index(cls, index, cfg: Config, *, chunk: int = 4096) -> \\"SpectralNeighborIndex\\":\\n        if getattr(index, \\"peak_mz\\", None) is None:\\n            raise ValueError(\\"structure index has no representative peaks\\")\\n        n = int(index.peak_mz.shape[0])\\n        binned = np.zeros((n, cfg.n_mz_bins), dtype=np.float32)\\n        for i in range(n):\\n            binned[i] = bin_spectrum(\\n                index.peak_mz[i],\\n                index.peak_intensity[i],\\n                index.peak_mask[i] if index.peak_mask is not None else None,\\n                cfg=cfg,\\n            )\\n            if (i + 1) % 50_000 == 0:\\n                print(f\\"[neighbor] binned {i+1}/{n}\\")\\n        binned = _l2_normalize(binned, dtype=np.float16)\\n        fps = np.asarray(index.fingerprints)\\n        ion = getattr(index, \\"ionization_mode\\", None)\\n        ion_code = np.zeros(n, dtype=np.int8)\\n        if ion is not None:\\n            for i, value in enumerate(ion):\\n                key = _ion_key(value)\\n                if key == \\"positive\\":\\n                    ion_code[i] = 1\\n                elif key == \\"negative\\":\\n                    ion_code[i] = 2\\n        print(f\\"[neighbor] index n={n} bins={binned.shape[1]} kNN cosine vs train spectra\\")\\n        return cls(binned=binned, fingerprints=fps, ion_code=ion_code, chunk=int(chunk))\\n\\n    def neighbor_fp(\\n        self,\\n        query_binned: np.ndarray,\\n        query_ion: Any = None,\\n        *,\\n        k: int = 20,\\n        temperature: float = 0.15,\\n    ) -> np.ndarray:\\n        q = _l2_normalize(np.asarray(query_binned, dtype=np.float32).ravel(), dtype=np.float32)\\n        n = int(self.binned.shape[0])\\n        bits = int(self.fingerprints.shape[1]) if self.fingerprints.ndim == 2 else 0\\n        if n == 0 or bits < 8:\\n            return np.zeros((max(bits, 1),), dtype=np.float32)[:bits] if bits else np.zeros((0,), dtype=np.float32)\\n        scores = np.empty(n, dtype=np.float32)\\n        q32 = q.astype(np.float32, copy=False)\\n        for start in range(0, n, int(self.chunk)):\\n            sl = slice(start, min(start + int(self.chunk), n))\\n            block = np.asarray(self.binned[sl], dtype=np.float32)\\n            scores[sl] = block @ q32\\n        q_ion = _ion_key(query_ion) if query_ion is not None else \\"\\"\\n        if self.ion_code is not None and q_ion in {\\"positive\\", \\"negative\\"}:\\n            ban = 2 if q_ion == \\"positive\\" else 1\\n            scores[self.ion_code == ban] = -1.0\\n        k = min(int(k), n)\\n        pick = np.argpartition(-scores, k - 1)[:k]\\n        pick = pick[np.argsort(-scores[pick])]\\n        w = _softmax(scores[pick], temperature=temperature)\\n        neigh = np.asarray(self.fingerprints[pick], dtype=np.float32)\\n        return (w[:, None] * neigh).sum(axis=0).astype(np.float32)\\n\\n    def blend_one(\\n        self,\\n        spec2fp: np.ndarray,\\n        feat: dict[str, Any],\\n        *,\\n        k: int = 20,\\n        alpha: float = 0.5,\\n    ) -> np.ndarray:\\n        nfp = self.neighbor_fp(\\n            feat.get(\\"binned\\"),\\n            feat.get(\\"ionization_mode\\"),\\n            k=k,\\n        )\\n        if nfp.size != np.asarray(spec2fp).size:\\n            return np.asarray(spec2fp, dtype=np.float32)\\n        return blend_fingerprints(spec2fp, nfp, alpha=alpha)\\n\\n\\ndef blend_predicted_fingerprints(\\n    spec2fp: np.ndarray,\\n    features: list[dict[str, Any]],\\n    neighbor_index: SpectralNeighborIndex | None,\\n    *,\\n    k: int = 20,\\n    alpha: float = 0.5,\\n) -> np.ndarray:\\n    fps = np.asarray(spec2fp, dtype=np.float32)\\n    if neighbor_index is None or fps.ndim != 2:\\n        return fps\\n    out = np.empty_like(fps)\\n    for i, feat in enumerate(features):\\n        out[i] = neighbor_index.blend_one(fps[i], feat, k=k, alpha=alpha)\\n        if (i + 1) % 50 == 0:\\n            print(f\\"[neighbor] blended {i+1}/{len(features)} k={k} a={alpha}\\")\\n    print(f\\"[neighbor] neighbor fp k={k} blend_a={alpha} n={len(features)}\\")\\n    return out\\n", "src/calibration.py": "\\"\\"\\"Binned P(true | modified cosine) and P(true | Tanimoto) for the V6 merge.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\n\\n# Conservative prior: cosine >= 0.8 maps to p ~= 0.92+ so V5 lock behavior is\\n# preserved if the empirical table is missing. Tanimoto is weaker than cosine.\\nDEFAULT_COSINE_X = (0.0, 0.30, 0.50, 0.65, 0.75, 0.80, 0.90, 1.00)\\nDEFAULT_COSINE_Y = (0.02, 0.06, 0.15, 0.35, 0.62, 0.92, 0.97, 0.995)\\nDEFAULT_TANI_X = (0.0, 0.15, 0.25, 0.35, 0.50, 0.70, 1.00)\\nDEFAULT_TANI_Y = (0.02, 0.06, 0.12, 0.22, 0.40, 0.62, 0.90)\\n\\n\\ndef _interp(x: np.ndarray, xp: np.ndarray, fp: np.ndarray) -> np.ndarray:\\n    x = np.asarray(x, dtype=np.float64)\\n    return np.clip(np.interp(x, xp, fp), 0.0, 1.0).astype(np.float32)\\n\\n\\n@dataclass\\nclass ScoreCalibration:\\n    cosine_x: np.ndarray\\n    cosine_y: np.ndarray\\n    tanimoto_x: np.ndarray\\n    tanimoto_y: np.ndarray\\n    adopt_soft_merge: bool = True\\n    keep_lock_above: float = 0.8\\n    neighbor_blend: float = 0.5\\n    class1_mrr_soft: float | None = None\\n    class1_mrr_lock: float | None = None\\n\\n    @classmethod\\n    def default(cls) -> \\"ScoreCalibration\\":\\n        return cls(\\n            cosine_x=np.asarray(DEFAULT_COSINE_X, dtype=np.float64),\\n            cosine_y=np.asarray(DEFAULT_COSINE_Y, dtype=np.float64),\\n            tanimoto_x=np.asarray(DEFAULT_TANI_X, dtype=np.float64),\\n            tanimoto_y=np.asarray(DEFAULT_TANI_Y, dtype=np.float64),\\n        )\\n\\n    def p_cosine(self, spec: np.ndarray | float) -> np.ndarray:\\n        return _interp(np.asarray(spec, dtype=np.float64), self.cosine_x, self.cosine_y)\\n\\n    def p_tanimoto(self, tani: np.ndarray | float) -> np.ndarray:\\n        return _interp(np.asarray(tani, dtype=np.float64), self.tanimoto_x, self.tanimoto_y)\\n\\n    def or_score(\\n        self,\\n        spec: np.ndarray | float,\\n        tani: np.ndarray | float,\\n        mass: np.ndarray | float | None = None,\\n    ) -> np.ndarray:\\n        \\"\\"\\"1 - (1-p_spec)*(1-p_fp), optionally multiplied by a mass Gaussian.\\"\\"\\"\\n        p_s = self.p_cosine(spec)\\n        p_f = self.p_tanimoto(tani)\\n        merged = 1.0 - (1.0 - p_s) * (1.0 - p_f)\\n        if mass is None:\\n            return merged.astype(np.float32)\\n        mass_arr = np.asarray(mass, dtype=np.float32)\\n        return (merged * mass_arr).astype(np.float32)\\n\\n    def to_dict(self) -> dict[str, Any]:\\n        return {\\n            \\"cosine\\": {\\"x\\": self.cosine_x.tolist(), \\"y\\": self.cosine_y.tolist()},\\n            \\"tanimoto\\": {\\"x\\": self.tanimoto_x.tolist(), \\"y\\": self.tanimoto_y.tolist()},\\n            \\"adopt_soft_merge\\": bool(self.adopt_soft_merge),\\n            \\"keep_lock_above\\": float(self.keep_lock_above),\\n            \\"neighbor_blend\\": float(self.neighbor_blend),\\n            \\"class1_mrr_soft\\": self.class1_mrr_soft,\\n            \\"class1_mrr_lock\\": self.class1_mrr_lock,\\n        }\\n\\n\\ndef _from_payload(payload: dict[str, Any]) -> ScoreCalibration:\\n    cos = payload.get(\\"cosine\\") or {}\\n    tani = payload.get(\\"tanimoto\\") or {}\\n    cal = ScoreCalibration(\\n        cosine_x=np.asarray(cos.get(\\"x\\") or DEFAULT_COSINE_X, dtype=np.float64),\\n        cosine_y=np.asarray(cos.get(\\"y\\") or DEFAULT_COSINE_Y, dtype=np.float64),\\n        tanimoto_x=np.asarray(tani.get(\\"x\\") or DEFAULT_TANI_X, dtype=np.float64),\\n        tanimoto_y=np.asarray(tani.get(\\"y\\") or DEFAULT_TANI_Y, dtype=np.float64),\\n        adopt_soft_merge=bool(payload.get(\\"adopt_soft_merge\\", True)),\\n        keep_lock_above=float(payload.get(\\"keep_lock_above\\", 0.8)),\\n        neighbor_blend=float(payload.get(\\"neighbor_blend\\", 0.5)),\\n        class1_mrr_soft=payload.get(\\"class1_mrr_soft\\"),\\n        class1_mrr_lock=payload.get(\\"class1_mrr_lock\\"),\\n    )\\n    if cal.class1_mrr_soft is not None and cal.class1_mrr_lock is not None:\\n        cal.adopt_soft_merge = float(cal.class1_mrr_soft) + 1e-9 >= float(cal.class1_mrr_lock)\\n    return cal\\n\\n\\ndef calibration_search_paths(extra: Path | None = None) -> list[Path]:\\n    here = Path(__file__).resolve().parent\\n    root = here.parent\\n    paths = [\\n        extra,\\n        root / \\"data\\" / \\"calibration.json\\",\\n        here / \\"calibration_data.json\\",\\n        Path(\\"/kaggle/working/src/calibration_data.json\\"),\\n        Path(\\"/kaggle/working/data/calibration.json\\"),\\n    ]\\n    return [p for p in paths if p is not None]\\n\\n\\ndef load_calibration(path: Path | str | None = None) -> ScoreCalibration:\\n    extra = Path(path) if path else None\\n    for cand in calibration_search_paths(extra):\\n        if cand.exists():\\n            try:\\n                payload = json.loads(cand.read_text())\\n                return _from_payload(payload)\\n            except Exception:\\n                continue\\n    return ScoreCalibration.default()\\n\\n\\ndef save_calibration(cal: ScoreCalibration, path: Path | str) -> None:\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    path.write_text(json.dumps(cal.to_dict(), indent=2))\\n", "src/calibration_data.json": "{\\n  \\"cosine\\": {\\n    \\"x\\": [\\n      0.0,\\n      0.15,\\n      0.3,\\n      0.4,\\n      0.5,\\n      0.575,\\n      0.65,\\n      0.7,\\n      0.75,\\n      0.775,\\n      0.8,\\n      0.8500000000000001,\\n      0.9,\\n      0.9550000000000001,\\n      1.0\\n    ],\\n    \\"y\\": [\\n      0.02,\\n      0.04,\\n      0.06,\\n      0.10500000000000001,\\n      0.15,\\n      0.24999999999999992,\\n      0.35,\\n      0.4849999999999998,\\n      0.62,\\n      0.77,\\n      0.92,\\n      0.9450000000000001,\\n      0.97,\\n      0.98375,\\n      0.995\\n    ]\\n  },\\n  \\"tanimoto\\": {\\n    \\"x\\": [\\n      0.0,\\n      0.075,\\n      0.15,\\n      0.2,\\n      0.25,\\n      0.3,\\n      0.35,\\n      0.425,\\n      0.5,\\n      0.6,\\n      0.7,\\n      0.8,\\n      0.855,\\n      1.0\\n    ],\\n    \\"y\\": [\\n      0.02,\\n      0.04,\\n      0.06,\\n      0.09000000000000001,\\n      0.12,\\n      0.16999999999999998,\\n      0.22,\\n      0.31,\\n      0.4,\\n      0.51,\\n      0.62,\\n      0.7133333333333334,\\n      0.7646666666666667,\\n      0.9\\n    ]\\n  },\\n  \\"adopt_soft_merge\\": true,\\n  \\"keep_lock_above\\": 0.8,\\n  \\"neighbor_blend\\": 0.5,\\n  \\"class1_mrr_soft\\": 0.9315033547947151,\\n  \\"class1_mrr_lock\\": 0.9310866881280484\\n}", "src/ranker.py": "\\"\\"\\"Ensemble ranking, InChIKey14 skeleton dedup, and submission formatting.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport csv\\nfrom pathlib import Path\\nfrom typing import Sequence\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom src.chem import canonical_smiles, has_rdkit, inchikey14_from_smiles, smiles_to_mol\\nfrom src.config import Config\\nfrom src.retrieval import fingerprint_cosine, tanimoto\\n\\nFALLBACK_SMILES = \\"CCO\\"\\n\\n\\ndef mass_gaussian_score(dm: np.ndarray, mass: np.ndarray, ppm_scale: float) -> np.ndarray:\\n    ppm = np.abs(dm) / np.clip(mass, 1e-6, None) * 1e6\\n    return np.exp(-0.5 * (ppm / max(ppm_scale, 1e-3)) ** 2).astype(np.float32)\\n\\n\\ndef combine_scores(\\n    spec_sim: np.ndarray,\\n    fp_tanimoto: np.ndarray,\\n    mass_score: np.ndarray,\\n    cfg: Config,\\n    fp_cosine: np.ndarray | None = None,\\n) -> np.ndarray:\\n    spec = np.nan_to_num(spec_sim, nan=0.0).astype(np.float32)\\n    fp = np.nan_to_num(fp_tanimoto, nan=0.0).astype(np.float32)\\n    if fp_cosine is not None:\\n        fp = 0.6 * fp + 0.4 * np.nan_to_num(fp_cosine, nan=0.0).astype(np.float32)\\n    mass = np.nan_to_num(mass_score, nan=0.0).astype(np.float32)\\n    w = float(cfg.w_spectral + cfg.w_fingerprint + cfg.w_mass)\\n    if w <= 0:\\n        w = 1.0\\n    return (cfg.w_spectral * spec + cfg.w_fingerprint * fp + cfg.w_mass * mass) / w\\n\\n\\ndef dedup_inchikey14(\\n    smiles: Sequence[str],\\n    scores: np.ndarray,\\n    inchikey14: Sequence[str] | None = None,\\n    top_k: int = 25,\\n) -> list[str]:\\n    \\"\\"\\"Keep the highest-scoring SMILES per RDKit InChIKey14, then take ``top_k``.\\n\\n    Submitting two stereoisomers of the same skeleton wastes an MRR@25 slot.\\n    \\"\\"\\"\\n    order = np.argsort(-np.asarray(scores, dtype=np.float64))\\n    seen: set[str] = set()\\n    picked: list[str] = []\\n    for idx in order:\\n        smi = str(smiles[int(idx)])\\n        if not smi or smi.lower() in {\\"nan\\", \\"none\\"}:\\n            continue\\n        if inchikey14 is not None:\\n            key = str(inchikey14[int(idx)] or \\"\\")\\n        else:\\n            key = \\"\\"\\n        if not key:\\n            key = inchikey14_from_smiles(smi)\\n        if not key:\\n            key = f\\"RAW::{smi}\\"\\n        if key in seen:\\n            continue\\n        seen.add(key)\\n        picked.append(smi)\\n        if len(picked) >= top_k:\\n            break\\n    return picked\\n\\n\\ndef merge_unique_smiles(\\n    primary: Sequence[str],\\n    extra: Sequence[str],\\n    *,\\n    top_k: int = 25,\\n) -> list[str]:\\n    \\"\\"\\"Append ``extra`` SMILES, dropping InChIKey14 collisions, up to ``top_k``.\\"\\"\\"\\n    out: list[str] = []\\n    seen: set[str] = set()\\n    for smi in list(primary) + list(extra):\\n        if not smi or str(smi).lower() in {\\"nan\\", \\"none\\"}:\\n            continue\\n        key = inchikey14_from_smiles(smi) or str(smi)\\n        if key in seen:\\n            continue\\n        seen.add(key)\\n        out.append(str(smi))\\n        if len(out) >= top_k:\\n            break\\n    return out\\n\\n\\ndef format_prediction_row(molecule_id: str, smiles_list: Sequence[str]) -> str:\\n    guesses = sanitize_guesses(smiles_list, top_k=25)\\n    if not guesses:\\n        guesses = [FALLBACK_SMILES]\\n    return \\";\\".join(guesses)\\n\\n\\ndef sanitize_guesses(smiles_list: Sequence[str] | None, top_k: int = 25) -> list[str]:\\n    \\"\\"\\"Keep at most ``top_k`` parseable unique-skeleton SMILES, no empties/nulls.\\"\\"\\"\\n    out: list[str] = []\\n    seen: set[str] = set()\\n    if not smiles_list:\\n        return out\\n    for raw in smiles_list:\\n        if raw is None:\\n            continue\\n        if isinstance(raw, float) and not np.isfinite(raw):\\n            continue\\n        text = str(raw).replace(\\"\\\\r\\", \\" \\").replace(\\"\\\\n\\", \\" \\").strip()\\n        if not text or text.lower() in {\\"nan\\", \\"none\\", \\"null\\"}:\\n            continue\\n        for part in text.split(\\";\\"):\\n            part = part.strip().strip(\'\\"\').strip(\\"\'\\")\\n            if not part or part.lower() in {\\"nan\\", \\"none\\", \\"null\\"}:\\n                continue\\n            mol = smiles_to_mol(part)\\n            if mol is not None:\\n                canon = canonical_smiles(part) or part\\n            elif has_rdkit():\\n                continue\\n            else:\\n                canon = part\\n            if not canon or \\";\\" in canon:\\n                continue\\n            key = inchikey14_from_smiles(canon) or canon\\n            if key in seen:\\n                continue\\n            seen.add(key)\\n            out.append(canon)\\n            if len(out) >= int(top_k):\\n                return out\\n    return out\\n\\n\\ndef official_molecule_ids(cfg: Config, test_df: pd.DataFrame | None = None) -> list[str]:\\n    sample_path = cfg.sample_submission_path\\n    if sample_path.exists():\\n        ids = pd.read_csv(sample_path, dtype=str)[\\"molecule_id\\"].astype(str).tolist()\\n        if ids:\\n            return ids\\n    if test_df is not None and \\"molecule_id\\" in test_df.columns:\\n        return list(dict.fromkeys(test_df[\\"molecule_id\\"].astype(str).tolist()))\\n    raise FileNotFoundError(\\"Need sample_submission.csv or a test frame with molecule_id\\")\\n\\n\\ndef predictions_to_submission(\\n    molecule_ids: Sequence[str],\\n    ranked_smiles: Sequence[Sequence[str]],\\n) -> pd.DataFrame:\\n    rows = []\\n    for mid, smiles in zip(molecule_ids, ranked_smiles):\\n        rows.append({\\"molecule_id\\": str(mid), \\"smiles\\": format_prediction_row(str(mid), smiles)})\\n    return pd.DataFrame(rows, columns=[\\"molecule_id\\", \\"smiles\\"])\\n\\n\\ndef build_submission_frame(\\n    molecule_ids: Sequence[str],\\n    ranked_map: dict[str, Sequence[str]],\\n    *,\\n    top_k: int = 25,\\n    fallback: str = FALLBACK_SMILES,\\n) -> pd.DataFrame:\\n    rows = []\\n    for mid in molecule_ids:\\n        guesses = sanitize_guesses(ranked_map.get(str(mid), []), top_k=top_k)\\n        if not guesses:\\n            guesses = sanitize_guesses([fallback], top_k=top_k) or [fallback]\\n        rows.append({\\"molecule_id\\": str(mid), \\"smiles\\": \\";\\".join(guesses[:top_k])})\\n    return pd.DataFrame(rows, columns=[\\"molecule_id\\", \\"smiles\\"])\\n\\n\\ndef write_submission_csv(df: pd.DataFrame, path: Path | str) -> Path:\\n    \\"\\"\\"Write a competition CSV with exactly two columns and no empty fields.\\"\\"\\"\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    out = df[[\\"molecule_id\\", \\"smiles\\"]].copy()\\n    out[\\"molecule_id\\"] = out[\\"molecule_id\\"].astype(str)\\n    out[\\"smiles\\"] = out[\\"smiles\\"].astype(str)\\n    with path.open(\\"w\\", encoding=\\"utf-8\\", newline=\\"\\") as f:\\n        writer = csv.writer(f, lineterminator=\\"\\\\n\\")\\n        writer.writerow([\\"molecule_id\\", \\"smiles\\"])\\n        for mid, smi in out.itertuples(index=False, name=None):\\n            if not str(mid).strip() or not str(smi).strip():\\n                raise ValueError(f\\"empty submission field for {mid!r}\\")\\n            writer.writerow([str(mid), str(smi)])\\n    validate_submission(path)\\n    return path\\n\\n\\ndef validate_submission(path: Path | str, sample_path: Path | str | None = None) -> None:\\n    path = Path(path)\\n    with path.open(\\"r\\", encoding=\\"utf-8\\", newline=\\"\\") as f:\\n        rows = list(csv.reader(f))\\n    if not rows:\\n        raise ValueError(\\"submission.csv is empty\\")\\n    header = [h.strip() for h in rows[0]]\\n    if header != [\\"molecule_id\\", \\"smiles\\"]:\\n        raise ValueError(f\\"bad header {header}\\")\\n    body = rows[1:]\\n    if not body:\\n        raise ValueError(\\"submission.csv has no data rows\\")\\n    ids: list[str] = []\\n    for i, row in enumerate(body, start=2):\\n        if len(row) != 2:\\n            raise ValueError(f\\"line {i} has {len(row)} columns\\")\\n        mid, smi = row[0].strip(), row[1].strip()\\n        if not mid or not smi:\\n            raise ValueError(f\\"line {i} has an empty molecule_id or smiles\\")\\n        parts = [p for p in smi.split(\\";\\") if p]\\n        if not parts:\\n            raise ValueError(f\\"line {i} has no SMILES guesses\\")\\n        if len(parts) > 25:\\n            raise ValueError(f\\"line {i} has {len(parts)} guesses\\")\\n        ids.append(mid)\\n    if len(ids) != len(set(ids)):\\n        raise ValueError(\\"duplicate molecule_id in submission.csv\\")\\n    if sample_path is not None and Path(sample_path).exists():\\n        expected = pd.read_csv(sample_path, dtype=str)[\\"molecule_id\\"].astype(str).tolist()\\n        if ids != expected:\\n            raise ValueError(\\n                f\\"molecule_id mismatch vs sample_submission \\"\\n                f\\"(got {len(ids)} expected {len(expected)})\\"\\n            )\\n\\n\\ndef soft_tanimoto(pred_prob: np.ndarray, cand_fp: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Probability-weighted Tanimoto between a predicted fp and candidate bits.\\"\\"\\"\\n    q = np.asarray(pred_prob, dtype=np.float32).ravel()\\n    c = np.asarray(cand_fp, dtype=np.float32)\\n    if c.ndim == 1:\\n        c = c.reshape(1, -1)\\n    inter = c @ q\\n    union = c.sum(axis=1) + float(q.sum()) - inter\\n    return (inter / np.clip(union, 1e-6, None)).astype(np.float32)\\n\\n\\ndef fingerprint_scores(\\n    pred_prob: np.ndarray,\\n    cand_fp: np.ndarray,\\n    threshold: float,\\n    *,\\n    soft_mix: float = 0.5,\\n) -> tuple[np.ndarray, np.ndarray]:\\n    bits = (pred_prob >= threshold).astype(np.float32)\\n    if bits.sum() == 0:\\n        # Avoid empty fingerprints: take top 64 predicted bits.\\n        k = min(64, pred_prob.size)\\n        top = np.argpartition(pred_prob, -k)[-k:]\\n        bits = np.zeros_like(pred_prob, dtype=np.float32)\\n        bits[top] = 1.0\\n    tani = tanimoto(bits, cand_fp)\\n    mix = float(np.clip(soft_mix, 0.0, 1.0))\\n    if mix > 0:\\n        tani_s = soft_tanimoto(pred_prob, cand_fp)\\n        tani = (1.0 - mix) * tani + mix * tani_s\\n    cos = fingerprint_cosine(pred_prob, cand_fp)\\n    return tani, cos\\n", "src/rerank.py": "\\"\\"\\"Learned candidate re-ranker (MS2Query-style).\\n\\nTrains a small gradient-boosted tree on pairwise features:\\nmodified cosine, Tanimoto, fingerprint cosine, mass error, matched peaks.\\nFalls back to the linear combine_scores weights if boosting is unavailable.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\nimport numpy as np\\n\\nfrom src.config import Config\\nfrom src.ranker import combine_scores\\n\\n\\nFEATURE_NAMES = (\\n    \\"modified_cosine\\",\\n    \\"tanimoto\\",\\n    \\"fp_cosine\\",\\n    \\"mass_gaussian\\",\\n    \\"abs_ppm\\",\\n    \\"n_match_frac\\",\\n    \\"is_class2\\",\\n)\\n\\n\\ndef candidate_feature_matrix(\\n    spec: np.ndarray,\\n    tani: np.ndarray,\\n    fcos: np.ndarray,\\n    mass_sc: np.ndarray,\\n    dm: np.ndarray,\\n    mass: np.ndarray,\\n    n_match: np.ndarray,\\n    is_class2: bool,\\n) -> np.ndarray:\\n    spec = np.asarray(spec, dtype=np.float32).ravel()\\n    tani = np.asarray(tani, dtype=np.float32).ravel()\\n    fcos = np.asarray(fcos, dtype=np.float32).ravel()\\n    mass_sc = np.asarray(mass_sc, dtype=np.float32).ravel()\\n    dm = np.asarray(dm, dtype=np.float64).ravel()\\n    mass = np.asarray(mass, dtype=np.float64).ravel()\\n    n_match = np.asarray(n_match, dtype=np.float32).ravel()\\n    ppm = np.abs(dm) / np.clip(mass, 1e-6, None) * 1e6\\n    flag = np.full(spec.shape, 1.0 if is_class2 else 0.0, dtype=np.float32)\\n    return np.stack(\\n        [\\n            spec,\\n            tani,\\n            fcos,\\n            mass_sc,\\n            np.clip(ppm, 0.0, 200.0).astype(np.float32),\\n            np.clip(n_match / 64.0, 0.0, 2.0),\\n            flag,\\n        ],\\n        axis=1,\\n    ).astype(np.float32)\\n\\n\\ndef _new_booster():\\n    try:\\n        import xgboost as xgb\\n\\n        return xgb.XGBClassifier(\\n            n_estimators=120,\\n            max_depth=4,\\n            learning_rate=0.08,\\n            subsample=0.85,\\n            colsample_bytree=0.85,\\n            n_jobs=4,\\n            eval_metric=\\"logloss\\",\\n            verbosity=0,\\n        )\\n    except Exception:\\n        pass\\n    try:\\n        from sklearn.ensemble import HistGradientBoostingClassifier\\n\\n        return HistGradientBoostingClassifier(\\n            max_depth=4,\\n            max_iter=80,\\n            learning_rate=0.08,\\n            l2_regularization=0.1,\\n        )\\n    except Exception:\\n        return None\\n\\n\\ndef train_reranker(\\n    X: np.ndarray,\\n    y: np.ndarray,\\n    *,\\n    sample_weight: np.ndarray | None = None,\\n) -> Any | None:\\n    \\"\\"\\"Fit a classifier that predicts P(same InChIKey14). ``None`` if boosting is missing.\\"\\"\\"\\n    X = np.asarray(X, dtype=np.float32)\\n    y = np.asarray(y, dtype=np.int32)\\n    if X.size == 0 or y.size < 20 or int(y.sum()) < 5:\\n        print(\\"[rerank] not enough positive pairs, keeping linear weights\\")\\n        return None\\n    model = _new_booster()\\n    if model is None:\\n        print(\\"[rerank] xgboost/sklearn missing, keeping linear weights\\")\\n        return None\\n    try:\\n        if sample_weight is not None:\\n            model.fit(X, y, sample_weight=np.asarray(sample_weight, dtype=np.float32))\\n        else:\\n            # Balance positives (true skeleton) vs mass-window decoys.\\n            n_pos = max(int(y.sum()), 1)\\n            n_neg = max(int((1 - y).sum()), 1)\\n            w = np.where(y > 0, n_neg / n_pos, 1.0).astype(np.float32)\\n            model.fit(X, y, sample_weight=w)\\n        print(f\\"[rerank] fitted {type(model).__name__} n={len(y)} pos={int(y.sum())}\\")\\n        return model\\n    except TypeError:\\n        model.fit(X, y)\\n        print(f\\"[rerank] fitted {type(model).__name__} n={len(y)} pos={int(y.sum())}\\")\\n        return model\\n    except Exception as exc:\\n        print(f\\"[rerank] fit failed ({exc}); keeping linear weights\\")\\n        return None\\n\\n\\ndef predict_rerank_scores(model: Any | None, X: np.ndarray, cfg: Config, **linear_parts) -> np.ndarray:\\n    \\"\\"\\"Probability of a true match, or linear combine_scores if ``model`` is None.\\"\\"\\"\\n    if model is None:\\n        return combine_scores(\\n            linear_parts[\\"spec\\"],\\n            linear_parts[\\"tani\\"],\\n            linear_parts[\\"mass_sc\\"],\\n            cfg,\\n            fp_cosine=linear_parts.get(\\"fcos\\"),\\n        )\\n    X = np.asarray(X, dtype=np.float32)\\n    if hasattr(model, \\"predict_proba\\"):\\n        proba = model.predict_proba(X)\\n        if proba.ndim == 2 and proba.shape[1] >= 2:\\n            return proba[:, 1].astype(np.float32)\\n        return proba.ravel().astype(np.float32)\\n    if hasattr(model, \\"decision_function\\"):\\n        z = np.asarray(model.decision_function(X), dtype=np.float64).ravel()\\n        return (1.0 / (1.0 + np.exp(-z))).astype(np.float32)\\n    pred = np.asarray(model.predict(X), dtype=np.float32).ravel()\\n    return pred\\n\\n\\ndef train_reranker_from_index(index, cfg: Config, *, n_queries: int = 2500, max_cand: int = 48):\\n    \\"\\"\\"Self-retrieval pairs from the spectral library (true skeleton vs mass isobars).\\"\\"\\"\\n    from src.ranker import fingerprint_scores, mass_gaussian_score\\n    from src.retrieval import modified_cosine\\n\\n    n = int(index.smiles.shape[0])\\n    if n < 30 or index.peak_mz is None:\\n        return None\\n    rng = np.random.default_rng(int(cfg.seed))\\n    queries = rng.choice(n, size=min(int(n_queries), n), replace=False)\\n    Xs: list[np.ndarray] = []\\n    ys: list[np.ndarray] = []\\n    for i in queries:\\n        mass = float(index.exact_mass[i])\\n        cand = index.query_masses([mass], cfg, use_fallback=False)\\n        if cand.size < 2:\\n            continue\\n        if cand.size > max_cand:\\n            true = np.where(index.inchikey14[cand] == index.inchikey14[i])[0]\\n            rest = np.setdiff1d(np.arange(cand.size), true)\\n            take = rng.choice(rest, size=min(int(max_cand) - 1, rest.size), replace=False)\\n            keep = np.concatenate([true[:1], take]) if true.size else take\\n            cand = cand[keep]\\n        spec, n_match = modified_cosine(\\n            index.peak_mz[i],\\n            index.peak_intensity[i],\\n            index.peak_mask[i],\\n            mass,\\n            index.peak_mz[cand],\\n            index.peak_intensity[cand],\\n            index.peak_mask[cand],\\n            index.exact_mass[cand].astype(np.float32),\\n            cfg.modified_cosine_mz_tol,\\n            return_n_match=True,\\n        )\\n        qfp = index.fingerprints[i].astype(np.float32)\\n        tani, fcos = fingerprint_scores(qfp, index.fingerprints[cand], cfg.fp_threshold)\\n        dm = np.abs(index.exact_mass[cand] - mass)\\n        mass_sc = mass_gaussian_score(dm, index.exact_mass[cand], cfg.mass_score_ppm_scale)\\n        X = candidate_feature_matrix(\\n            spec, tani, fcos, mass_sc, dm, index.exact_mass[cand], n_match, is_class2=False\\n        )\\n        y = (index.inchikey14[cand] == index.inchikey14[i]).astype(np.int32)\\n        Xs.append(X)\\n        ys.append(y)\\n    if not Xs:\\n        return None\\n    return train_reranker(np.concatenate(Xs, axis=0), np.concatenate(ys, axis=0))\\n", "src/formula_head.py": "\\"\\"\\"Lightweight formula-count head (MIST-CF lite).\\n\\nPredicts C/H/N/O/P/S/Cl/Br counts from a fingerprint + exact mass so Class 2\\nisobaric formulas can be down-weighted without enumerating SENIOR formulas.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn as nn\\n\\nfrom src.chem import FORMULA_ELEMENTS, formula_count_vector\\n\\n\\nclass FormulaNet(nn.Module):\\n    def __init__(self, fp_bits: int = 2048, n_out: int = 8) -> None:\\n        super().__init__()\\n        self.net = nn.Sequential(\\n            nn.Linear(fp_bits + 1, 128),\\n            nn.GELU(),\\n            nn.Dropout(0.05),\\n            nn.Linear(128, n_out),\\n        )\\n\\n    def forward(self, fp: torch.Tensor, mass: torch.Tensor) -> torch.Tensor:\\n        if mass.ndim == 1:\\n            mass = mass.unsqueeze(-1)\\n        return self.net(torch.cat([fp, mass / 1000.0], dim=-1))\\n\\n\\ndef train_formula_head(\\n    fps: np.ndarray,\\n    masses: np.ndarray,\\n    formulas: list[str] | np.ndarray,\\n    *,\\n    epochs: int = 4,\\n    batch_size: int = 256,\\n    lr: float = 1e-3,\\n    max_n: int = 40_000,\\n    seed: int = 42,\\n) -> FormulaNet | None:\\n    y = np.stack([formula_count_vector(f) for f in formulas]).astype(np.float32)\\n    keep = y.sum(axis=1) > 0\\n    fps = np.asarray(fps, dtype=np.float32)[keep]\\n    masses = np.asarray(masses, dtype=np.float32)[keep]\\n    y = y[keep]\\n    n = int(fps.shape[0])\\n    if n < 50:\\n        print(\\"[formula] not enough labeled formulas\\")\\n        return None\\n    rng = np.random.default_rng(seed)\\n    if n > max_n:\\n        pick = rng.choice(n, size=max_n, replace=False)\\n        fps, masses, y = fps[pick], masses[pick], y[pick]\\n        n = max_n\\n    # Spec2FP fingerprints are noisy; jitter bits so the head is not overconfident.\\n    fps = np.clip(fps + rng.normal(0.0, 0.12, fps.shape).astype(np.float32), 0.0, 1.0)\\n    device = torch.device(\\"cpu\\")\\n    model = FormulaNet(fp_bits=int(fps.shape[1]), n_out=y.shape[1]).to(device)\\n    opt = torch.optim.Adam(model.parameters(), lr=lr)\\n    x_fp = torch.from_numpy(fps)\\n    x_m = torch.from_numpy(masses)\\n    y_t = torch.from_numpy(y)\\n    model.train()\\n    for epoch in range(int(epochs)):\\n        perm = rng.permutation(n)\\n        losses = []\\n        for start in range(0, n, int(batch_size)):\\n            sl = perm[start : start + int(batch_size)]\\n            pred = model(x_fp[sl], x_m[sl])\\n            loss = torch.mean((pred - y_t[sl]).abs())\\n            opt.zero_grad()\\n            loss.backward()\\n            opt.step()\\n            losses.append(float(loss.item()))\\n        print(f\\"[formula] epoch={epoch+1}/{epochs} l1={np.mean(losses):.3f} n={n} elems={len(FORMULA_ELEMENTS)}\\")\\n    model.eval()\\n    return model\\n\\n\\n@torch.no_grad()\\ndef predict_formula_counts(model: FormulaNet | None, fp: np.ndarray, mass: float) -> np.ndarray | None:\\n    if model is None:\\n        return None\\n    fp_t = torch.from_numpy(np.asarray(fp, dtype=np.float32).reshape(1, -1))\\n    mass_t = torch.tensor([float(mass)], dtype=torch.float32)\\n    out = model(fp_t, mass_t).cpu().numpy().ravel()\\n    return np.clip(out, 0.0, None).astype(np.float32)\\n", "src/metrics.py": "\\"\\"\\"Competition metric: Mean Reciprocal Rank @ K on InChIKey14.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Sequence\\n\\nimport numpy as np\\n\\nfrom src.chem import inchikey14_from_smiles\\n\\n\\ndef reciprocal_rank(ranked_keys: Sequence[str], true_key: str, k: int = 25) -> float:\\n    if not true_key:\\n        return 0.0\\n    for i, key in enumerate(ranked_keys[:k]):\\n        if key == true_key:\\n            return 1.0 / float(i + 1)\\n    return 0.0\\n\\n\\ndef smiles_to_keys(smiles_list: Sequence[str]) -> list[str]:\\n    return [inchikey14_from_smiles(s) for s in smiles_list]\\n\\n\\ndef mrr_at_k(\\n    predictions: Sequence[Sequence[str]],\\n    true_smiles: Sequence[str] | None = None,\\n    true_keys: Sequence[str] | None = None,\\n    k: int = 25,\\n) -> float:\\n    \\"\\"\\"``predictions[i]`` is a ranked SMILES list for query i.\\"\\"\\"\\n    if true_keys is None:\\n        if true_smiles is None:\\n            raise ValueError(\\"Provide true_smiles or true_keys\\")\\n        true_keys = [inchikey14_from_smiles(s) for s in true_smiles]\\n    scores = []\\n    for pred, gold in zip(predictions, true_keys):\\n        keys = smiles_to_keys(pred)\\n        scores.append(reciprocal_rank(keys, gold, k=k))\\n    if not scores:\\n        return 0.0\\n    return float(np.mean(scores))\\n", "src/tpu_trainer.py": "\\"\\"\\"PyTorch / PyTorch-XLA training loop for Spec2FP.\\n\\nOn TPU the loop uses ``xm.optimizer_step`` and ``pl.ParallelLoader``. Input\\nbatches are padded to a fixed ``batch_size`` so XLA never recompiles on the\\npartial last step.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nimport random\\nimport time\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\nimport torch\\nfrom torch.utils.data import DataLoader, Dataset, RandomSampler, SequentialSampler, Subset\\n\\nfrom src.config import Config, get_config\\nfrom src.data import SpectrumFingerprintDataset, assert_static_shapes, collate_fixed\\nfrom src.model import FocalBCELoss, Spec2FP, model_from_config\\n\\n\\ndef try_import_xla():\\n    try:\\n        import torch_xla.core.xla_model as xm\\n        import torch_xla.distributed.parallel_loader as pl\\n\\n        return xm, pl\\n    except Exception:\\n        return None, None\\n\\n\\ndef set_seed(seed: int) -> None:\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    torch.manual_seed(seed)\\n\\n\\ndef get_device(prefer_xla: bool = True):\\n    xm, _ = try_import_xla()\\n    if prefer_xla and xm is not None:\\n        return xm.xla_device(), True\\n    if torch.cuda.is_available():\\n        return torch.device(\\"cuda\\"), False\\n    if hasattr(torch.backends, \\"mps\\") and torch.backends.mps.is_available():\\n        return torch.device(\\"mps\\"), False\\n    return torch.device(\\"cpu\\"), False\\n\\n\\ndef _pad_to_batch_size(batch: dict[str, torch.Tensor], target: int) -> dict[str, torch.Tensor]:\\n    n = int(batch[\\"peak_mz\\"].shape[0])\\n    if n == target:\\n        return batch\\n    if n > target:\\n        return {k: v[:target] for k, v in batch.items()}\\n    pad = target - n\\n    out = {}\\n    for k, v in batch.items():\\n        if v.dim() == 0:\\n            out[k] = v\\n            continue\\n        pad_shape = (pad,) + tuple(v.shape[1:])\\n        z = torch.zeros(pad_shape, dtype=v.dtype, device=v.device)\\n        out[k] = torch.cat([v, z], dim=0)\\n    return out\\n\\n\\ndef _move(batch: dict[str, torch.Tensor], device) -> dict[str, torch.Tensor]:\\n    return {k: v.to(device) for k, v in batch.items()}\\n\\n\\ndef _forward(model: Spec2FP, batch: dict[str, torch.Tensor]) -> torch.Tensor:\\n    return model(\\n        peak_mz=batch[\\"peak_mz\\"],\\n        peak_intensity=batch[\\"peak_intensity\\"],\\n        peak_nl=batch[\\"peak_nl\\"],\\n        peak_mask=batch[\\"peak_mask\\"],\\n        binned=batch[\\"binned\\"],\\n        precursor_feat=batch[\\"precursor_feat\\"],\\n        adduct_id=batch[\\"adduct_id\\"],\\n    )\\n\\n\\ndef _bit_metrics(logits: torch.Tensor, target: torch.Tensor) -> dict[str, float]:\\n    with torch.no_grad():\\n        pred = (torch.sigmoid(logits) >= 0.5).float()\\n        tgt = target.float()\\n        tp = (pred * tgt).sum()\\n        fp = (pred * (1 - tgt)).sum()\\n        fn = ((1 - pred) * tgt).sum()\\n        prec = tp / (tp + fp + 1e-6)\\n        rec = tp / (tp + fn + 1e-6)\\n        f1 = 2 * prec * rec / (prec + rec + 1e-6)\\n        acc = (pred == tgt).float().mean()\\n    return {\\n        \\"bit_acc\\": float(acc.item()),\\n        \\"bit_f1\\": float(f1.item()),\\n    }\\n\\n\\ndef save_checkpoint(model: Spec2FP, path: Path, cfg: Config, extra: dict[str, Any] | None = None) -> None:\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    payload = {\\n        \\"model\\": {k: v.detach().cpu() for k, v in model.state_dict().items()},\\n        \\"config\\": cfg.__dict__.copy(),\\n        \\"extra\\": extra or {},\\n    }\\n    torch.save(payload, path)\\n\\n\\ndef load_model(path: Path | str, cfg: Config | None = None, map_location: str | torch.device = \\"cpu\\") -> Spec2FP:\\n    try:\\n        blob = torch.load(str(path), map_location=map_location, weights_only=False)\\n    except TypeError:\\n        blob = torch.load(str(path), map_location=map_location)\\n    cfg = cfg or get_config()\\n    model = model_from_config(cfg)\\n    model.load_state_dict(blob[\\"model\\"] if \\"model\\" in blob else blob)\\n    model.eval()\\n    return model\\n\\n\\ndef split_dataset(ds: Dataset, val_fraction: float, seed: int) -> tuple[Dataset, Dataset]:\\n    n = len(ds)\\n    n_val = max(1, int(round(n * val_fraction))) if n > 1 else 0\\n    g = torch.Generator().manual_seed(seed)\\n    perm = torch.randperm(n, generator=g).tolist()\\n    val_idx = perm[:n_val]\\n    train_idx = perm[n_val:] or perm\\n    return Subset(ds, train_idx), Subset(ds, val_idx) if n_val else Subset(ds, [])\\n\\n\\n@torch.no_grad()\\ndef evaluate(\\n    model: Spec2FP,\\n    loader: DataLoader,\\n    device,\\n    cfg: Config,\\n    criterion: torch.nn.Module,\\n    use_xla: bool,\\n) -> dict[str, float]:\\n    model.eval()\\n    xm, pl = try_import_xla()\\n    if use_xla and pl is not None:\\n        iterator = pl.ParallelLoader(loader, [device]).per_device_loader(device)\\n    else:\\n        iterator = loader\\n    losses = []\\n    metrics = []\\n    for batch in iterator:\\n        if not use_xla:\\n            batch = _move(batch, device)\\n        n = int(batch[\\"peak_mz\\"].shape[0])\\n        batch = _pad_to_batch_size(batch, cfg.batch_size)\\n        logits = _forward(model, batch)[:n]\\n        target = batch[\\"fingerprint\\"][:n]\\n        loss = criterion(logits, target)\\n        losses.append(float(loss.detach().cpu().item()))\\n        metrics.append(_bit_metrics(logits, target))\\n        if use_xla and xm is not None:\\n            xm.mark_step()\\n    out = {\\"val_loss\\": float(np.mean(losses) if losses else 0.0)}\\n    if metrics:\\n        out[\\"val_bit_acc\\"] = float(np.mean([m[\\"bit_acc\\"] for m in metrics]))\\n        out[\\"val_bit_f1\\"] = float(np.mean([m[\\"bit_f1\\"] for m in metrics]))\\n    return out\\n\\n\\ndef train_spec2fp(\\n    dataset: SpectrumFingerprintDataset,\\n    cfg: Config | None = None,\\n    *,\\n    device=None,\\n    time_limit_s: float | None = None,\\n    seed: int | None = None,\\n    ckpt_name: str | None = None,\\n) -> Spec2FP:\\n    cfg = cfg or get_config()\\n    if seed is not None:\\n        cfg.seed = int(seed)\\n    if ckpt_name is not None:\\n        cfg.ckpt_name = str(ckpt_name)\\n    set_seed(cfg.seed)\\n    cfg.artifact_dir.mkdir(parents=True, exist_ok=True)\\n    xm, pl = try_import_xla()\\n    if device is None:\\n        device, use_xla = get_device(prefer_xla=True)\\n    else:\\n        use_xla = xm is not None and \\"xla\\" in str(device).lower()\\n\\n    train_ds, val_ds = split_dataset(dataset, cfg.val_fraction, cfg.seed)\\n    train_loader = DataLoader(\\n        train_ds,\\n        batch_size=cfg.batch_size,\\n        sampler=RandomSampler(train_ds),\\n        num_workers=cfg.num_workers,\\n        collate_fn=collate_fixed,\\n        drop_last=False,\\n        pin_memory=torch.cuda.is_available() and not use_xla,\\n    )\\n    val_loader = (\\n        DataLoader(\\n            val_ds,\\n            batch_size=cfg.batch_size,\\n            sampler=SequentialSampler(val_ds),\\n            num_workers=0,\\n            collate_fn=collate_fixed,\\n            drop_last=False,\\n        )\\n        if len(val_ds) > 0\\n        else None\\n    )\\n\\n    model = model_from_config(cfg).to(device)\\n    criterion = FocalBCELoss(gamma=cfg.focal_gamma, pos_weight=cfg.bce_pos_weight).to(device)\\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)\\n    n_steps = max(1, math.ceil(len(train_ds) / cfg.batch_size) * cfg.num_epochs)\\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_steps)\\n\\n    # Probe static shapes on a dummy batch so XLA failures happen immediately.\\n    probe = collate_fixed([dataset[0] for _ in range(min(2, len(dataset)))])\\n    assert_static_shapes(probe, cfg)\\n\\n    t0 = time.time()\\n    limit = cfg.train_time_limit_s if time_limit_s is None else time_limit_s\\n    global_step = 0\\n    best_val = float(\\"inf\\")\\n    print(f\\"[train] device={device} xla={use_xla} n_train={len(train_ds)} n_val={len(val_ds)}\\")\\n\\n    for epoch in range(cfg.num_epochs):\\n        model.train()\\n        if use_xla and pl is not None:\\n            iterator = pl.ParallelLoader(train_loader, [device]).per_device_loader(device)\\n        else:\\n            iterator = train_loader\\n        running = 0.0\\n        n_seen = 0\\n        for batch in iterator:\\n            if limit and (time.time() - t0) > limit:\\n                print(\\"[train] time limit reached, stopping\\")\\n                save_checkpoint(model, cfg.checkpoint_path, cfg, extra={\\"epoch\\": epoch, \\"step\\": global_step})\\n                return model\\n            if not use_xla:\\n                batch = _move(batch, device)\\n            n = int(batch[\\"peak_mz\\"].shape[0])\\n            batch = _pad_to_batch_size(batch, cfg.batch_size)\\n            optimizer.zero_grad(set_to_none=True)\\n            logits = _forward(model, batch)\\n            loss = criterion(logits[:n], batch[\\"fingerprint\\"][:n])\\n            loss.backward()\\n            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)\\n            if use_xla and xm is not None:\\n                xm.optimizer_step(optimizer)\\n                xm.mark_step()\\n            else:\\n                optimizer.step()\\n            scheduler.step()\\n            running += float(loss.detach().cpu().item()) * n\\n            n_seen += n\\n            global_step += 1\\n            if global_step % cfg.log_every == 0:\\n                print(\\n                    f\\"[train] epoch={epoch+1} step={global_step} loss={running / max(n_seen, 1):.4f} \\"\\n                    f\\"lr={scheduler.get_last_lr()[0]:.2e}\\"\\n                )\\n        msg = f\\"[train] epoch {epoch+1}/{cfg.num_epochs} train_loss={running / max(n_seen, 1):.4f}\\"\\n        if val_loader is not None and len(val_ds) > 0:\\n            stats = evaluate(model, val_loader, device, cfg, criterion, use_xla)\\n            msg += \\" \\" + \\" \\".join(f\\"{k}={v:.4f}\\" for k, v in stats.items())\\n            if stats[\\"val_loss\\"] < best_val:\\n                best_val = stats[\\"val_loss\\"]\\n                save_checkpoint(\\n                    model,\\n                    cfg.checkpoint_path,\\n                    cfg,\\n                    extra={\\"epoch\\": epoch, \\"val_loss\\": best_val},\\n                )\\n        else:\\n            save_checkpoint(model, cfg.checkpoint_path, cfg, extra={\\"epoch\\": epoch})\\n        print(msg)\\n\\n    save_checkpoint(model, cfg.checkpoint_path, cfg, extra={\\"epoch\\": cfg.num_epochs, \\"step\\": global_step})\\n    return model\\n", "src/infer.py": "\\"\\"\\"End-to-end retrieval + fingerprint inference + ranked SMILES export.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections import defaultdict\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\n\\nfrom src.chem import (\\n    exact_mass_from_smiles,\\n    formula_to_mass,\\n    inchikey14_from_smiles,\\n    morgan_fingerprint,\\n    plausible_neutral_masses,\\n    unpack_fingerprints,\\n)\\nfrom src.config import Config, get_config\\nfrom src.data import iter_train_row_groups, load_test\\nfrom src.model import Spec2FP\\nfrom src.preprocessing import featurize_aggregated, featurize_spectrum, stack_features\\nfrom src.ranker import (\\n    FALLBACK_SMILES,\\n    combine_scores,\\n    dedup_inchikey14,\\n    fingerprint_scores,\\n    mass_gaussian_score,\\n    merge_unique_smiles,\\n    official_molecule_ids,\\n    build_submission_frame,\\n    sanitize_guesses,\\n    write_submission_csv,\\n)\\nfrom src.retrieval import StructureIndex, fingerprint_knn, load_external_structure_index, modified_cosine\\nfrom src.tpu_trainer import _forward, _move, _pad_to_batch_size, get_device, load_model\\n\\n\\ndef build_library_from_train(\\n    cfg: Config,\\n    *,\\n    max_row_groups: int | None = None,\\n    progress: bool = True,\\n) -> StructureIndex:\\n    \\"\\"\\"Stream train.parquet and keep one representative spectrum per InChIKey14.\\n\\n    Two passes: (1) record the max peak count per skeleton, (2) featurize only\\n    those winning rows so we do not pay RDKit/numpy work 2.5M times.\\n    \\"\\"\\"\\n    cols = [\\n        \\"normalized_smiles\\",\\n        \\"inchikey14\\",\\n        \\"molecular_formula\\",\\n        \\"adduct\\",\\n        \\"precursor_mz\\",\\n        \\"ms2_mzs\\",\\n        \\"ms2_normalized_intensities\\",\\n        \\"num_peaks\\",\\n        \\"collision_energy_ev\\",\\n        \\"ionization_mode\\",\\n    ]\\n    smiles_map: dict[str, str] = {}\\n    formula_map: dict[str, str] = {}\\n    best_n: dict[str, int] = {}\\n\\n    n_groups_done = 0\\n    for df in iter_train_row_groups(cfg.train_path, columns=[\\"normalized_smiles\\", \\"inchikey14\\", \\"molecular_formula\\", \\"num_peaks\\"]):\\n        n_groups_done += 1\\n        ikeys = df[\\"inchikey14\\"].astype(str).to_numpy()\\n        smiles = df[\\"normalized_smiles\\"].astype(str).to_numpy()\\n        formulas = df[\\"molecular_formula\\"].astype(str).to_numpy()\\n        npeaks = df[\\"num_peaks\\"].fillna(0).astype(np.int32).to_numpy()\\n        for ikey, smi, formula, n_peaks in zip(ikeys, smiles, formulas, npeaks):\\n            if not ikey or ikey == \\"nan\\" or not smi or smi == \\"nan\\":\\n                continue\\n            prev = best_n.get(ikey, -1)\\n            if n_peaks > prev:\\n                best_n[ikey] = int(n_peaks)\\n                smiles_map[ikey] = smi\\n                formula_map[ikey] = formula\\n            elif ikey not in smiles_map:\\n                smiles_map[ikey] = smi\\n                formula_map[ikey] = formula\\n                best_n[ikey] = int(n_peaks)\\n        if progress:\\n            print(f\\"[library] pass1 row_group={n_groups_done} unique_skeletons={len(smiles_map)}\\")\\n        if max_row_groups is not None and n_groups_done >= max_row_groups:\\n            break\\n\\n    filled: set[str] = set()\\n    best_feat: dict[str, dict[str, Any]] = {}\\n    n_groups_done = 0\\n    for df in iter_train_row_groups(cfg.train_path, columns=cols):\\n        n_groups_done += 1\\n        npeaks = df[\\"num_peaks\\"].fillna(0).astype(np.int32).to_numpy()\\n        ikeys = df[\\"inchikey14\\"].astype(str).to_numpy()\\n        take = np.zeros(len(df), dtype=bool)\\n        for i, (ikey, n_peaks) in enumerate(zip(ikeys, npeaks)):\\n            if ikey in filled:\\n                continue\\n            if best_n.get(ikey, -1) == int(n_peaks):\\n                take[i] = True\\n        if not take.any():\\n            continue\\n        sub = df.loc[take]\\n        for rec in sub.itertuples(index=False):\\n            ikey = str(getattr(rec, \\"inchikey14\\") or \\"\\")\\n            if ikey in filled:\\n                continue\\n            feat = featurize_spectrum(\\n                getattr(rec, \\"ms2_mzs\\"),\\n                getattr(rec, \\"ms2_normalized_intensities\\"),\\n                float(getattr(rec, \\"precursor_mz\\") or 0.0),\\n                getattr(rec, \\"adduct\\"),\\n                cfg=cfg,\\n                collision_energy=getattr(rec, \\"collision_energy_ev\\"),\\n                ionization_mode=getattr(rec, \\"ionization_mode\\"),\\n            )\\n            best_feat[ikey] = feat\\n            filled.add(ikey)\\n        if progress:\\n            print(f\\"[library] pass2 row_group={n_groups_done} featurized={len(filled)}/{len(smiles_map)}\\")\\n        if max_row_groups is not None and n_groups_done >= max_row_groups:\\n            break\\n        if len(filled) >= len(smiles_map):\\n            break\\n\\n    keys = list(smiles_map.keys())\\n    smiles = [smiles_map[k] for k in keys]\\n    formulas = [formula_map[k] for k in keys]\\n    masses = np.array([formula_to_mass(f) for f in formulas], dtype=np.float64)\\n    # Fall back to representative precursor-derived mass if formula failed.\\n    for i, k in enumerate(keys):\\n        if masses[i] <= 0 and k in best_feat:\\n            masses[i] = float(best_feat[k][\\"neutral_mass\\"])\\n\\n    fps = np.zeros((len(keys), cfg.fp_bits), dtype=np.uint8)\\n    for i, smi in enumerate(smiles):\\n        fps[i] = morgan_fingerprint(smi, n_bits=cfg.fp_bits, radius=cfg.morgan_radius)\\n        if progress and (i + 1) % 25000 == 0:\\n            print(f\\"[library] fingerprints {i+1}/{len(keys)}\\")\\n\\n    peak_mz = np.zeros((len(keys), cfg.top_n_peaks), dtype=np.float32)\\n    peak_int = np.zeros((len(keys), cfg.top_n_peaks), dtype=np.float32)\\n    peak_mask = np.zeros((len(keys), cfg.top_n_peaks), dtype=np.float32)\\n    peak_nl = np.zeros((len(keys), cfg.top_n_peaks), dtype=np.float32)\\n    precursor_feat = np.full((len(keys), cfg.precursor_feat_dim), np.nan, dtype=np.float32)\\n    adduct_id = np.full(len(keys), -1, dtype=np.int64)\\n    for i, k in enumerate(keys):\\n        feat = best_feat.get(k)\\n        if feat is None:\\n            continue\\n        peak_mz[i] = feat[\\"peak_mz\\"]\\n        peak_int[i] = feat[\\"peak_intensity\\"]\\n        peak_mask[i] = feat[\\"peak_mask\\"]\\n        peak_nl[i] = feat[\\"peak_nl\\"]\\n        precursor_feat[i] = feat[\\"precursor_feat\\"]\\n        adduct_id[i] = feat[\\"adduct_id\\"]\\n\\n    ion_mode = np.array(\\n        [str((best_feat.get(k) or {}).get(\\"ionization_mode\\") or \\"\\") for k in keys],\\n        dtype=object,\\n    )\\n    return StructureIndex.build(\\n        smiles=smiles,\\n        inchikey14=keys,\\n        exact_mass=masses,\\n        fingerprints=fps,\\n        peak_mz=peak_mz,\\n        peak_intensity=peak_int,\\n        peak_mask=peak_mask,\\n        formulas=formulas,\\n        ionization_mode=ion_mode,\\n        peak_nl=peak_nl,\\n        precursor_feat=precursor_feat,\\n        adduct_id=adduct_id,\\n    )\\n\\n\\ndef build_library_from_frame(df: pd.DataFrame, cfg: Config) -> StructureIndex:\\n    \\"\\"\\"Small-data path used by tests: unique InChIKey14 from an in-memory frame.\\"\\"\\"\\n    smiles_map: dict[str, str] = {}\\n    formula_map: dict[str, str] = {}\\n    best_n: dict[str, int] = {}\\n    best_feat: dict[str, dict[str, Any]] = {}\\n    for rec in df.itertuples(index=False):\\n        ikey = str(getattr(rec, \\"inchikey14\\") or \\"\\")\\n        smi = str(getattr(rec, \\"normalized_smiles\\") or \\"\\")\\n        if not ikey or not smi:\\n            continue\\n        n_peaks = int(getattr(rec, \\"num_peaks\\") or 0)\\n        if ikey not in smiles_map:\\n            smiles_map[ikey] = smi\\n            formula_map[ikey] = str(getattr(rec, \\"molecular_formula\\") or \\"\\")\\n            best_n[ikey] = -1\\n        if n_peaks < best_n[ikey]:\\n            continue\\n        feat = featurize_spectrum(\\n            getattr(rec, \\"ms2_mzs\\"),\\n            getattr(rec, \\"ms2_normalized_intensities\\"),\\n            float(getattr(rec, \\"precursor_mz\\") or 0.0),\\n            getattr(rec, \\"adduct\\"),\\n            cfg=cfg,\\n            collision_energy=getattr(rec, \\"collision_energy_ev\\", None),\\n            ionization_mode=getattr(rec, \\"ionization_mode\\", None),\\n        )\\n        best_n[ikey] = n_peaks\\n        best_feat[ikey] = feat\\n    keys = list(smiles_map)\\n    smiles = [smiles_map[k] for k in keys]\\n    masses = np.array([formula_to_mass(formula_map[k]) for k in keys], dtype=np.float64)\\n    for i, k in enumerate(keys):\\n        if masses[i] <= 0:\\n            masses[i] = float(best_feat[k][\\"neutral_mass\\"])\\n    fps = np.stack(\\n        [morgan_fingerprint(s, n_bits=cfg.fp_bits, radius=cfg.morgan_radius) for s in smiles]\\n    )\\n    peak_mz = np.stack([best_feat[k][\\"peak_mz\\"] for k in keys])\\n    peak_int = np.stack([best_feat[k][\\"peak_intensity\\"] for k in keys])\\n    peak_mask = np.stack([best_feat[k][\\"peak_mask\\"] for k in keys])\\n    peak_nl = np.stack([best_feat[k][\\"peak_nl\\"] for k in keys])\\n    precursor_feat = np.stack([best_feat[k][\\"precursor_feat\\"] for k in keys])\\n    adduct_id = np.asarray([best_feat[k][\\"adduct_id\\"] for k in keys], dtype=np.int64)\\n    ion_mode = np.array(\\n        [str(best_feat[k].get(\\"ionization_mode\\") or \\"\\") for k in keys],\\n        dtype=object,\\n    )\\n    return StructureIndex.build(\\n        smiles=smiles,\\n        inchikey14=keys,\\n        exact_mass=masses,\\n        fingerprints=fps,\\n        peak_mz=peak_mz,\\n        peak_intensity=peak_int,\\n        peak_mask=peak_mask,\\n        formulas=[formula_map[k] for k in keys],\\n        ionization_mode=ion_mode,\\n        peak_nl=peak_nl,\\n        precursor_feat=precursor_feat,\\n        adduct_id=adduct_id,\\n    )\\n\\n\\n@torch.no_grad()\\ndef predict_fingerprints(\\n    model: Spec2FP | None,\\n    features: list[dict[str, Any]],\\n    cfg: Config,\\n    device=None,\\n    *,\\n    return_logits: bool = False,\\n) -> np.ndarray:\\n    \\"\\"\\"Sigmoid fingerprint probabilities, shape ``(N, fp_bits)``. Uniform if no model.\\n\\n    ``return_logits=True`` skips the sigmoid so callers can average an ensemble.\\n    \\"\\"\\"\\n    n = len(features)\\n    if n == 0:\\n        return np.zeros((0, cfg.fp_bits), dtype=np.float32)\\n    if model is None:\\n        if return_logits:\\n            return np.zeros((n, cfg.fp_bits), dtype=np.float32)\\n        return np.full((n, cfg.fp_bits), 0.5, dtype=np.float32)\\n    if device is None:\\n        device, _ = get_device(prefer_xla=False)\\n    model = model.to(device)\\n    model.eval()\\n    stacked = stack_features(features)\\n    out = np.zeros((n, cfg.fp_bits), dtype=np.float32)\\n    bs = cfg.batch_size\\n    for start in range(0, n, bs):\\n        sl = slice(start, min(start + bs, n))\\n        batch = {\\n            \\"peak_mz\\": torch.from_numpy(stacked[\\"peak_mz\\"][sl]),\\n            \\"peak_intensity\\": torch.from_numpy(stacked[\\"peak_intensity\\"][sl]),\\n            \\"peak_nl\\": torch.from_numpy(stacked[\\"peak_nl\\"][sl]),\\n            \\"peak_mask\\": torch.from_numpy(stacked[\\"peak_mask\\"][sl]),\\n            \\"binned\\": torch.from_numpy(stacked[\\"binned\\"][sl]),\\n            \\"precursor_feat\\": torch.from_numpy(stacked[\\"precursor_feat\\"][sl]),\\n            \\"adduct_id\\": torch.from_numpy(stacked[\\"adduct_id\\"][sl]),\\n        }\\n        real_n = int(batch[\\"peak_mz\\"].shape[0])\\n        batch = _pad_to_batch_size(_move(batch, device), bs)\\n        logits = _forward(model, batch)[:real_n]\\n        arr = logits.detach().cpu().numpy().astype(np.float32)\\n        if return_logits:\\n            out[sl] = arr\\n        else:\\n            out[sl] = 1.0 / (1.0 + np.exp(-np.clip(arr, -20.0, 20.0)))\\n    return out\\n\\n\\ndef predict_fingerprint_ensemble(\\n    models: list[Spec2FP | None] | Spec2FP | None,\\n    features: list[dict[str, Any]],\\n    cfg: Config,\\n    device=None,\\n) -> np.ndarray:\\n    \\"\\"\\"Average pre-sigmoid logits across trained seeds, then sigmoid.\\"\\"\\"\\n    if models is None:\\n        seq: list[Spec2FP | None] = [None]\\n    elif isinstance(models, list):\\n        seq = models\\n    else:\\n        seq = [models]\\n    acc = None\\n    n_ok = 0\\n    for model in seq:\\n        if model is None:\\n            continue\\n        logits = predict_fingerprints(model, features, cfg, device=device, return_logits=True)\\n        acc = logits if acc is None else acc + logits\\n        n_ok += 1\\n    if acc is None or n_ok == 0:\\n        return predict_fingerprints(None, features, cfg)\\n    mean = acc / float(n_ok)\\n    print(f\\"[ensemble] models={n_ok} averaged pre-sigmoid logits\\")\\n    return (1.0 / (1.0 + np.exp(-np.clip(mean, -20.0, 20.0)))).astype(np.float32)\\n\\n\\n@dataclass\\nclass _Hit:\\n    smiles: str\\n    inchikey14: str\\n    score: float\\n    spec: float = 0.0\\n    tani: float = 0.0\\n    mass: float = 1.0\\n    n_match: float = 0.0\\n\\n\\ndef _collect_query_masses(\\n    feat: dict[str, Any],\\n    extra_groups: list[dict[str, Any]],\\n    *,\\n    exhaustive: bool = False,\\n) -> list[float]:\\n    groups = extra_groups if extra_groups else [feat]\\n    masses: list[float] = []\\n    for g in groups:\\n        prec = float(g.get(\\"precursor_mz\\") or 0.0)\\n        adduct = g.get(\\"adduct\\") or feat.get(\\"adduct\\")\\n        ion = g.get(\\"ionization_mode\\") or feat.get(\\"ionization_mode\\")\\n        masses.extend(\\n            plausible_neutral_masses(\\n                prec,\\n                adduct if isinstance(adduct, str) else None,\\n                ionization_mode=str(ion) if ion is not None else None,\\n                exhaustive=exhaustive,\\n            )\\n        )\\n        masses.append(float(g.get(\\"neutral_mass\\") or 0.0))\\n    masses.append(float(feat.get(\\"neutral_mass\\") or 0.0))\\n    out: list[float] = []\\n    seen: set[float] = set()\\n    for m in masses:\\n        if m <= 0 or not np.isfinite(m):\\n            continue\\n        key = round(float(m), 4)\\n        if key in seen:\\n            continue\\n        seen.add(key)\\n        out.append(float(m))\\n    return out or [0.0]\\n\\n\\ndef _fingerprints_for(index: StructureIndex, cand_idx: np.ndarray, cfg: Config) -> np.ndarray:\\n    fps = getattr(index, \\"fingerprints\\", None)\\n    if fps is not None and fps.ndim == 2 and fps.shape[1] >= 8:\\n        return np.ascontiguousarray(fps[cand_idx])\\n    packed = getattr(index, \\"fp_packed\\", None)\\n    if packed is not None and getattr(packed, \\"ndim\\", 0) == 2 and packed.shape[0] == len(index.smiles):\\n        return unpack_fingerprints(np.ascontiguousarray(packed[cand_idx]), cfg.fp_bits).astype(np.uint8)\\n    return np.stack(\\n        [morgan_fingerprint(str(index.smiles[int(i)]), n_bits=cfg.fp_bits) for i in cand_idx]\\n    )\\n\\n\\n_CALIB = None\\n\\n\\ndef _score_calibration():\\n    global _CALIB\\n    if _CALIB is None:\\n        from src.calibration import load_calibration\\n\\n        _CALIB = load_calibration()\\n    return _CALIB\\n\\n\\ndef _score_candidates(\\n    feat: dict[str, Any],\\n    extra_groups: list[dict[str, Any]],\\n    pred_fp: np.ndarray,\\n    index: StructureIndex,\\n    cand_idx: np.ndarray,\\n    cfg: Config,\\n    *,\\n    use_spectral: bool,\\n    is_class2: bool = False,\\n    reranker=None,\\n    formula_counts: np.ndarray | None = None,\\n) -> list[_Hit]:\\n    if cand_idx is None or cand_idx.size == 0:\\n        return []\\n    groups = extra_groups if extra_groups else [feat]\\n    masses = np.asarray(_collect_query_masses(feat, groups), dtype=np.float64)\\n    cand_mass = index.exact_mass[cand_idx]\\n    dm = np.min(np.abs(cand_mass[:, None] - masses[None, :]), axis=1)\\n    mass_sc = mass_gaussian_score(dm, cand_mass, cfg.mass_score_ppm_scale)\\n\\n    spec = np.zeros(cand_idx.size, dtype=np.float32)\\n    n_match = np.zeros(cand_idx.size, dtype=np.float32)\\n    spec_cap = 512\\n    if use_spectral and index.peak_mz is not None and cand_idx.size:\\n        if cand_idx.size > spec_cap:\\n            pick = np.argpartition(-mass_sc, spec_cap - 1)[:spec_cap]\\n        else:\\n            pick = np.arange(cand_idx.size)\\n        spec_pick = np.zeros(pick.size, dtype=np.float32)\\n        match_pick = np.zeros(pick.size, dtype=np.float32)\\n        for g in groups:\\n            s, m = modified_cosine(\\n                g[\\"peak_mz\\"],\\n                g[\\"peak_intensity\\"],\\n                g[\\"peak_mask\\"],\\n                float(g[\\"precursor_mz\\"]),\\n                index.peak_mz[cand_idx[pick]],\\n                index.peak_intensity[cand_idx[pick]],\\n                index.peak_mask[cand_idx[pick]],\\n                cand_mass[pick].astype(np.float32),\\n                cfg.modified_cosine_mz_tol,\\n                return_n_match=True,\\n            )\\n            better = s > spec_pick\\n            match_pick = np.where(better, m, match_pick)\\n            spec_pick = np.maximum(spec_pick, s)\\n        spec[pick] = spec_pick\\n        n_match[pick] = match_pick\\n\\n    fps = _fingerprints_for(index, cand_idx, cfg)\\n    tani, fcos = fingerprint_scores(\\n        pred_fp, fps, cfg.fp_threshold, soft_mix=float(getattr(cfg, \\"fp_soft_mix\\", 0.5))\\n    )\\n    spec = np.nan_to_num(spec, nan=0.0).astype(np.float32)\\n    tani = np.nan_to_num(tani, nan=0.0).astype(np.float32)\\n    fcos = np.nan_to_num(fcos, nan=0.0).astype(np.float32)\\n    mass_sc = np.nan_to_num(mass_sc, nan=0.0).astype(np.float32)\\n    if bool(getattr(cfg, \\"use_soft_merge\\", True)):\\n        spec_for_p = spec if use_spectral else np.zeros_like(spec)\\n        scores = _score_calibration().or_score(spec_for_p, tani, mass_sc)\\n    else:\\n        scores = combine_scores(spec, tani, mass_sc, cfg, fp_cosine=fcos)\\n    scores = np.nan_to_num(scores, nan=0.0)\\n    smiles = index.smiles[cand_idx]\\n    keys = index.inchikey14[cand_idx]\\n    hits: list[_Hit] = []\\n    for i in range(cand_idx.size):\\n        smi = str(smiles[i])\\n        key = str(keys[i] or \\"\\") or inchikey14_from_smiles(smi) or smi\\n        hits.append(\\n            _Hit(\\n                smiles=smi,\\n                inchikey14=key,\\n                score=float(scores[i]),\\n                spec=float(spec[i]),\\n                tani=float(tani[i]),\\n                mass=float(mass_sc[i]),\\n                n_match=float(n_match[i]),\\n            )\\n        )\\n    hits.sort(key=lambda h: (-h.spec, -h.score) if use_spectral else (-h.score,))\\n    return hits\\n\\n\\ndef _merge_hits(*groups: list[_Hit], top_k: int) -> list[_Hit]:\\n    best: dict[str, _Hit] = {}\\n    for group in groups:\\n        for hit in group:\\n            if not hit.smiles:\\n                continue\\n            key = hit.inchikey14 or inchikey14_from_smiles(hit.smiles) or hit.smiles\\n            prev = best.get(key)\\n            if prev is None or hit.score > prev.score:\\n                best[key] = _Hit(\\n                    hit.smiles,\\n                    key,\\n                    hit.score,\\n                    spec=hit.spec,\\n                    tani=hit.tani,\\n                    mass=hit.mass,\\n                    n_match=hit.n_match,\\n                )\\n    return sorted(best.values(), key=lambda h: -h.score)[: int(top_k)]\\n\\n\\ndef _hits_to_smiles(hits: list[_Hit]) -> list[str]:\\n    return [h.smiles for h in hits]\\n\\n\\ndef _rank_one(\\n    feat: dict[str, Any],\\n    extra_groups: list[dict[str, Any]],\\n    pred_fp: np.ndarray,\\n    index: StructureIndex,\\n    cfg: Config,\\n    *,\\n    reranker=None,\\n    formula_counts: np.ndarray | None = None,\\n) -> list[_Hit]:\\n    masses = _collect_query_masses(feat, extra_groups, exhaustive=False)\\n    tight = index.query_masses(masses, cfg, use_fallback=False)\\n    return _score_candidates(\\n        feat,\\n        extra_groups,\\n        pred_fp,\\n        index,\\n        tight,\\n        cfg,\\n        use_spectral=True,\\n        is_class2=False,\\n        reranker=reranker,\\n        formula_counts=formula_counts,\\n    )\\n\\n\\ndef _rank_fingerprint_only(\\n    feat: dict[str, Any],\\n    extra_groups: list[dict[str, Any]],\\n    pred_fp: np.ndarray,\\n    index: StructureIndex,\\n    cfg: Config,\\n    *,\\n    reranker=None,\\n    formula_counts: np.ndarray | None = None,\\n) -> list[_Hit]:\\n    \\"\\"\\"Class 2: mass window + fingerprint Tanimoto (no spectral library).\\"\\"\\"\\n    masses = _collect_query_masses(feat, extra_groups, exhaustive=True)\\n    cand_idx = index.query_masses(\\n        masses,\\n        cfg,\\n        use_fallback=False,\\n        ppm=float(cfg.mass_ppm),\\n        abs_da=float(cfg.mass_abs_da),\\n        max_candidates=max(int(cfg.max_mass_candidates), 8192),\\n    )\\n    if cand_idx.size < cfg.top_k:\\n        extra = index.query_masses(\\n            masses,\\n            cfg,\\n            use_fallback=False,\\n            ppm=25.0,\\n            abs_da=0.03,\\n            max_candidates=8192,\\n        )\\n        if extra.size:\\n            cand_idx = np.unique(np.concatenate([cand_idx, extra])) if cand_idx.size else extra\\n    return _score_candidates(\\n        feat,\\n        extra_groups,\\n        pred_fp,\\n        index,\\n        cand_idx,\\n        cfg,\\n        use_spectral=False,\\n        is_class2=True,\\n        reranker=reranker,\\n        formula_counts=formula_counts,\\n    )\\n\\n\\ndef _rank_mass_shifted_analogs(\\n    feat: dict[str, Any],\\n    extra_groups: list[dict[str, Any]],\\n    pred_fp: np.ndarray,\\n    index: StructureIndex,\\n    cfg: Config,\\n) -> list[_Hit]:\\n    \\"\\"\\"Retrieve mass-shifted parents, apply NP transforms, keep in-window products.\\"\\"\\"\\n    from src.analogs import expand_smiles, shifted_parent_masses\\n\\n    masses = _collect_query_masses(feat, extra_groups, exhaustive=True)\\n    parent_masses = shifted_parent_masses(masses)\\n    if not parent_masses:\\n        return []\\n    cand_idx = index.query_masses(\\n        parent_masses,\\n        cfg,\\n        use_fallback=False,\\n        ppm=float(cfg.mass_ppm),\\n        abs_da=float(cfg.mass_abs_da),\\n        max_candidates=max(int(cfg.max_mass_candidates), 4096),\\n    )\\n    if cand_idx.size == 0:\\n        return []\\n    fps = _fingerprints_for(index, cand_idx, cfg)\\n    tani, _ = fingerprint_scores(\\n        pred_fp, fps, cfg.fp_threshold, soft_mix=float(getattr(cfg, \\"fp_soft_mix\\", 0.5))\\n    )\\n    k = min(40, int(cand_idx.size))\\n    pick = np.argpartition(-tani, k - 1)[:k]\\n    pick = pick[np.argsort(-tani[pick])]\\n    parents = [str(index.smiles[int(cand_idx[int(i)])]) for i in pick]\\n    qmass = float(feat.get(\\"neutral_mass\\") or 0.0)\\n    products = expand_smiles(\\n        parents,\\n        query_mass=qmass,\\n        mass_ppm=float(cfg.mass_ppm),\\n        max_per_parent=8,\\n        max_total=40,\\n        in_window_only=True,\\n    )\\n    if not products:\\n        return []\\n    calib = _score_calibration() if bool(getattr(cfg, \\"use_soft_merge\\", True)) else None\\n    hits: list[_Hit] = []\\n    for smi in products:\\n        key = inchikey14_from_smiles(smi) or smi\\n        mass = exact_mass_from_smiles(smi)\\n        fp = morgan_fingerprint(smi, n_bits=cfg.fp_bits)\\n        t, fcos = fingerprint_scores(\\n            pred_fp, fp.reshape(1, -1), cfg.fp_threshold, soft_mix=float(getattr(cfg, \\"fp_soft_mix\\", 0.5))\\n        )\\n        dm = np.array([abs(mass - qmass) if mass > 0 and qmass > 0 else 50.0], dtype=np.float64)\\n        mass_sc = mass_gaussian_score(dm, np.array([max(mass, qmass, 1.0)]), cfg.mass_score_ppm_scale)\\n        if calib is not None:\\n            score = float(calib.or_score(0.0, t, mass_sc)[0])\\n        else:\\n            spec = np.array([0.0], dtype=np.float32)\\n            score = float(combine_scores(spec, t, mass_sc, cfg, fp_cosine=fcos)[0])\\n        hits.append(_Hit(smi, key, score, spec=0.0, tani=float(t[0]), mass=float(mass_sc[0])))\\n    hits.sort(key=lambda h: -h.score)\\n    return hits\\n\\n\\ndef _class3_denovo(\\n    feat: dict[str, Any],\\n    pred_fp: np.ndarray,\\n    cfg: Config,\\n    decoder,\\n    tokenizer,\\n) -> list[str]:\\n    \\"\\"\\"Sample the SMILES decoder on CPU to fill remaining InChIKey14 slots.\\"\\"\\"\\n    if decoder is None or tokenizer is None:\\n        return []\\n    from src.models.smiles_decoder import generate_candidates_for_molecule\\n\\n    cpu_decoder = decoder.to(\\"cpu\\").eval()\\n    fp_t = torch.from_numpy(np.asarray(pred_fp, dtype=np.float32))\\n    adduct_idx = int(feat.get(\\"adduct_id\\") or 0)\\n    return generate_candidates_for_molecule(\\n        cpu_decoder,\\n        fp_t,\\n        float(feat[\\"neutral_mass\\"]),\\n        adduct_idx,\\n        tokenizer,\\n        num_samples=int(cfg.smiles_num_samples),\\n        mass_tolerance_ppm=float(cfg.smiles_mass_ppm),\\n        target_k=int(cfg.top_k),\\n        temp=float(cfg.smiles_temperature),\\n        top_p=float(cfg.smiles_top_p),\\n        max_len=int(cfg.smiles_gen_max_len),\\n        keep_mass_misses=True,\\n    )\\n\\n\\ndef _fingerprint_pad(pred_fp: np.ndarray, index: StructureIndex, cfg: Config, already: list[str]) -> list[str]:\\n    have = {inchikey14_from_smiles(s) or s for s in already}\\n    need = max(int(cfg.top_k) - len(already), 0)\\n    if need <= 0:\\n        return []\\n    pick, _ = fingerprint_knn(\\n        pred_fp,\\n        index.fingerprints,\\n        k=min(need + 16, 64),\\n        threshold=cfg.fp_threshold,\\n    )\\n    out: list[str] = []\\n    for i in pick:\\n        key = str(index.inchikey14[int(i)])\\n        if key in have:\\n            continue\\n        out.append(str(index.smiles[int(i)]))\\n        if len(out) >= need:\\n            break\\n    return out\\n\\n\\ndef _hits_from_smiles(\\n    smiles_list: list[str],\\n    pred_fp: np.ndarray,\\n    query_mass: float,\\n    cfg: Config,\\n    *,\\n    base_score: float = 0.15,\\n) -> list[_Hit]:\\n    hits: list[_Hit] = []\\n    for smi in smiles_list:\\n        key = inchikey14_from_smiles(smi) or smi\\n        mass = 0.0\\n        try:\\n            from src.chem import exact_mass_from_smiles\\n\\n            mass = exact_mass_from_smiles(smi)\\n        except Exception:\\n            mass = 0.0\\n        fp = morgan_fingerprint(smi, n_bits=cfg.fp_bits)\\n        tani, fcos = fingerprint_scores(pred_fp, fp.reshape(1, -1), cfg.fp_threshold)\\n        dm = np.array([abs(mass - query_mass) if mass > 0 and query_mass > 0 else 50.0])\\n        mass_sc = mass_gaussian_score(dm, np.array([max(mass, query_mass, 1.0)]), cfg.mass_score_ppm_scale)\\n        spec = np.array([0.0], dtype=np.float32)\\n        score = float(combine_scores(spec, tani, mass_sc, cfg, fp_cosine=fcos)[0])\\n        hits.append(\\n            _Hit(\\n                smi,\\n                key,\\n                max(score, base_score * 0.01),\\n                spec=0.0,\\n                tani=float(tani[0]),\\n                mass=float(mass_sc[0]),\\n            )\\n        )\\n    return hits\\n\\n\\nCLASS1_LOCK_COSINE = 0.75\\n\\n\\ndef _linear_score(hit: _Hit, cfg: Config) -> float:\\n    \\"\\"\\"Equal-footing merge score: 0.42 spec + 0.46 Tanimoto + 0.12 mass.\\"\\"\\"\\n    w_s = float(cfg.w_spectral)\\n    w_f = float(cfg.w_fingerprint)\\n    w_m = float(cfg.w_mass)\\n    w = w_s + w_f + w_m\\n    if w <= 0:\\n        w = 1.0\\n    return float((w_s * hit.spec + w_f * hit.tani + w_m * hit.mass) / w)\\n\\n\\ndef _with_linear_score(hit: _Hit, cfg: Config) -> _Hit:\\n    return _Hit(\\n        hit.smiles,\\n        hit.inchikey14,\\n        _linear_score(hit, cfg),\\n        spec=hit.spec,\\n        tani=hit.tani,\\n        mass=hit.mass,\\n        n_match=hit.n_match,\\n    )\\n\\n\\ndef _merge_tiers(\\n    class1: list[_Hit],\\n    class2: list[_Hit],\\n    analog_hits: list[_Hit],\\n    cfg: Config,\\n    *,\\n    lock_cut: float | None = None,\\n) -> list[_Hit]:\\n    \\"\\"\\"Lock cosine>=0.75 Class 1; compete weak Class 1, Class 2, and analogs by score.\\"\\"\\"\\n    cut = float(lock_cut if lock_cut is not None else getattr(cfg, \\"soft_merge_lock_cosine\\", CLASS1_LOCK_COSINE) or CLASS1_LOCK_COSINE)\\n    top_k = int(cfg.top_k)\\n    locked = [h for h in class1 if h.spec >= cut]\\n    locked.sort(key=lambda h: (-h.spec, -h.score))\\n    locked = locked[:top_k]\\n    locked_keys = {h.inchikey14 for h in locked}\\n    remain = max(top_k - len(locked), 0)\\n    weak_c1 = [_with_linear_score(h, cfg) for h in class1 if h.inchikey14 not in locked_keys]\\n    valid_c2 = [_with_linear_score(h, cfg) for h in class2 if h.inchikey14 not in locked_keys]\\n    analog_scored = [_with_linear_score(h, cfg) for h in analog_hits if h.inchikey14 not in locked_keys]\\n    competed = _merge_hits(weak_c1, valid_c2, analog_scored, top_k=remain)\\n    return (locked + competed)[:top_k]\\n\\n\\ndef rank_molecules(\\n    molecule_features: dict[str, dict[str, Any]],\\n    pred_fps: dict[str, np.ndarray],\\n    index: StructureIndex,\\n    cfg: Config,\\n    *,\\n    external_index: StructureIndex | None = None,\\n    decoder=None,\\n    tokenizer=None,\\n    reranker=None,\\n    formula_model=None,\\n) -> dict[str, list[str]]:\\n    \\"\\"\\"Lock cosine>=0.75 Class 1; compete weak train, Class 2, and mass-shifted analogs.\\"\\"\\"\\n    from src.analogs import expand_smiles\\n\\n    lock_cut = float(getattr(cfg, \\"soft_merge_lock_cosine\\", CLASS1_LOCK_COSINE) or CLASS1_LOCK_COSINE)\\n    out: dict[str, list[str]] = {}\\n    n = len(molecule_features)\\n    n_c2 = 0\\n    n_c2_hits = 0\\n    n_c2_empty = 0\\n    n_mol_locked = 0\\n    n_shift = 0\\n    n_c2_placed = 0\\n    n_analog_placed = 0\\n    c2_logged = False\\n    analog_logged = False\\n    for i, (mid, feat) in enumerate(molecule_features.items()):\\n        groups = feat.get(\\"group_features\\") or [feat]\\n        qmass = float(feat.get(\\"neutral_mass\\") or 0.0)\\n        class1 = _rank_one(feat, groups, pred_fps[mid], index, cfg)\\n        locked = [h for h in class1 if h.spec >= lock_cut]\\n        locked.sort(key=lambda h: (-h.spec, -h.score))\\n        locked = locked[: int(cfg.top_k)]\\n        if locked:\\n            n_mol_locked += 1\\n        locked_keys = {h.inchikey14 for h in locked}\\n        class2: list[_Hit] = []\\n        analog_hits: list[_Hit] = []\\n        if external_index is not None:\\n            if not c2_logged:\\n                print(\\n                    \\"[class2] dynamic mass window lookup via np.searchsorted on sorted exact_mass \\"\\n                    f\\"(n_lib={len(external_index.smiles)} \\"\\n                    f\\"mass={float(np.min(external_index.exact_mass)):.1f}-\\"\\n                    f\\"{float(np.max(external_index.exact_mass)):.1f} Da; \\"\\n                    \\"not prefiltered on test.parquet)\\"\\n                )\\n                n_lib = len(external_index.smiles)\\n                if 20_000 <= n_lib < 250_000:\\n                    print(\\n                        f\\"[class2] WARNING: n_lib={n_lib} looks like LOTUS-only (~133k). \\"\\n                        \\"V6 expects COCONUT \\u222a LOTUS \\u2248 400k.\\"\\n                    )\\n                c2_logged = True\\n            raw_c2 = _rank_fingerprint_only(feat, groups, pred_fps[mid], external_index, cfg)\\n            if not raw_c2:\\n                n_c2_empty += 1\\n            class2 = [h for h in raw_c2 if h.inchikey14 not in locked_keys]\\n            if class2:\\n                n_c2 += 1\\n                n_c2_hits += len(class2)\\n            remain_after_lock = max(int(cfg.top_k) - len(locked), 0)\\n            if remain_after_lock > 0 and bool(getattr(cfg, \\"use_mass_shift\\", True)):\\n                if not analog_logged:\\n                    print(\\n                        \\"[analog] mass-shifted NP products compete with weak Class 1 and Class 2\\"\\n                    )\\n                    analog_logged = True\\n                analog_hits = [\\n                    h\\n                    for h in _rank_mass_shifted_analogs(\\n                        feat, groups, pred_fps[mid], external_index, cfg\\n                    )\\n                    if h.inchikey14 not in locked_keys\\n                ]\\n                if analog_hits:\\n                    n_shift += 1\\n        analog_keys = {h.inchikey14 for h in analog_hits}\\n        merged = _merge_tiers(class1, class2, analog_hits, cfg, lock_cut=lock_cut)\\n        shifted = [h for h in merged if h.inchikey14 in analog_keys]\\n        n_analog_placed += len(shifted)\\n        n_c2_placed += sum(1 for h in merged if h.inchikey14 in {x.inchikey14 for x in class2})\\n        smiles = _hits_to_smiles(merged)[: cfg.top_k]\\n        if len(smiles) < cfg.top_k:\\n            analog_smi = expand_smiles(\\n                smiles[:5] or _hits_to_smiles(class1[:5]),\\n                query_mass=qmass,\\n                mass_ppm=max(float(cfg.mass_ppm), 20.0),\\n            )\\n            smiles = merge_unique_smiles(smiles, analog_smi, top_k=cfg.top_k)\\n        if len(smiles) < cfg.top_k and decoder is not None:\\n            denovo = _class3_denovo(feat, pred_fps[mid], cfg, decoder, tokenizer)\\n            smiles = merge_unique_smiles(smiles, denovo, top_k=cfg.top_k)\\n        if len(smiles) < cfg.top_k:\\n            smiles = merge_unique_smiles(\\n                smiles,\\n                _fingerprint_pad(pred_fps[mid], index, cfg, smiles),\\n                top_k=cfg.top_k,\\n            )\\n        if len(smiles) < cfg.top_k:\\n            smiles = merge_unique_smiles(\\n                smiles,\\n                _nearest_library_smiles(index, qmass, cfg.top_k),\\n                top_k=cfg.top_k,\\n            )\\n        out[mid] = smiles[: cfg.top_k]\\n        if (i + 1) % 50 == 0:\\n            print(\\n                f\\"[rank] {i+1}/{n} last_n={len(out[mid])} locked={len(locked)} \\"\\n                f\\"class2_hits={len(class2)} shifted={len(shifted)}\\"\\n            )\\n    print(f\\"[rank] molecules with locked Class 1 (cosine>={lock_cut}): {n_mol_locked}/{n}\\")\\n    if external_index is not None:\\n        print(\\n            f\\"[rank] molecules with Class 2 mass hits: {n_c2}/{n} \\"\\n            f\\"empty_windows={n_c2_empty} \\"\\n            f\\"mean_hits={0 if n_c2 == 0 else n_c2_hits / n_c2:.1f} \\"\\n            f\\"mass_shifted={n_shift} class2_slots={n_c2_placed} analog_slots={n_analog_placed}\\"\\n        )\\n        if n_c2 == 0:\\n            print(\\n                \\"[rank] WARNING: 0 Class 2 mass hits from np.searchsorted. \\"\\n                \\"Check that class2_candidates.parquet is attached and spans 50-2000 Da.\\"\\n            )\\n    return out\\n\\n\\ndef featurize_test_molecules(test_df: pd.DataFrame, cfg: Config) -> dict[str, dict[str, Any]]:\\n    grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)\\n    for rec in test_df.to_dict(orient=\\"records\\"):\\n        grouped[str(rec[\\"molecule_id\\"])].append(rec)\\n    feats = {}\\n    n_fail = 0\\n    for mid, rows in grouped.items():\\n        try:\\n            feats[mid] = featurize_aggregated(rows, cfg=cfg)\\n        except Exception as exc:\\n            n_fail += 1\\n            row = rows[0]\\n            feats[mid] = featurize_spectrum(\\n                row.get(\\"ms2_mzs\\"),\\n                row.get(\\"ms2_normalized_intensities\\"),\\n                float(row.get(\\"precursor_mz\\") or 0.0),\\n                row.get(\\"adduct\\"),\\n                cfg=cfg,\\n                collision_energy=None,\\n                ionization_mode=row.get(\\"ionization_mode\\"),\\n            )\\n            print(f\\"[warn] featurize_aggregated failed for {mid}: {exc}; used first spectrum\\")\\n    if n_fail:\\n        print(f\\"[warn] featurize fallback on {n_fail}/{len(grouped)} molecules\\")\\n    return feats\\n\\n\\ndef _nearest_library_smiles(index: StructureIndex, mass: float, top_k: int) -> list[str]:\\n    n = int(index.smiles.shape[0])\\n    if n == 0:\\n        return []\\n    k = min(int(top_k), n)\\n    if not np.isfinite(mass) or mass <= 0:\\n        return [str(s) for s in index.smiles[:k].tolist()]\\n    err = np.abs(index.exact_mass - float(mass))\\n    pick = np.argpartition(err, k - 1)[:k]\\n    pick = pick[np.argsort(err[pick])]\\n    return [str(index.smiles[int(i)]) for i in pick]\\n\\n\\ndef predict_test(\\n    cfg: Config | None = None,\\n    *,\\n    index: StructureIndex | None = None,\\n    model: Spec2FP | None = None,\\n    models: list[Spec2FP | None] | None = None,\\n    test_df: pd.DataFrame | None = None,\\n    output_path: Path | str | None = None,\\n    decoder=None,\\n    tokenizer=None,\\n    external_index: StructureIndex | None = None,\\n    reranker=None,\\n    formula_model=None,\\n) -> pd.DataFrame:\\n    cfg = cfg or get_config()\\n    if test_df is None:\\n        test_df = load_test(cfg)\\n    if index is None:\\n        index = build_library_from_train(cfg)\\n    ensemble = [m for m in (models or []) if m is not None]\\n    if not ensemble and model is None and cfg.checkpoint_path.exists():\\n        model = load_model(cfg.checkpoint_path, cfg)\\n    if not ensemble and model is not None:\\n        ensemble = [model]\\n    if decoder is None and cfg.smiles_checkpoint_path.exists():\\n        from src.smiles_train import load_decoder\\n\\n        decoder, tokenizer = load_decoder(cfg.smiles_checkpoint_path, cfg, map_location=\\"cpu\\")\\n    if external_index is None:\\n        external_index = load_external_structure_index(None, cfg)\\n    if external_index is None:\\n        print(\\"[class2] no LOTUS/COCONUT table attached \\u2014 ranking Class 1 only\\")\\n    # V4 reranker / formula head are disabled until Class 1 is recovered.\\n    reranker = None\\n    formula_model = None\\n\\n    mol_feats = featurize_test_molecules(test_df, cfg)\\n    mids = list(mol_feats.keys())\\n    feat_list = [mol_feats[m] for m in mids]\\n    fps = predict_fingerprint_ensemble(ensemble, feat_list, cfg)\\n    if bool(getattr(cfg, \\"use_neighbor_fp\\", True)) and getattr(index, \\"peak_mz\\", None) is not None:\\n        try:\\n            from src.neighbors import SpectralNeighborIndex, blend_predicted_fingerprints\\n\\n            nbr = SpectralNeighborIndex.from_structure_index(index, cfg)\\n            fps = blend_predicted_fingerprints(\\n                fps,\\n                feat_list,\\n                nbr,\\n                k=int(getattr(cfg, \\"neighbor_k\\", 20)),\\n                alpha=float(getattr(cfg, \\"neighbor_blend\\", 0.5)),\\n            )\\n        except Exception as exc:\\n            print(f\\"[neighbor] skipped: {exc}\\")\\n    pred_map = {m: fps[i] for i, m in enumerate(mids)}\\n    ranked = rank_molecules(\\n        mol_feats,\\n        pred_map,\\n        index,\\n        cfg,\\n        external_index=external_index,\\n        decoder=decoder,\\n        tokenizer=tokenizer,\\n        reranker=reranker,\\n        formula_model=formula_model,\\n    )\\n    ids = official_molecule_ids(cfg, test_df)\\n    n_empty = 0\\n    filled: dict[str, list[str]] = {}\\n    n_short = 0\\n    for mid in ids:\\n        guesses = sanitize_guesses(ranked.get(mid, []), top_k=cfg.top_k)\\n        if len(guesses) < cfg.top_k:\\n            n_short += 1\\n            feat = mol_feats.get(mid) or {}\\n            mass = float(feat.get(\\"neutral_mass\\") or 0.0)\\n            guesses = merge_unique_smiles(\\n                guesses,\\n                sanitize_guesses(_nearest_library_smiles(index, mass, cfg.top_k), top_k=cfg.top_k),\\n                top_k=cfg.top_k,\\n            )\\n        if not guesses:\\n            n_empty += 1\\n            guesses = [FALLBACK_SMILES]\\n        filled[mid] = guesses\\n    if n_empty:\\n        print(f\\"[submit] filled {n_empty} empty rankings with fallback SMILES\\")\\n    if n_short:\\n        print(f\\"[submit] padded {n_short} rows to {cfg.top_k} guesses\\")\\n    sub = build_submission_frame(ids, filled, top_k=cfg.top_k)\\n    if output_path is not None:\\n        write_submission_csv(sub, output_path)\\n        from src.ranker import validate_submission\\n\\n        validate_submission(output_path, cfg.sample_submission_path)\\n    return sub\\n", "src/train_tpu.py": "\\"\\"\\"CLI: train Spec2FP on a streamed / cached train sample (CPU, GPU, or TPU).\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport sys\\nfrom pathlib import Path\\n\\n# Allow `python src/train_tpu.py` from repo root.\\nROOT = Path(__file__).resolve().parent.parent\\nif str(ROOT) not in sys.path:\\n    sys.path.insert(0, str(ROOT))\\n\\nfrom src.config import get_config\\nfrom src.data import dataset_from_dataframe, iter_train_row_groups, load_train_slice\\nfrom src.tpu_trainer import train_spec2fp\\n\\n\\ndef parse_args() -> argparse.Namespace:\\n    p = argparse.ArgumentParser(description=\\"Train Spec2FP (PyTorch / PyTorch-XLA)\\")\\n    p.add_argument(\\"--max-spectra\\", type=int, default=None)\\n    p.add_argument(\\"--epochs\\", type=int, default=None)\\n    p.add_argument(\\"--batch-size\\", type=int, default=None)\\n    p.add_argument(\\"--time-limit-s\\", type=float, default=None)\\n    p.add_argument(\\"--slice-rows\\", type=int, default=None, help=\\"Debug: only the first N parquet rows\\")\\n    return p.parse_args()\\n\\n\\ndef main() -> None:\\n    args = parse_args()\\n    overrides = {}\\n    if args.max_spectra is not None:\\n        overrides[\\"max_train_spectra\\"] = args.max_spectra\\n    if args.epochs is not None:\\n        overrides[\\"num_epochs\\"] = args.epochs\\n    if args.batch_size is not None:\\n        overrides[\\"batch_size\\"] = args.batch_size\\n    cfg = get_config(**overrides)\\n\\n    if args.slice_rows:\\n        df = load_train_slice(cfg, args.slice_rows)\\n        ds = dataset_from_dataframe(df, cfg)\\n    else:\\n        frames = []\\n        n = 0\\n        cap = cfg.max_train_spectra\\n        for g in iter_train_row_groups(cfg.train_path):\\n            take = min(len(g), cap - n)\\n            if take <= 0:\\n                break\\n            frames.append(g.iloc[:take])\\n            n += take\\n            print(f\\"[cache] collected {n} spectra\\")\\n            if n >= cap:\\n                break\\n        import pandas as pd\\n\\n        df = pd.concat(frames, ignore_index=True)\\n        ds = dataset_from_dataframe(df, cfg)\\n\\n    print(\\n        \\"static shapes:\\",\\n        {k: tuple(v.shape) if hasattr(v, \\"shape\\") else None for k, v in ds[0].items()},\\n    )\\n    train_spec2fp(ds, cfg, time_limit_s=args.time_limit_s)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')
for rel, content in FILES.items():
    path = ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)
    print("wrote", path, "bytes", len(content))


## 1b. Locate the attached Class 2 parquet (COCONUT ∪ LOTUS)

In [ ]:
def find_class2_table():
    names = {
        "class2_candidates.parquet",
        "class2_candidates.csv.gz",
        "class2_candidates.csv",
        "lotus.parquet",
    }
    roots = [Path("/kaggle/input"), ROOT / "data", Path("data")]
    explicit = [
        Path("/kaggle/input/class2_candidates.parquet2/class2_candidates.parquet"),
        Path("/kaggle/input/class2-candidates-parquet2/class2_candidates.parquet"),
        Path("/kaggle/input/class2_candidates.parquet/class2_candidates.parquet"),
    ]
    hits = [p for p in explicit if p.exists() and p.is_file()]
    for root in roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob("*"):
                if not p.is_file():
                    continue
                low = p.name.lower()
                parent = p.parent.name.lower()
                if low in names or ("class2" in low and p.suffix.lower() in {".parquet", ".csv", ".gz"}):
                    hits.append(p)
                elif "class2" in parent and low.endswith(".parquet"):
                    hits.append(p)
        except Exception:
            continue
    uniq = []
    seen = set()
    for p in hits:
        key = str(p)
        if key in seen:
            continue
        seen.add(key)
        uniq.append(p)
    # Prefer the largest parquet so the 453k COCONUT pool wins over old LOTUS (~19 MB).
    uniq.sort(key=lambda p: (0 if p.suffix.lower() == ".parquet" else 1, -p.stat().st_size))
    if uniq:
        print("class2 candidates found:")
        for p in uniq:
            print(" ", p, "bytes", p.stat().st_size)
    return uniq[0] if uniq else None

CLASS2_PATH = find_class2_table()
if CLASS2_PATH is None:
    msg = (
        "class2_candidates.parquet not attached. Add the dataset whose folder may be "
        "class2_candidates.parquet2 and whose file is class2_candidates.parquet "
        "(~61 MB COCONUT ∪ LOTUS, ~453k structures). Kernel source must stay < 1 MB."
    )
    if IS_KAGGLE and REQUIRE_CLASS2:
        raise FileNotFoundError(msg)
    print("WARNING:", msg, "Ranking will be Class 1 only.")
else:
    print("class2 table", CLASS2_PATH, "bytes", CLASS2_PATH.stat().st_size)
    nbytes = CLASS2_PATH.stat().st_size
    if nbytes < 5_000_000:
        print(
            "WARNING: Class 2 file is < 5 MB — this may be the old 57k test-mass slice. "
            "Attach the ~61 MB parquet (~453k structures, 50-2000 Da)."
        )
    elif nbytes < 40_000_000:
        print(
            "WARNING: Class 2 file is < 40 MB — this looks like LOTUS-only (~133k). "
            "V6 expects COCONUT ∪ LOTUS ≈ 453k (~61 MB)."
        )
    else:
        print("class2 parquet size looks like the V6 COCONUT ∪ LOTUS pool")


In [ ]:
import numpy as np
import pandas as pd
import torch

print("numpy", np.__version__, "pandas", pd.__version__, "torch", torch.__version__)
try:
    torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", "4")))
except Exception:
    pass

try:
    from rdkit import Chem
    print("rdkit", Chem.rdBase.rdkitVersion)
    HAS_RDKIT = True
    _probe = Chem.MolFromSmiles("CCO")
    assert _probe is not None, "RDKit imported but MolFromSmiles('CCO') returned None"
    assert HAS_RDKIT is True
except Exception as e:
    print("WARNING: RDKit missing — fingerprint training disabled, spectral+mass only.", e)
    HAS_RDKIT = False
    raise ImportError(
        "HAS_RDKIT is False. The first cell must pip-install the attached "
        "~35 MB rdkit .whl. Attaching a 20 KB stub does not install the package."
    ) from e

try:
    import torch_xla.core.xla_model as xm  # noqa: F401
    HAS_XLA = True
    print("torch_xla present (not used unless PREFER_XLA=True)")
except Exception:
    HAS_XLA = False
    print("torch_xla not available (expected on CPU/GPU sessions)")


## 2. Config, library, train, infer

In [ ]:
from src.config import get_config
from src.chem import has_rdkit, inchikey14_from_smiles
from src.data import dataset_from_dataframe, dataset_from_structure_index, load_test, load_train_slice
from src.infer import (
    build_library_from_frame,
    build_library_from_train,
    featurize_test_molecules,
    predict_fingerprint_ensemble,
    predict_fingerprints,
    predict_test,
    rank_molecules,
)
from src.metrics import mrr_at_k
from src.tpu_trainer import get_device, train_spec2fp

def find_competition_dir() -> Path:
    candidates = [
        Path("/kaggle/input/enveda-casmi26-molecule-id-mass-spectra"),
        Path("/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra"),
        ROOT / "data",
        ROOT / "enveda-CASMI26-molecule-id-mass-spectra",
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for parquet in input_root.rglob("train.parquet"):
            parent = parquet.parent
            if (parent / "test.parquet").exists():
                candidates.insert(0, parent)
    for cand in candidates:
        if cand.exists() and (cand / "train.parquet").exists():
            return cand
    raise FileNotFoundError("Could not find train.parquet under /kaggle/input or ./data")

data_dir = find_competition_dir()
print("competition data_dir", data_dir)

# CPU-sized model; GPU sessions get a slightly larger batch automatically after device pick.
cfg = get_config(
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    max_train_spectra=MAX_TRAIN_SPECTRA,
    use_representative_metadata=bool(USE_REPRESENTATIVE_METADATA),
    train_time_limit_s=TRAIN_TIME_LIMIT_S,
    top_n_peaks=TOP_N_PEAKS,
    n_mz_bins=N_MZ_BINS,
    fp_bits=FP_BITS,
    log_every=50,
    d_model=128,
    n_heads=4,
    n_transformer_layers=2,
    n_conv_channels=64,
    smiles_d_model=128,
    smiles_nhead=4,
    smiles_num_layers=2,
    smiles_dim_feedforward=512,
    smiles_num_epochs=1,
    smiles_batch_size=64,
    smiles_train_time_limit_s=DECODER_TIME_LIMIT_S,
    smiles_num_samples=48,
    smiles_gen_max_len=80,
    smiles_mass_ppm=50.0,
    num_workers=0,
    max_mass_candidates=8192,
    use_neighbor_fp=bool(USE_NEIGHBOR_FP),
    neighbor_blend=float(NEIGHBOR_BLEND),
    ensemble_seeds=int(ENSEMBLE_SEEDS),
)
cfg.data_dir = data_dir
cfg.artifact_dir = Path("/tmp/casmi_artifacts") if IS_KAGGLE else (ROOT / "artifacts")
cfg.artifact_dir.mkdir(parents=True, exist_ok=True)
cfg.external_candidates_path = CLASS2_PATH
print("class2", cfg.external_candidates_path, "exists", CLASS2_PATH is not None and Path(CLASS2_PATH).exists())

print("train", cfg.train_path.exists(), cfg.train_path)
print("test", cfg.test_path.exists(), cfg.test_path)
assert cfg.train_path.exists() and cfg.test_path.exists(), "Attach the competition dataset."


## 3. Build candidate library from train (mass + fingerprints + representative MS2)

In [ ]:
t_lib = time.time()
if FAST_DEV_RUN:
    print("FAST_DEV_RUN: library from first 8,000 train rows")
    slice_df = load_train_slice(cfg, 8_000)
    index = build_library_from_frame(slice_df, cfg)
else:
    index = build_library_from_train(cfg, progress=True)

print(
    f"library n={len(index.smiles)} peaks={None if index.peak_mz is None else index.peak_mz.shape} "
    f"fps={index.fingerprints.shape} in {time.time()-t_lib:.1f}s"
)
print("elapsed_s", round(time.time() - T_START, 1))


## 4. Train Spec2FP on CPU/GPU from the library

In [ ]:
device, use_xla = get_device(prefer_xla=bool(PREFER_XLA))
if str(device).startswith("cuda"):
    cfg.batch_size = max(cfg.batch_size, 128)
    cfg.d_model = 256
    cfg.n_heads = 8
    cfg.n_transformer_layers = 3
print("train device", device, "xla", use_xla)

if FAST_DEV_RUN:
    cfg.num_epochs = 1
    cfg.train_time_limit_s = 180.0
    cfg.max_train_spectra = 4_000
    cfg.log_every = 20

t_data = time.time()
n_fit = 4_000 if FAST_DEV_RUN else int(cfg.max_train_spectra)
print(
    f"building Spec2FP dataset from library n={len(index.smiles)} cap={n_fit} "
    f"neighbor_fp={cfg.use_neighbor_fp} blend={cfg.neighbor_blend}"
)
ds = dataset_from_structure_index(index, cfg, max_n=n_fit, seed=cfg.seed)
probe = ds[0]
print("example tensor shapes:", {k: tuple(v.shape) for k, v in probe.items()})
for k, expected in {
    "peak_mz": (cfg.top_n_peaks,),
    "peak_intensity": (cfg.top_n_peaks,),
    "peak_nl": (cfg.top_n_peaks,),
    "peak_mask": (cfg.top_n_peaks,),
    "binned": (cfg.n_mz_bins,),
    "fingerprint": (cfg.fp_bits,),
}.items():
    assert tuple(probe[k].shape) == expected, (k, probe[k].shape, expected)
print(f"dataset ready in {time.time()-t_data:.1f}s  n={len(ds)}")

t_tr = time.time()
models = []
n_seeds = 1 if FAST_DEV_RUN else int(ENSEMBLE_SEEDS)
seed_list = [42, 7, 123, 99, 2024][:n_seeds]
if HAS_RDKIT:
    for si, seed in enumerate(seed_list):
        seeds_left = n_seeds - si
        left = SESSION_BUDGET_S - (time.time() - T_START)
        budget = min(float(PER_SEED_MAX_S), max(0.0, (left - float(RANK_RESERVE_S)) / max(seeds_left, 1)))
        budget = min(budget, float(TRAIN_TIME_LIMIT_S))
        if FAST_DEV_RUN:
            budget = min(budget, 180.0)
        if budget < 8 * 60 and si > 0:
            print(f"skipping remaining seeds; budget_s={budget:.0f}")
            break
        cfg.seed = int(seed)
        cfg.ckpt_name = f"spec2fp_s{seed}.pt"
        cfg.train_time_limit_s = float(budget)
        print(f"ensemble seed={seed} time_limit_s={budget:.0f} epochs={cfg.num_epochs} ckpt={cfg.ckpt_name}")
        try:
            m = train_spec2fp(ds, cfg, device=device, seed=int(seed), ckpt_name=cfg.ckpt_name, time_limit_s=budget)
            models.append(m.to("cpu").eval())
        except Exception:
            traceback.print_exc()
            print(f"Training failed for seed={seed}")
    if not models:
        print("Training failed — continuing with spectral + mass ranking.")
else:
    print("Skipping neural training (no RDKit).")
model = models[0] if models else None
print("ensemble models", len(models))
print(f"train wall {time.time()-t_tr:.1f}s")
del ds


## 4b. Train prefix-conditioned SMILES decoder (Class 3)

In [ ]:
from src.smiles_tokenizer import SmilesTokenizer
from src.smiles_train import dataset_from_structures, train_smiles_decoder
from src.models.smiles_decoder import count_parameters, decoder_from_config

decoder = None
smiles_tokenizer = None
remaining = SESSION_BUDGET_S - (time.time() - T_START)
do_decoder = bool(TRAIN_DECODER) and HAS_RDKIT and remaining > 15 * 60
print("decoder remaining_s", round(remaining, 1), "do_decoder", do_decoder)
if do_decoder:
    try:
        smiles_tokenizer = SmilesTokenizer.default()
        smiles_tokenizer.save(cfg.smiles_vocab_path)
        if FAST_DEV_RUN:
            cfg.smiles_num_epochs = 1
            cfg.smiles_train_time_limit_s = 120.0
        cap = 2_000 if FAST_DEV_RUN else int(DECODER_MAX_SMILES)
        n_smi = min(len(index.smiles), cap)
        rng = np.random.default_rng(cfg.seed)
        pick = (
            np.sort(rng.choice(len(index.smiles), size=n_smi, replace=False))
            if len(index.smiles) > n_smi
            else np.arange(len(index.smiles))
        )
        smi_ds = dataset_from_structures(
            index.smiles[pick].tolist(),
            index.exact_mass[pick],
            index.fingerprints[pick].astype(np.float32),
            smiles_tokenizer,
            cfg,
        )
        probe = smi_ds[0]
        assert tuple(probe["tgt_tokens"].shape) == (cfg.smiles_max_len,), probe["tgt_tokens"].shape
        n_params = count_parameters(decoder_from_config(smiles_tokenizer, cfg))
        print("smiles vocab", smiles_tokenizer.vocab_size, "n", len(smi_ds), "params", n_params)
        print("decoder train n", len(smi_ds), "device", "cpu")
        decoder = train_smiles_decoder(smi_ds, smiles_tokenizer, cfg, device=torch.device("cpu"))
        decoder = decoder.to("cpu").eval()
        del smi_ds
        print("decoder ckpt", cfg.smiles_checkpoint_path.exists(), cfg.smiles_checkpoint_path)
    except Exception:
        traceback.print_exc()
        print("SMILES decoder training failed or skipped — Class 3 fallback disabled.")
        decoder = None
        smiles_tokenizer = smiles_tokenizer
else:
    print("Skipping SMILES decoder (flag/time/RDKit).")


## 5. Rank test molecules (Class 1/2/3, InChIKey14-deduped top 25)

In [ ]:
from src.ranker import validate_submission, write_submission_csv

out_path = (KAGGLE_WORKING if IS_KAGGLE else cfg.artifact_dir) / "submission.csv"
# If the session is almost out of time, skip expensive Class 3 sampling.
remaining = SESSION_BUDGET_S - (time.time() - T_START)
if remaining < 20 * 60:
    print("low time remaining — ranking without Class 3 decoder")
    decoder = None
sub = predict_test(
    cfg,
    index=index,
    model=model,
    models=models,
    decoder=decoder,
    tokenizer=smiles_tokenizer,
    output_path=out_path,
)
validate_submission(out_path, cfg.sample_submission_path)
print(sub.head())
print("rows", len(sub), "empty smiles", int((sub["smiles"].astype(str).str.strip() == "").sum()))
print("max guesses", sub["smiles"].str.split(";").map(lambda x: len([t for t in x if t])).max())
assert list(sub.columns) == ["molecule_id", "smiles"]
if not FAST_DEV_RUN:
    assert len(sub) == 400, len(sub)
    assert not (sub["smiles"].astype(str).str.strip() == "").any()
print("wrote", out_path, "bytes", out_path.stat().st_size)
print("elapsed_s", round(time.time() - T_START, 1))


## 6. Optional: MRR@25 on a labeled train holdout (does not affect submission)

In [ ]:
try:
    hold = load_train_slice(cfg, 2_000 if not FAST_DEV_RUN else 400)
    q = hold.iloc[-80:].reset_index(drop=True)
    from src.preprocessing import featurize_spectrum

    q_feats = [
        featurize_spectrum(
            q.iloc[i]["ms2_mzs"],
            q.iloc[i]["ms2_normalized_intensities"],
            float(q.iloc[i]["precursor_mz"]),
            q.iloc[i]["adduct"],
            cfg=cfg,
            collision_energy=q.iloc[i]["collision_energy_ev"],
            ionization_mode=q.iloc[i]["ionization_mode"],
        )
        for i in range(len(q))
    ]
    q_fps = predict_fingerprints(model, q_feats, cfg)
    mol_feats = {f"h{i}": q_feats[i] for i in range(len(q_feats))}
    pred_map = {f"h{i}": q_fps[i] for i in range(len(q_feats))}
    ranked = rank_molecules(mol_feats, pred_map, index, cfg)
    pred_lists = [ranked[f"h{i}"] for i in range(len(q_feats))]
    score = mrr_at_k(pred_lists, true_keys=q["inchikey14"].astype(str).tolist(), k=25)
    print(f"holdout MRR@25 (train skeletons in library): {score:.4f}")
except Exception:
    traceback.print_exc()
    print("holdout MRR skipped (does not affect submission.csv)")
print("done  elapsed_s", round(time.time() - T_START, 1))


## 7. Leave only `submission.csv` in `/kaggle/working`

In [ ]:
import shutil

final_path = (KAGGLE_WORKING if IS_KAGGLE else Path(".")) / "submission.csv"
if Path(out_path).resolve() != final_path.resolve():
    final_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(out_path, final_path)
write_submission_csv(sub, final_path)
validate_submission(final_path, cfg.sample_submission_path)

if IS_KAGGLE:
    keep = {"submission.csv"}
    for p in Path("/kaggle/working").iterdir():
        if p.name in keep or p.name.startswith("."):
            continue
        try:
            if p.is_dir():
                shutil.rmtree(p, ignore_errors=True)
            else:
                p.unlink()
        except Exception as exc:
            print("could not remove", p, exc)
    leftover = sorted(x.name for x in Path("/kaggle/working").iterdir())
    csvs = [x.name for x in Path("/kaggle/working").glob("*.csv")]
    print("kaggle/working leftover", leftover, "csvs", csvs)
    # Extra CSVs are what confuse the scorer; other Kaggle system files may remain.
    assert csvs == ["submission.csv"], csvs

print("final submission", final_path, "bytes", final_path.stat().st_size)
print("done")
